# 3\.3\.1 Prepare Anomaly Detection Frameworks

This notebook marks the handoff from representation learning into anomaly detection\. The shared representation assets are now fixed: Raw Scaled, PCA at 80% cumulative explained variance, and PCA→UMAP 10D\. The work here is to define the comparison universes, samples, and baseline assumptions that downstream anomaly\-generation notebooks will inherit\.

The central design question is what each mobility state should be compared against when it is scored as unusual\. Representation learning can remain global, but anomaly scoring may need a more local comparison universe, especially by temporal bucket\. This notebook prepares that framework before DBSCAN, Isolation Forest, and Gaussian Mixture Model notebooks generate method\-specific anomaly outputs\.

Before the framework sections begin, this notebook sets up the shared imports, constants, asset locations, and helper functions that the rest of the anomaly\-preparation workflow will reuse\. Keeping that scaffold centralized here helps the later sections stay shorter and reduces path drift across notebooks\.

In [1]:
# ---------------------------------------------------------------------
# Load notebook setup imports
# ---------------------------------------------------------------------

from __future__ import annotations

from pathlib import Path
from time import perf_counter
import json

import duckdb
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.metrics import pairwise_distances
from sklearn.neighbors import NearestNeighbors
from sklearn.cluster import DBSCAN

from project_branding import (
    BRAND_COLORS,
    BRAND_COLOR_SEQUENCE,
    BRAND_DIVERGING_SEQUENCE,
    BRAND_MAP_COLORS,
)

pd.set_option("display.max_rows", 1000)

px.defaults.color_discrete_sequence = BRAND_COLOR_SEQUENCE
px.defaults.template = "plotly_white"


def apply_brand_plotly_layout(
    fig,
    *,
    title=None,
    width=None,
    height=None,
    legend_title=None,
):
    layout_updates = {
        "template": "plotly_white",
        "paper_bgcolor": "white",
        "plot_bgcolor": BRAND_COLORS["ice"],
        "font": {"color": BRAND_COLORS["dark_teal"]},
        "margin": {"l": 70, "r": 40, "t": 80, "b": 70},
        "legend": {
            "title": {"text": legend_title} if legend_title else None,
            "bgcolor": "rgba(255,255,255,0.85)",
            "bordercolor": BRAND_COLORS["seafoam"],
            "borderwidth": 1,
            "font": {"color": BRAND_COLORS["dark_teal"]},
        },
    }

    if title is not None:
        layout_updates["title"] = {
            "text": title,
            "font": {"color": BRAND_COLORS["dark_teal"]},
        }
    if width is not None:
        layout_updates["width"] = width
    if height is not None:
        layout_updates["height"] = height

    fig.update_layout(**layout_updates)
    fig.update_xaxes(
        gridcolor="rgba(11, 78, 74, 0.14)",
        zerolinecolor="rgba(11, 78, 74, 0.20)",
        tickfont={"color": BRAND_COLORS["dark_teal"]},
        title_font={"color": BRAND_COLORS["dark_teal"]},
    )
    fig.update_yaxes(
        gridcolor="rgba(11, 78, 74, 0.14)",
        zerolinecolor="rgba(11, 78, 74, 0.20)",
        tickfont={"color": BRAND_COLORS["dark_teal"]},
        title_font={"color": BRAND_COLORS["dark_teal"]},
    )
    return fig


print("Notebook setup imports loaded.")

Notebook setup imports loaded.


In [2]:
# ---------------------------------------------------------------------
# Define shared notebook constants, defaults, and rebuild toggles
# ---------------------------------------------------------------------

MODEL_FEATURE_SET_NAMES = ["bus", "subway", "taxi", "fhvhv", "multimodal"]
REPRESENTATION_NAMES = ["raw_scaled", "pca_80pct", "umap_pca_to_umap_10d"]
REPRESENTATION_FAMILY_BY_NAME = {
    "raw_scaled": "raw",
    "pca_80pct": "pca",
    "umap_pca_to_umap_10d": "umap",
}
REPRESENTATION_ROLE_BY_NAME = {
    "raw_scaled": "source_baseline",
    "pca_80pct": "linear_reduced_candidate",
    "umap_pca_to_umap_10d": "nonlinear_reduced_candidate",
}

PCA_80_COMPONENTS_BY_SET = {
    "bus": 14,
    "subway": 11,
    "taxi": 15,
    "fhvhv": 13,
    "multimodal": 46,
}

MODEL_ID_COLUMN = "modeling_row_id"
TEMPORAL_BUCKET_COLUMN = "temporal_bucket"
DATE_COLUMN = "date"
ZONE_ID_COLUMN = "taxi_zone_id"
ZONE_COLUMN = "zone"
BOROUGH_COLUMN = "borough"
PRE_POST_CP_COLUMN = "pre_post_cp"
POLICY_GEOGRAPHY_COLUMN = "policy_geography_class"

DEFAULT_METADATA_COLUMNS = [
    MODEL_ID_COLUMN,
    ZONE_ID_COLUMN,
    DATE_COLUMN,
    TEMPORAL_BUCKET_COLUMN,
    ZONE_COLUMN,
    BOROUGH_COLUMN,
    PRE_POST_CP_COLUMN,
    POLICY_GEOGRAPHY_COLUMN,
]

DEFAULT_COMPARISON_UNIVERSE_NAMES = [
    "global",
    "temporal_bucket",
    "temporal_bucket_pre_post_cp",
]

COMPARISON_UNIVERSE_GROUPING_COLUMNS = {
    "global_all_rows": [],
    "same_temporal_bucket": [TEMPORAL_BUCKET_COLUMN],
    "same_temporal_bucket_and_pre_post_cp": [TEMPORAL_BUCKET_COLUMN, PRE_POST_CP_COLUMN],
    "same_temporal_bucket_and_policy_geography": [TEMPORAL_BUCKET_COLUMN, POLICY_GEOGRAPHY_COLUMN],
}

DBSCAN_PILOT_REPRESENTATION_NAME = "pca_80pct"
DBSCAN_PILOT_MIN_SAMPLES = 15
DBSCAN_PILOT_EPS_QUANTILE = 0.95
DBSCAN_PILOT_MAX_ROWS_PER_GROUP = 4_000
DBSCAN_PILOT_MIN_ROWS_PER_GROUP = 100

ANOMALY_CALIBRATION_SAMPLE_ROWS = 50_000
ANOMALY_METRIC_SAMPLE_ROWS = 15_000
ANOMALY_FRAMEWORK_RANDOM_STATE = 42
DEFAULT_NEIGHBOR_K = 15

TAXI_PCA_PERIOD_VALUES = ["pre_cp", "post_cp"]
TAXI_PCA_PERIOD_COMPARISON_REPRESENTATIONS = [
    "taxi_pca_shared",
    "taxi_pca_prepost_split",
]

REBUILD_ANOMALY_INPUT_SUMMARIES = False
REBUILD_ANOMALY_CALIBRATION_SAMPLES = False
REBUILD_COMPARISON_UNIVERSE_TABLES = False
REBUILD_BASELINE_FEASIBILITY_TABLES = False
REBUILD_TAXI_PCA_PERIOD_DIAGNOSTICS = False
REBUILD_SELF_NORMALIZATION_DIAGNOSTICS = False
WRITE_331_OUTPUTS = True

BOROUGH_COLOR_MAP = {
    "Manhattan": BRAND_COLORS["dark_teal"],
    "Brooklyn": BRAND_COLORS["terracotta"],
    "Queens": BRAND_COLORS["seafoam"],
    "Bronx": BRAND_COLORS["pale_peach"],
    "Staten Island": BRAND_MAP_COLORS.get("secondary", BRAND_COLORS["ice"]),
    "EWR": BRAND_MAP_COLORS.get("accent", BRAND_COLORS["terracotta"]),
    "Unknown": BRAND_COLORS["ice"],
    "Missing": BRAND_COLORS["ice"],
}

POLICY_GEOGRAPHY_COLOR_MAP = {
    "cbd": BRAND_COLORS["dark_teal"],
    "gateway_or_adjacent": BRAND_COLORS["terracotta"],
    "outside": BRAND_COLORS["seafoam"],
    "Missing": BRAND_COLORS["pale_peach"],
}

DAY_TYPE_COLOR_MAP = {
    "weekday": BRAND_COLORS["dark_teal"],
    "weekend": BRAND_COLORS["terracotta"],
    "Missing": BRAND_COLORS["ice"],
}

print("Shared notebook constants and toggles defined.")

Shared notebook constants and toggles defined.


In [3]:
# ---------------------------------------------------------------------
# Define canonical asset directories and final-asset path lookups
# ---------------------------------------------------------------------

PROJECT_ROOT = Path("/datasets/_deepnote_work")
PIPELINE_DATA_DIR = PROJECT_ROOT / "pipeline_data"

FINAL_311_DIR = PIPELINE_DATA_DIR / "3.1.1.final_tables"
FINAL_321_DIR = PIPELINE_DATA_DIR / "3.2.1.final_tables"
FINAL_324_DIR = PIPELINE_DATA_DIR / "3.2.4.final_tables"

INTERMEDIATE_331_DIR = PIPELINE_DATA_DIR / "3.3.1.intermediate_tables"
FINAL_331_DIR = PIPELINE_DATA_DIR / "3.3.1.final_tables"

for output_dir in [INTERMEDIATE_331_DIR, FINAL_331_DIR]:
    output_dir.mkdir(parents=True, exist_ok=True)

ROW_METADATA_PATHS_BY_SET = {
    feature_set: FINAL_311_DIR / f"{feature_set}_row_metadata.parquet"
    for feature_set in MODEL_FEATURE_SET_NAMES
}
RAW_MATRIX_PATHS_BY_SET = {
    feature_set: FINAL_311_DIR / f"{feature_set}_raw_modeling_matrix.parquet"
    for feature_set in MODEL_FEATURE_SET_NAMES
}
SCALED_MATRIX_PATHS_BY_SET = {
    feature_set: FINAL_311_DIR / f"{feature_set}_scaled_modeling_matrix.parquet"
    for feature_set in MODEL_FEATURE_SET_NAMES
}



PCA_80_FINAL_PATHS_BY_SET = {
    feature_set: FINAL_321_DIR / f"{feature_set}_pca_modeling_scores.parquet"
    for feature_set in MODEL_FEATURE_SET_NAMES
}

UMAP_PCA_TO_UMAP_10D_FINAL_PATHS_BY_SET = {
    feature_set: FINAL_324_DIR / f"{feature_set}_umap_pca_to_umap_10d.parquet"
    for feature_set in MODEL_FEATURE_SET_NAMES
}

REPRESENTATION_PATHS_BY_SET_AND_NAME = {
    feature_set: {
        "raw_scaled": SCALED_MATRIX_PATHS_BY_SET[feature_set],
        "pca_80pct": PCA_80_FINAL_PATHS_BY_SET[feature_set],
        "umap_pca_to_umap_10d": UMAP_PCA_TO_UMAP_10D_FINAL_PATHS_BY_SET[feature_set],
    }
    for feature_set in MODEL_FEATURE_SET_NAMES
}

DBSCAN_PILOT_SUMMARY_PATH = (
    INTERMEDIATE_331_DIR / "dbscan_baseline_pilot_summary.parquet"
)
DBSCAN_PILOT_GROUP_DETAIL_PATH = (
    INTERMEDIATE_331_DIR / "dbscan_baseline_pilot_group_detail.parquet"
)
DBSCAN_UNIVERSE_GROUP_SIZE_PATH = (
    INTERMEDIATE_331_DIR / "comparison_universe_group_sizes.parquet"
)


ANOMALY_CALIBRATION_ID_PATHS_BY_SET = {
    feature_set: INTERMEDIATE_331_DIR / f"{feature_set}_anomaly_calibration_ids.parquet"
    for feature_set in MODEL_FEATURE_SET_NAMES
}
ANOMALY_CALIBRATION_METADATA_PATHS_BY_SET = {
    feature_set: INTERMEDIATE_331_DIR / f"{feature_set}_anomaly_calibration_metadata.parquet"
    for feature_set in MODEL_FEATURE_SET_NAMES
}

FINAL_ANOMALY_CALIBRATION_ID_PATHS_BY_SET = {
    feature_set: FINAL_331_DIR / f"{feature_set}_anomaly_calibration_ids.parquet"
    for feature_set in MODEL_FEATURE_SET_NAMES
}

FINAL_ANOMALY_CALIBRATION_METADATA_PATHS_BY_SET = {
    feature_set: FINAL_331_DIR / f"{feature_set}_anomaly_calibration_metadata.parquet"
    for feature_set in MODEL_FEATURE_SET_NAMES
}

FINAL_322_DIR = PIPELINE_DATA_DIR / "3.2.2.final_tables"

TAXI_PERIOD_PCA_80_FINAL_PATHS_BY_PERIOD = {
    "pre_cp": FINAL_322_DIR / "taxi_pre_cp_pca_80pct_modeling_scores.parquet",
    "post_cp": FINAL_322_DIR / "taxi_post_cp_pca_80pct_modeling_scores.parquet",
}

TAXI_SHARED_PCA_REFERENCE_PATH = PCA_80_FINAL_PATHS_BY_SET["taxi"]

TAXI_PCA_PERIOD_DIAGNOSTIC_SUMMARY_PATH = (
    INTERMEDIATE_331_DIR / "taxi_pca_period_diagnostic_summary.parquet"
)
TAXI_PCA_PERIOD_DIAGNOSTIC_DETAIL_PATH = (
    INTERMEDIATE_331_DIR / "taxi_pca_period_diagnostic_detail.parquet"
)

path_overview_df = pd.DataFrame(
    [
        {"asset_group": "upstream_3.1.1", "path": str(FINAL_311_DIR)},
        {"asset_group": "upstream_3.2.1", "path": str(FINAL_321_DIR)},
        {"asset_group": "upstream_3.2.4", "path": str(FINAL_324_DIR)},
        {"asset_group": "current_3.3.1_intermediate", "path": str(INTERMEDIATE_331_DIR)},
        {"asset_group": "current_3.3.1_final", "path": str(FINAL_331_DIR)},
    ]
)

display(path_overview_df)
print("Canonical final-asset directories and path lookups defined.")

,asset_group,path
0,upstream_3.1.1,/datasets/_deepnote_work/pipeline_data/3.1.1.f...
1,upstream_3.2.1,/datasets/_deepnote_work/pipeline_data/3.2.1.f...
2,upstream_3.2.4,/datasets/_deepnote_work/pipeline_data/3.2.4.f...
3,current_3.3.1_intermediate,/datasets/_deepnote_work/pipeline_data/3.3.1.i...
4,current_3.3.1_final,/datasets/_deepnote_work/pipeline_data/3.3.1.f...


Canonical final-asset directories and path lookups defined.


In [4]:
# ---------------------------------------------------------------------
# Define shared helper functions
# ---------------------------------------------------------------------

def sql_path(path_like: str | Path) -> str:
    return str(Path(path_like)).replace("\\", "/")


def duckdb_identifier(name: str) -> str:
    return '"' + str(name).replace('"', '""') + '"'


def should_rebuild(output_path: str | Path, rebuild_flag: bool) -> bool:
    return rebuild_flag or (not Path(output_path).exists())


def display_path_status(path_dict: dict[str, Path], value_column_name: str) -> pd.DataFrame:
    return pd.DataFrame(
        [
            {
                "feature_set": feature_set,
                value_column_name: str(path),
                "path_exists": Path(path).exists(),
            }
            for feature_set, path in path_dict.items()
        ]
    )


def load_representation_table(feature_set: str, representation_name: str) -> pd.DataFrame:
    return pd.read_parquet(REPRESENTATION_PATHS_BY_SET_AND_NAME[feature_set][representation_name])


def load_representation_columns(feature_set: str, representation_name: str) -> list[str]:
    representation_df = load_representation_table(feature_set, representation_name)
    return [
        column
        for column in representation_df.columns
        if column not in DEFAULT_METADATA_COLUMNS
    ]


print("Shared helper functions and canonical final-asset lookups defined.")

Shared helper functions and canonical final-asset lookups defined.


## 3\.3\.1\.1 Load Candidate Anomaly Representations

Load the three downstream candidate representations selected in 3\.2\.4: Raw Scaled, PCA at 80% cumulative explained variance, and PCA→UMAP 10D\. Each representation should be available for Bus, Subway, Taxi, FHVHV, and Multimodal, with the metadata fields needed for temporal, geographic, and policy\-context interpretation\.

This section confirms that the candidate anomaly\-detection inputs are present, aligned, and shaped the way the downstream framework notebooks expect before we start building shared calibration samples and comparison universes\.

In [5]:
# ---------------------------------------------------------------------
# Validate candidate anomaly-representation asset availability
# ---------------------------------------------------------------------

representation_asset_records = []

# Check that each modality has the full downstream handoff set plus row metadata.
for feature_set_name in MODEL_FEATURE_SET_NAMES:
    representation_asset_records.append(
        {
            "feature_set": feature_set_name,
            "raw_scaled_exists": REPRESENTATION_PATHS_BY_SET_AND_NAME[feature_set_name]["raw_scaled"].exists(),
            "pca_80pct_exists": REPRESENTATION_PATHS_BY_SET_AND_NAME[feature_set_name]["pca_80pct"].exists(),
            "umap_pca_to_umap_10d_exists": REPRESENTATION_PATHS_BY_SET_AND_NAME[feature_set_name]["umap_pca_to_umap_10d"].exists(),
            "row_metadata_exists": ROW_METADATA_PATHS_BY_SET[feature_set_name].exists(),
            "status": "pass"
            if (
                REPRESENTATION_PATHS_BY_SET_AND_NAME[feature_set_name]["raw_scaled"].exists()
                and REPRESENTATION_PATHS_BY_SET_AND_NAME[feature_set_name]["pca_80pct"].exists()
                and REPRESENTATION_PATHS_BY_SET_AND_NAME[feature_set_name]["umap_pca_to_umap_10d"].exists()
                and ROW_METADATA_PATHS_BY_SET[feature_set_name].exists()
            )
            else "review",
        }
    )

representation_asset_status_df = pd.DataFrame(representation_asset_records)

display(representation_asset_status_df)

assert representation_asset_status_df["status"].eq("pass").all(), (
    "One or more candidate anomaly-representation assets is missing."
)

print("Candidate anomaly-representation assets are available for all feature sets.")

,feature_set,raw_scaled_exists,pca_80pct_exists,umap_pca_to_umap_10d_exists,row_metadata_exists,status
0,bus,True,True,True,True,pass
1,subway,True,True,True,True,pass
2,taxi,True,True,True,True,pass
3,fhvhv,True,True,True,True,pass
4,multimodal,True,True,True,True,pass


Candidate anomaly-representation assets are available for all feature sets.


Findings\. All three downstream anomaly representations are available for every modality, and the supporting row metadata is present as well\. That means the handoff from 3\.2\.4 is intact: Raw Scaled, PCA at 80% explained variance, and PCA→UMAP 10D are all ready to be used in the anomaly\-preparation workflow for Bus, Subway, Taxi, FHVHV, and Multimodal\.

Once the files are on hand, the next useful check is row alignment\. We want to know that each representation still sits on the same observation grain as the row metadata, so later calibration and baseline comparisons are truly like\-for\-like\.

In [6]:
# ---------------------------------------------------------------------
# Validate row alignment across metadata and candidate representations
# ---------------------------------------------------------------------

representation_alignment_records = []

# Each candidate representation should stay on the same full observation grain as row metadata.
for feature_set_name in MODEL_FEATURE_SET_NAMES:
    metadata_df = pd.read_parquet(
        ROW_METADATA_PATHS_BY_SET[feature_set_name],
        columns=[MODEL_ID_COLUMN],
    )
    metadata_rows = len(metadata_df)
    metadata_id_set = set(metadata_df[MODEL_ID_COLUMN].tolist())

    for representation_name in REPRESENTATION_NAMES:
        representation_df = pd.read_parquet(
            REPRESENTATION_PATHS_BY_SET_AND_NAME[feature_set_name][representation_name],
            columns=[MODEL_ID_COLUMN],
        )
        representation_rows = len(representation_df)
        duplicate_modeling_rows = int(representation_df[MODEL_ID_COLUMN].duplicated().sum())
        representation_id_set = set(representation_df[MODEL_ID_COLUMN].tolist())

        representation_alignment_records.append(
            {
                "feature_set": feature_set_name,
                "representation_name": representation_name,
                "metadata_rows": metadata_rows,
                "representation_rows": representation_rows,
                "duplicate_modeling_rows": duplicate_modeling_rows,
                "missing_from_representation": len(metadata_id_set - representation_id_set),
                "extra_in_representation": len(representation_id_set - metadata_id_set),
                "status": "pass"
                if (
                    representation_rows == metadata_rows
                    and duplicate_modeling_rows == 0
                    and metadata_id_set == representation_id_set
                )
                else "review",
            }
        )

representation_alignment_df = pd.DataFrame(representation_alignment_records)

display(representation_alignment_df)

assert representation_alignment_df["status"].eq("pass").all(), (
    "One or more candidate representations is not aligned to row metadata."
)

print("Row alignment is valid across metadata and all candidate representations.")

,feature_set,representation_name,metadata_rows,representation_rows,duplicate_modeling_rows,missing_from_representation,extra_in_representation,status
0,bus,raw_scaled,1472947,1472947,0,0,0,pass
1,bus,pca_80pct,1472947,1472947,0,0,0,pass
2,bus,umap_pca_to_umap_10d,1472947,1472947,0,0,0,pass
3,subway,raw_scaled,911455,911455,0,0,0,pass
4,subway,pca_80pct,911455,911455,0,0,0,pass
5,subway,umap_pca_to_umap_10d,911455,911455,0,0,0,pass
6,taxi,raw_scaled,1541800,1541800,0,0,0,pass
7,taxi,pca_80pct,1541800,1541800,0,0,0,pass
8,taxi,umap_pca_to_umap_10d,1541800,1541800,0,0,0,pass
9,fhvhv,raw_scaled,1541800,1541800,0,0,0,pass


Row alignment is valid across metadata and all candidate representations.


Findings\. The downstream candidate representations are fully aligned to row metadata across all five feature sets\. For Raw Scaled, PCA 80%, and PCA\-\>UMAP 10D, row counts match exactly, there are no duplicate modeling\_row\_id values, and no rows are missing or added relative to metadata\. That gives us a clean full\-panel foundation for anomaly\-framework preparation without any representation\-specific alignment repair\.

With availability and alignment confirmed, it helps to make the comparison set explicit in one place: which representation family each asset belongs to, what role it will play downstream, and how much dimensionality each modality is carrying into anomaly work\.

In [7]:
# ---------------------------------------------------------------------
# Summarize the downstream candidate anomaly representations
# ---------------------------------------------------------------------

representation_summary_records = []

# Pull the representation dimensions from the actual handoff assets rather than re-hardcoding them.
for feature_set_name in MODEL_FEATURE_SET_NAMES:
    for representation_name in REPRESENTATION_NAMES:
        representation_columns = load_representation_columns(feature_set_name, representation_name)

        representation_summary_records.append(
            {
                "feature_set": feature_set_name,
                "representation_name": representation_name,
                "representation_family": REPRESENTATION_FAMILY_BY_NAME[representation_name],
                "representation_role": REPRESENTATION_ROLE_BY_NAME[representation_name],
                "input_dimensions": len(representation_columns),
            }
        )

candidate_representation_summary_df = pd.DataFrame(representation_summary_records)

candidate_representation_summary_display_df = (
    candidate_representation_summary_df
    .sort_values(["feature_set", "representation_role"])
    .reset_index(drop=True)
)

display(candidate_representation_summary_display_df)

print("Downstream anomaly-representation set summarized.")

,feature_set,representation_name,representation_family,representation_role,input_dimensions
0,bus,pca_80pct,pca,linear_reduced_candidate,14
1,bus,umap_pca_to_umap_10d,umap,nonlinear_reduced_candidate,10
2,bus,raw_scaled,raw,source_baseline,40
3,fhvhv,pca_80pct,pca,linear_reduced_candidate,13
4,fhvhv,umap_pca_to_umap_10d,umap,nonlinear_reduced_candidate,10
5,fhvhv,raw_scaled,raw,source_baseline,46
6,multimodal,pca_80pct,pca,linear_reduced_candidate,46
7,multimodal,umap_pca_to_umap_10d,umap,nonlinear_reduced_candidate,10
8,multimodal,raw_scaled,raw,source_baseline,233
9,subway,pca_80pct,pca,linear_reduced_candidate,11


Downstream anomaly-representation set summarized.


Findings\. The anomaly\-preparation notebook is now working with a consistent three\-representation comparison set for every modality: the full raw baseline, a linear reduced PCA candidate, and a nonlinear reduced UMAP candidate\. PCA remains moderately compressed while preserving modality\-specific dimensionality, ranging from 11 components for Subway to 46 for Multimodal, while UMAP gives a standardized 10\-dimensional nonlinear representation across all five feature sets\. This makes the downstream comparison easy to reason about: Raw carries the full feature space, PCA provides the fidelity\-preserving reduced view, and UMAP provides the more aggressive nonlinear alternative\.

Before moving into calibration sampling, let’s confirm that the three candidate representation tables all follow the expected downstream format\. Raw, PCA, and UMAP should each behave like representation tables rather than mixed analysis tables: one modeling\_row\_id, at least one numeric modeling column, and no unexpected metadata fields embedded inside the representation asset itself\.

In [8]:
# ---------------------------------------------------------------------
# Validate downstream representation-table format
# ---------------------------------------------------------------------

representation_format_records = []

# Raw, PCA, and UMAP should all follow the same contract:
# one modeling_row_id column plus numeric representation columns only.
for feature_set_name in MODEL_FEATURE_SET_NAMES:
    for representation_name in REPRESENTATION_NAMES:
        representation_path = REPRESENTATION_PATHS_BY_SET_AND_NAME[feature_set_name][representation_name]
        representation_df = pd.read_parquet(representation_path)

        all_columns = representation_df.columns.tolist()
        non_id_columns = [
            column_name
            for column_name in all_columns
            if column_name != MODEL_ID_COLUMN
        ]

        numeric_non_id_columns = [
            column_name
            for column_name in non_id_columns
            if pd.api.types.is_numeric_dtype(representation_df[column_name])
        ]

        unexpected_metadata_columns = [
            column_name
            for column_name in non_id_columns
            if column_name in DEFAULT_METADATA_COLUMNS
        ]

        if representation_name == "pca_80pct":
            family_specific_unexpected_columns = [
                column_name
                for column_name in non_id_columns
                if not str(column_name).startswith("PC")
            ]
        elif representation_name == "umap_pca_to_umap_10d":
            family_specific_unexpected_columns = [
                column_name
                for column_name in non_id_columns
                if not str(column_name).startswith("umap_")
            ]
        else:
            family_specific_unexpected_columns = []

        representation_format_records.append(
            {
                "feature_set": feature_set_name,
                "representation_name": representation_name,
                "total_columns": len(all_columns),
                "non_id_columns": len(non_id_columns),
                "numeric_non_id_columns": len(numeric_non_id_columns),
                "unexpected_metadata_columns": (
                    "none"
                    if not unexpected_metadata_columns
                    else ", ".join(unexpected_metadata_columns)
                ),
                "family_specific_unexpected_columns": (
                    "none"
                    if not family_specific_unexpected_columns
                    else ", ".join(family_specific_unexpected_columns)
                ),
                "status": "pass"
                if (
                    MODEL_ID_COLUMN in all_columns
                    and len(non_id_columns) > 0
                    and len(non_id_columns) == len(numeric_non_id_columns)
                    and len(unexpected_metadata_columns) == 0
                    and len(family_specific_unexpected_columns) == 0
                )
                else "review",
            }
        )

representation_format_df = pd.DataFrame(representation_format_records)

display(representation_format_df)

assert representation_format_df["status"].eq("pass").all(), (
    "One or more downstream representation tables does not match the expected format contract."
)

print("Downstream representation-table format validated.")

,feature_set,representation_name,total_columns,non_id_columns,numeric_non_id_columns,unexpected_metadata_columns,family_specific_unexpected_columns,status
0,bus,raw_scaled,41,40,40,none,none,pass
1,bus,pca_80pct,15,14,14,none,none,pass
2,bus,umap_pca_to_umap_10d,11,10,10,none,none,pass
3,subway,raw_scaled,22,21,21,none,none,pass
4,subway,pca_80pct,12,11,11,none,none,pass
5,subway,umap_pca_to_umap_10d,11,10,10,none,none,pass
6,taxi,raw_scaled,54,53,53,none,none,pass
7,taxi,pca_80pct,16,15,15,none,none,pass
8,taxi,umap_pca_to_umap_10d,11,10,10,none,none,pass
9,fhvhv,raw_scaled,47,46,46,none,none,pass


Downstream representation-table format validated.


Findings\. All three candidate representation families now follow the expected downstream format contract\. For every modality, the Raw, PCA, and UMAP tables each contain one modeling\_row\_id, at least one numeric modeling column, and no unexpected metadata fields embedded in the representation asset\. That means the anomaly\-preparation notebook can treat the three representation spaces consistently as modeling inputs, while continuing to load temporal and geographic context separately from the shared metadata tables\.

## 3\.3\.1\.2 Build Shared Anomaly Calibration Samples

This section sets up the shared calibration universe that the downstream anomaly notebooks will reuse across DBSCAN, Isolation Forest, and Gaussian Mixture Models\. The aim is to keep those later comparisons grounded on the same observations instead of letting each method drift into its own sample and its own notion of what “normal” looks like\.

We want the calibration samples to preserve the temporal structure that is most likely to matter for anomaly interpretation, especially temporal bucket and policy period, while still staying small enough for repeated nearest\-neighbor, density, and score\-distribution checks\. We will also keep an eye on geographic concentration after the samples are built, so we can tell the difference between faithfully representing the citywide mobility universe and accidentally over\-centering calibration on Manhattan or congestion\-pricing\-heavy regimes\.

### Profile the source universe and define the sampling design

Before we build shared calibration samples, it helps to see the shape of the full anomaly universe we are sampling from\. We’ll start with a compact source summary, then look more closely at the temporal\-bucket and policy\-period structure that should anchor the sample design\.

In [9]:
# ---------------------------------------------------------------------
# Summarize the source calibration universe
# ---------------------------------------------------------------------

source_universe_summary_records = []

for feature_set_name in MODEL_FEATURE_SET_NAMES:
    metadata_df = pd.read_parquet(ROW_METADATA_PATHS_BY_SET[feature_set_name])

    source_universe_summary_records.append(
        {
            "feature_set": feature_set_name,
            "total_rows": len(metadata_df),
            "temporal_buckets": metadata_df["temporal_bucket"].nunique(dropna=True),
            "pre_post_periods": metadata_df["pre_post_cp"].nunique(dropna=True),
            "boroughs": metadata_df["borough"].nunique(dropna=True),
            "policy_geography_classes": metadata_df["policy_geography_class"].nunique(dropna=True),
        }
    )

source_universe_summary_df = pd.DataFrame(source_universe_summary_records)

display(source_universe_summary_df)

print("Source calibration universe summarized.")

,feature_set,total_rows,temporal_buckets,pre_post_periods,boroughs,policy_geography_classes
0,bus,1472947,10,2,5,3
1,subway,911455,10,2,5,3
2,taxi,1541800,10,2,6,3
3,fhvhv,1541800,10,2,6,3
4,multimodal,1541800,10,2,6,3


Source calibration universe summarized.


Findings\. The full anomaly universe is structurally consistent across modalities, with 10 temporal buckets, 2 policy periods, and the expected borough and policy\-geography coverage present in every representation family\.

The source summary tells us how much coverage each modality has overall\. The next step is narrower: we want to understand how those rows are distributed across temporal buckets and policy periods, since that is the structure we want the shared calibration samples to preserve\.

In [10]:
# ---------------------------------------------------------------------
# Profile temporal-bucket and policy-period composition
# ---------------------------------------------------------------------

source_temporal_profile_records = []

for feature_set_name in MODEL_FEATURE_SET_NAMES:
    metadata_df = pd.read_parquet(
        ROW_METADATA_PATHS_BY_SET[feature_set_name],
        columns=["temporal_bucket", "pre_post_cp"],
    ).copy()

    metadata_df["temporal_bucket"] = metadata_df["temporal_bucket"].astype("string")
    metadata_df["pre_post_cp"] = metadata_df["pre_post_cp"].astype("string")

    total_rows = len(metadata_df)

    temporal_profile_df = (
        metadata_df.groupby(["temporal_bucket", "pre_post_cp"], dropna=False)
        .size()
        .reset_index(name="rows")
        .sort_values(["temporal_bucket", "pre_post_cp"])
        .reset_index(drop=True)
    )

    temporal_profile_df["feature_set"] = feature_set_name
    temporal_profile_df["row_share"] = (
        temporal_profile_df["rows"] / total_rows
    ).round(4)

    source_temporal_profile_records.append(
        temporal_profile_df[
            [
                "feature_set",
                "temporal_bucket",
                "pre_post_cp",
                "rows",
                "row_share",
            ]
        ]
    )

source_temporal_profile_df = pd.concat(
    source_temporal_profile_records,
    ignore_index=True,
)

display(source_temporal_profile_df)

print("Temporal-bucket and policy-period composition profiled.")

,feature_set,temporal_bucket,pre_post_cp,rows,row_share
0,bus,weekday_am_peak,post_cp,80822,0.0549
1,bus,weekday_am_peak,pre_cp,131524,0.0893
2,bus,weekday_evening,post_cp,80178,0.0544
3,bus,weekday_evening,pre_cp,130476,0.0886
4,bus,weekday_midday,post_cp,80626,0.0547
5,bus,weekday_midday,pre_cp,131439,0.0892
6,bus,weekday_overnight,post_cp,79595,0.0540
7,bus,weekday_overnight,pre_cp,129952,0.0882
8,bus,weekday_pm_peak,post_cp,81144,0.0551
9,bus,weekday_pm_peak,pre_cp,132048,0.0896


Temporal-bucket and policy-period composition profiled.


Findings\. The source universe shows a stable temporal\-bucket and pre/post structure that can be preserved directly in sampling, with larger weekday buckets and smaller weekend\-overnight buckets appearing consistently across modalities\.

With the temporal structure in view, we can translate it into an explicit sampling rule\. The main choice here is to keep the sample shared across representations, preserve temporal\-bucket coverage, and carry the observed pre/post mix within each temporal bucket rather than force a new balance artificially\.

In [11]:
# ---------------------------------------------------------------------
# Define the shared calibration sampling rules
# ---------------------------------------------------------------------

CALIBRATION_SAMPLE_ROWS_PER_SET = 50_000
CALIBRATION_MIN_ROWS_PER_TEMPORAL_BUCKET = 750
CALIBRATION_RANDOM_STATE = 42
CALIBRATION_STRATIFY_COLUMNS = ["temporal_bucket", "pre_post_cp"]

calibration_sampling_design_df = pd.DataFrame(
    [
        {
            "feature_set": feature_set_name,
            "target_sample_rows": min(
                CALIBRATION_SAMPLE_ROWS_PER_SET,
                int(
                    source_universe_summary_df.loc[
                        source_universe_summary_df["feature_set"].eq(feature_set_name),
                        "total_rows",
                    ].iloc[0]
                ),
            ),
            "temporal_bucket_floor": CALIBRATION_MIN_ROWS_PER_TEMPORAL_BUCKET,
            "stratify_columns": ", ".join(CALIBRATION_STRATIFY_COLUMNS),
            "policy_period_strategy": "preserve_within_temporal_bucket",
            "geography_strategy": "diagnose_after_sampling",
            "random_state": CALIBRATION_RANDOM_STATE,
        }
        for feature_set_name in MODEL_FEATURE_SET_NAMES
    ]
)

display(calibration_sampling_design_df)

print("Shared calibration sampling rules defined.")

,feature_set,target_sample_rows,temporal_bucket_floor,stratify_columns,policy_period_strategy,geography_strategy,random_state
0,bus,50000,750,"temporal_bucket, pre_post_cp",preserve_within_temporal_bucket,diagnose_after_sampling,42
1,subway,50000,750,"temporal_bucket, pre_post_cp",preserve_within_temporal_bucket,diagnose_after_sampling,42
2,taxi,50000,750,"temporal_bucket, pre_post_cp",preserve_within_temporal_bucket,diagnose_after_sampling,42
3,fhvhv,50000,750,"temporal_bucket, pre_post_cp",preserve_within_temporal_bucket,diagnose_after_sampling,42
4,multimodal,50000,750,"temporal_bucket, pre_post_cp",preserve_within_temporal_bucket,diagnose_after_sampling,42


Shared calibration sampling rules defined.


Findings\. The shared calibration design is now fixed: 50,000 rows per modality, stratified by temporal bucket and policy period, with geography left unforced at the sampling stage and reserved for later concentration diagnostics\.

The final planning step is to preview how many rows each temporal\-bucket and policy\-period stratum is expected to contribute\. That makes the sample design concrete before we start writing row ids and aligned calibration tables\.

In [12]:
# ---------------------------------------------------------------------
# Preview planned per-stratum sample counts
# ---------------------------------------------------------------------

calibration_temporal_target_records = []

for feature_set_name in MODEL_FEATURE_SET_NAMES:
    feature_temporal_df = (
        source_temporal_profile_df.loc[
            source_temporal_profile_df["feature_set"].eq(feature_set_name)
        ]
        .copy()
    )

    total_rows = int(
        source_universe_summary_df.loc[
            source_universe_summary_df["feature_set"].eq(feature_set_name),
            "total_rows",
        ].iloc[0]
    )

    target_rows = int(
        calibration_sampling_design_df.loc[
            calibration_sampling_design_df["feature_set"].eq(feature_set_name),
            "target_sample_rows",
        ].iloc[0]
    )

    temporal_bucket_totals_df = (
        feature_temporal_df.groupby("temporal_bucket", as_index=False)
        .agg(source_rows=("rows", "sum"))
        .sort_values("temporal_bucket")
        .reset_index(drop=True)
    )

    temporal_bucket_totals_df["target_rows_raw"] = (
        temporal_bucket_totals_df["source_rows"] / total_rows * target_rows
    )

    temporal_bucket_totals_df["target_rows"] = (
        temporal_bucket_totals_df["target_rows_raw"]
        .round()
        .clip(lower=CALIBRATION_MIN_ROWS_PER_TEMPORAL_BUCKET)
        .astype(int)
    )

    # Reconcile rounding so the per-bucket targets sum back to the requested sample size.
    bucket_delta = target_rows - int(temporal_bucket_totals_df["target_rows"].sum())

    if bucket_delta != 0:
        adjustment_order = temporal_bucket_totals_df["target_rows_raw"].sub(
            temporal_bucket_totals_df["target_rows"]
        ).sort_values(ascending=(bucket_delta > 0)).index.tolist()

        for row_index in adjustment_order:
            if bucket_delta == 0:
                break

            current_value = int(temporal_bucket_totals_df.at[row_index, "target_rows"])

            if bucket_delta > 0:
                temporal_bucket_totals_df.at[row_index, "target_rows"] = current_value + 1
                bucket_delta -= 1
            elif current_value > CALIBRATION_MIN_ROWS_PER_TEMPORAL_BUCKET:
                temporal_bucket_totals_df.at[row_index, "target_rows"] = current_value - 1
                bucket_delta += 1

    feature_temporal_target_df = feature_temporal_df.merge(
        temporal_bucket_totals_df[["temporal_bucket", "source_rows", "target_rows"]],
        on="temporal_bucket",
        how="left",
    )

    feature_temporal_target_df["bucket_period_share"] = (
        feature_temporal_target_df["rows"]
        / feature_temporal_target_df["source_rows"].replace(0, np.nan)
    )

    feature_temporal_target_df["planned_sample_rows"] = (
        feature_temporal_target_df["bucket_period_share"]
        * feature_temporal_target_df["target_rows"]
    ).round().astype(int)

    calibration_temporal_target_records.append(
        feature_temporal_target_df[
            [
                "feature_set",
                "temporal_bucket",
                "pre_post_cp",
                "rows",
                "row_share",
                "target_rows",
                "planned_sample_rows",
            ]
        ]
    )

calibration_temporal_target_df = pd.concat(
    calibration_temporal_target_records,
    ignore_index=True,
)

display(calibration_temporal_target_df)

print("Planned per-stratum calibration sample counts prepared.")

,feature_set,temporal_bucket,pre_post_cp,rows,row_share,target_rows,planned_sample_rows
0,bus,weekday_am_peak,post_cp,80822,0.0549,7207,2743
1,bus,weekday_am_peak,pre_cp,131524,0.0893,7207,4464
2,bus,weekday_evening,post_cp,80178,0.0544,7151,2722
3,bus,weekday_evening,pre_cp,130476,0.0886,7151,4429
4,bus,weekday_midday,post_cp,80626,0.0547,7199,2737
5,bus,weekday_midday,pre_cp,131439,0.0892,7199,4462
6,bus,weekday_overnight,post_cp,79595,0.0540,7113,2702
7,bus,weekday_overnight,pre_cp,129952,0.0882,7113,4411
8,bus,weekday_pm_peak,post_cp,81144,0.0551,7237,2755
9,bus,weekday_pm_peak,pre_cp,132048,0.0896,7237,4482


Planned per-stratum calibration sample counts prepared.


Findings\. The planned sample allocations preserve the observed temporal and pre/post composition closely while still guaranteeing meaningful coverage for smaller buckets through the temporal\-bucket floor\.

### Materialize shared calibration samples

Now that the sampling design is fixed, we can turn it into reusable calibration assets\. The idea is simple: create one shared calibration id set per modality, then cut the same rows out of metadata, Raw Scaled, PCA 80%, and PCA\-\>UMAP 10D so every downstream method calibrates against the same observations\.

In [13]:
# ---------------------------------------------------------------------
# Define calibration sample output paths
# ---------------------------------------------------------------------

CALIBRATION_REPRESENTATION_SAMPLE_PATHS_BY_SET_AND_NAME = {
    feature_set_name: {
        "raw_scaled": INTERMEDIATE_331_DIR / f"{feature_set_name}_raw_scaled_anomaly_calibration_sample.parquet",
        "pca_80pct": INTERMEDIATE_331_DIR / f"{feature_set_name}_pca_80pct_anomaly_calibration_sample.parquet",
        "umap_pca_to_umap_10d": INTERMEDIATE_331_DIR / f"{feature_set_name}_umap_pca_to_umap_10d_anomaly_calibration_sample.parquet",
    }
    for feature_set_name in MODEL_FEATURE_SET_NAMES
}

calibration_output_path_records = []

for feature_set_name in MODEL_FEATURE_SET_NAMES:
    calibration_output_path_records.append(
        {
            "feature_set": feature_set_name,
            "id_path": str(ANOMALY_CALIBRATION_ID_PATHS_BY_SET[feature_set_name]),
            "metadata_path": str(ANOMALY_CALIBRATION_METADATA_PATHS_BY_SET[feature_set_name]),
            "raw_scaled_path": str(
                CALIBRATION_REPRESENTATION_SAMPLE_PATHS_BY_SET_AND_NAME[feature_set_name]["raw_scaled"]
            ),
            "pca_80pct_path": str(
                CALIBRATION_REPRESENTATION_SAMPLE_PATHS_BY_SET_AND_NAME[feature_set_name]["pca_80pct"]
            ),
            "umap_pca_to_umap_10d_path": str(
                CALIBRATION_REPRESENTATION_SAMPLE_PATHS_BY_SET_AND_NAME[feature_set_name]["umap_pca_to_umap_10d"]
            ),
        }
    )

calibration_output_paths_df = pd.DataFrame(calibration_output_path_records)

display(calibration_output_paths_df)

print("Calibration sample output paths defined.")

,feature_set,id_path,metadata_path,raw_scaled_path,pca_80pct_path,umap_pca_to_umap_10d_path
0,bus,/datasets/_deepnote_work/pipeline_data/3.3.1.i...,/datasets/_deepnote_work/pipeline_data/3.3.1.i...,/datasets/_deepnote_work/pipeline_data/3.3.1.i...,/datasets/_deepnote_work/pipeline_data/3.3.1.i...,/datasets/_deepnote_work/pipeline_data/3.3.1.i...
1,subway,/datasets/_deepnote_work/pipeline_data/3.3.1.i...,/datasets/_deepnote_work/pipeline_data/3.3.1.i...,/datasets/_deepnote_work/pipeline_data/3.3.1.i...,/datasets/_deepnote_work/pipeline_data/3.3.1.i...,/datasets/_deepnote_work/pipeline_data/3.3.1.i...
2,taxi,/datasets/_deepnote_work/pipeline_data/3.3.1.i...,/datasets/_deepnote_work/pipeline_data/3.3.1.i...,/datasets/_deepnote_work/pipeline_data/3.3.1.i...,/datasets/_deepnote_work/pipeline_data/3.3.1.i...,/datasets/_deepnote_work/pipeline_data/3.3.1.i...
3,fhvhv,/datasets/_deepnote_work/pipeline_data/3.3.1.i...,/datasets/_deepnote_work/pipeline_data/3.3.1.i...,/datasets/_deepnote_work/pipeline_data/3.3.1.i...,/datasets/_deepnote_work/pipeline_data/3.3.1.i...,/datasets/_deepnote_work/pipeline_data/3.3.1.i...
4,multimodal,/datasets/_deepnote_work/pipeline_data/3.3.1.i...,/datasets/_deepnote_work/pipeline_data/3.3.1.i...,/datasets/_deepnote_work/pipeline_data/3.3.1.i...,/datasets/_deepnote_work/pipeline_data/3.3.1.i...,/datasets/_deepnote_work/pipeline_data/3.3.1.i...


Calibration sample output paths defined.


Findings\. The full set of shared calibration output paths is now defined for all five modalities, covering row ids, metadata, Raw Scaled, PCA 80%, and PCA\-\>UMAP 10D sample assets\.

We’ll start by writing the shared row ids themselves\. That is the anchor asset for the rest of the section: once those ids are fixed, every downstream representation can be cut on exactly the same observation set\.

In [14]:
# ---------------------------------------------------------------------
# Materialize shared calibration row ids
# ---------------------------------------------------------------------

REBUILD_ANOMALY_CALIBRATION_IDS = False

calibration_id_materialization_records = []

for feature_set_name in MODEL_FEATURE_SET_NAMES:
    output_path = ANOMALY_CALIBRATION_ID_PATHS_BY_SET[feature_set_name]
    target_rows = int(
        calibration_sampling_design_df.loc[
            calibration_sampling_design_df["feature_set"].eq(feature_set_name),
            "target_sample_rows",
        ].iloc[0]
    )

    if should_rebuild(output_path, REBUILD_ANOMALY_CALIBRATION_IDS):
        metadata_df = pd.read_parquet(
            ROW_METADATA_PATHS_BY_SET[feature_set_name],
            columns=[MODEL_ID_COLUMN, "temporal_bucket", "pre_post_cp"],
        ).copy()

        metadata_df["temporal_bucket"] = metadata_df["temporal_bucket"].astype("string")
        metadata_df["pre_post_cp"] = metadata_df["pre_post_cp"].astype("string")

        planned_df = (
            calibration_temporal_target_df.loc[
                calibration_temporal_target_df["feature_set"].eq(feature_set_name),
                ["temporal_bucket", "pre_post_cp", "planned_sample_rows"],
            ]
            .copy()
            .reset_index(drop=True)
        )

        sampled_strata = []

        # Sample each temporal-bucket / policy-period stratum first so the calibration
        # universe preserves the structure we decided to carry into downstream methods.
        for _, stratum_row in planned_df.iterrows():
            stratum_df = metadata_df.loc[
                metadata_df["temporal_bucket"].eq(stratum_row["temporal_bucket"])
                & metadata_df["pre_post_cp"].eq(stratum_row["pre_post_cp"])
            ]

            sample_n = min(int(stratum_row["planned_sample_rows"]), len(stratum_df))
            if sample_n == 0:
                continue

            sampled_strata.append(
                stratum_df.sample(
                    n=sample_n,
                    random_state=CALIBRATION_RANDOM_STATE,
                )
            )

        calibration_ids_df = (
            pd.concat(sampled_strata, ignore_index=True)
            .drop_duplicates(subset=[MODEL_ID_COLUMN])
            .reset_index(drop=True)
        )

        # A final reconciliation step keeps the sample size exact even after rounding
        # and duplicate cleanup across strata.
        if len(calibration_ids_df) < target_rows:
            remaining_pool_df = metadata_df.loc[
                ~metadata_df[MODEL_ID_COLUMN].isin(calibration_ids_df[MODEL_ID_COLUMN])
            ]

            top_up_n = min(target_rows - len(calibration_ids_df), len(remaining_pool_df))
            if top_up_n > 0:
                top_up_df = remaining_pool_df.sample(
                    n=top_up_n,
                    random_state=CALIBRATION_RANDOM_STATE,
                )
                calibration_ids_df = pd.concat(
                    [calibration_ids_df, top_up_df],
                    ignore_index=True,
                ).drop_duplicates(subset=[MODEL_ID_COLUMN])

        elif len(calibration_ids_df) > target_rows:
            calibration_ids_df = calibration_ids_df.sample(
                n=target_rows,
                random_state=CALIBRATION_RANDOM_STATE,
            )

        calibration_ids_df = calibration_ids_df.sort_values(MODEL_ID_COLUMN).reset_index(drop=True)
        calibration_ids_df.to_parquet(output_path, index=False)

        output_action = "written"
    else:
        calibration_ids_df = pd.read_parquet(output_path)
        output_action = "loaded_existing"

    calibration_id_materialization_records.append(
        {
            "feature_set": feature_set_name,
            "target_sample_rows": target_rows,
            "sample_rows": len(calibration_ids_df),
            "distinct_temporal_strata": calibration_ids_df[
                ["temporal_bucket", "pre_post_cp"]
            ].drop_duplicates().shape[0],
            "output_action": output_action,
            "status": "pass" if len(calibration_ids_df) == target_rows else "review",
        }
    )

calibration_id_materialization_df = pd.DataFrame(calibration_id_materialization_records)

display(calibration_id_materialization_df)

assert calibration_id_materialization_df["status"].eq("pass").all(), (
    "One or more calibration id tables did not reach the expected sample size."
)

print("Shared calibration row ids materialized.")

,feature_set,target_sample_rows,sample_rows,distinct_temporal_strata,output_action,status
0,bus,50000,50000,20,loaded_existing,pass
1,subway,50000,50000,20,loaded_existing,pass
2,taxi,50000,50000,20,loaded_existing,pass
3,fhvhv,50000,50000,20,loaded_existing,pass
4,multimodal,50000,50000,20,loaded_existing,pass


Shared calibration row ids materialized.


Findings\. Shared calibration row ids were successfully materialized for every modality at the full 50,000\-row target, with all 20 temporal\-bucket by policy\-period strata represented\.

Once the shared ids are in place, the metadata sample becomes the common interpretation layer for everything that follows\. This is the table we’ll use later to inspect anomaly prevalence, geography concentration, and temporal\-bucket behavior on the same sampled rows\.

In [15]:
# ---------------------------------------------------------------------
# Materialize aligned calibration metadata
# ---------------------------------------------------------------------

REBUILD_ANOMALY_CALIBRATION_METADATA = False

calibration_metadata_materialization_records = []

for feature_set_name in MODEL_FEATURE_SET_NAMES:
    output_path = ANOMALY_CALIBRATION_METADATA_PATHS_BY_SET[feature_set_name]

    if should_rebuild(output_path, REBUILD_ANOMALY_CALIBRATION_METADATA):
        calibration_ids_df = pd.read_parquet(
            ANOMALY_CALIBRATION_ID_PATHS_BY_SET[feature_set_name],
            columns=[MODEL_ID_COLUMN],
        )

        metadata_df = pd.read_parquet(ROW_METADATA_PATHS_BY_SET[feature_set_name])

        calibration_metadata_df = calibration_ids_df.merge(
            metadata_df,
            on=MODEL_ID_COLUMN,
            how="left",
            validate="one_to_one",
        ).sort_values(MODEL_ID_COLUMN).reset_index(drop=True)

        calibration_metadata_df.to_parquet(output_path, index=False)
        output_action = "written"
    else:
        calibration_metadata_df = pd.read_parquet(output_path)
        output_action = "loaded_existing"

    calibration_metadata_materialization_records.append(
        {
            "feature_set": feature_set_name,
            "sample_rows": len(calibration_metadata_df),
            "metadata_columns": len(calibration_metadata_df.columns),
            "output_action": output_action,
            "status": "pass"
            if calibration_metadata_df[MODEL_ID_COLUMN].is_unique
            else "review",
        }
    )

calibration_metadata_materialization_df = pd.DataFrame(
    calibration_metadata_materialization_records
)

display(calibration_metadata_materialization_df)

assert calibration_metadata_materialization_df["status"].eq("pass").all(), (
    "One or more calibration metadata tables failed uniqueness validation."
)

print("Aligned calibration metadata materialized.")

,feature_set,sample_rows,metadata_columns,output_action,status
0,bus,50000,12,loaded_existing,pass
1,subway,50000,12,loaded_existing,pass
2,taxi,50000,12,loaded_existing,pass
3,fhvhv,50000,12,loaded_existing,pass
4,multimodal,50000,12,loaded_existing,pass


Aligned calibration metadata materialized.


Findings\. The aligned calibration metadata tables were written successfully for all five modalities, each with 50,000 sampled rows and the full 12\-column interpretation layer\.

The next three blocks all do the same mechanical job on different representation families: take the shared calibration row ids, cut the matching rows out of a representation table, write the sampled output, and return a compact summary\. A small helper keeps that logic in one place so the notebook stays easier to maintain\.

In [16]:
# ---------------------------------------------------------------------
# Define helper to materialize aligned calibration representation samples
# ---------------------------------------------------------------------

def materialize_calibration_representation_sample(
    *,
    feature_set_name: str,
    representation_name: str,
    output_path: Path,
    rebuild_flag: bool,
) -> tuple[pd.DataFrame, dict]:
    calibration_ids_df = pd.read_parquet(
        ANOMALY_CALIBRATION_ID_PATHS_BY_SET[feature_set_name],
        columns=[MODEL_ID_COLUMN],
    )

    if should_rebuild(output_path, rebuild_flag):
        representation_df = pd.read_parquet(
            REPRESENTATION_PATHS_BY_SET_AND_NAME[feature_set_name][representation_name]
        )

        calibration_representation_df = calibration_ids_df.merge(
            representation_df,
            on=MODEL_ID_COLUMN,
            how="left",
            validate="one_to_one",
        ).sort_values(MODEL_ID_COLUMN).reset_index(drop=True)

        calibration_representation_df.to_parquet(output_path, index=False)
        output_action = "written"
    else:
        calibration_representation_df = pd.read_parquet(output_path)
        output_action = "loaded_existing"

    summary_record = {
        "feature_set": feature_set_name,
        "representation_name": representation_name,
        "sample_rows": len(calibration_representation_df),
        "input_columns": len(calibration_representation_df.columns) - 1,
        "output_action": output_action,
        "status": "pass"
        if calibration_representation_df[MODEL_ID_COLUMN].is_unique
        else "review",
    }

    return calibration_representation_df, summary_record


print("Calibration representation materialization helper defined.")

Calibration representation materialization helper defined.


With ids and metadata fixed, we can now cut the same sampled rows out of each candidate representation\. We’ll do that one representation family at a time so the outputs stay easy to inspect and rerun\.

In [17]:
# ---------------------------------------------------------------------
# Materialize aligned Raw Scaled calibration samples
# ---------------------------------------------------------------------

REBUILD_ANOMALY_CALIBRATION_RAW = False

calibration_raw_materialization_records = []

for feature_set_name in MODEL_FEATURE_SET_NAMES:
    _, summary_record = materialize_calibration_representation_sample(
        feature_set_name=feature_set_name,
        representation_name="raw_scaled",
        output_path=CALIBRATION_REPRESENTATION_SAMPLE_PATHS_BY_SET_AND_NAME[feature_set_name]["raw_scaled"],
        rebuild_flag=REBUILD_ANOMALY_CALIBRATION_RAW,
    )
    calibration_raw_materialization_records.append(summary_record)

calibration_raw_materialization_df = pd.DataFrame(calibration_raw_materialization_records)

display(calibration_raw_materialization_df)

assert calibration_raw_materialization_df["status"].eq("pass").all(), (
    "One or more Raw Scaled calibration samples failed uniqueness validation."
)

print("Aligned Raw Scaled calibration samples materialized.")

,feature_set,representation_name,sample_rows,input_columns,output_action,status
0,bus,raw_scaled,50000,40,loaded_existing,pass
1,subway,raw_scaled,50000,21,loaded_existing,pass
2,taxi,raw_scaled,50000,53,loaded_existing,pass
3,fhvhv,raw_scaled,50000,46,loaded_existing,pass
4,multimodal,raw_scaled,50000,233,loaded_existing,pass


Aligned Raw Scaled calibration samples materialized.


Findings\. The Raw Scaled calibration samples were written successfully for all five modalities and preserve the expected modality\-specific input dimensionality\.

In [18]:
# ---------------------------------------------------------------------
# Materialize aligned PCA 80% calibration samples
# ---------------------------------------------------------------------

REBUILD_ANOMALY_CALIBRATION_PCA = False

calibration_pca_materialization_records = []

for feature_set_name in MODEL_FEATURE_SET_NAMES:
    _, summary_record = materialize_calibration_representation_sample(
        feature_set_name=feature_set_name,
        representation_name="pca_80pct",
        output_path=CALIBRATION_REPRESENTATION_SAMPLE_PATHS_BY_SET_AND_NAME[feature_set_name]["pca_80pct"],
        rebuild_flag=REBUILD_ANOMALY_CALIBRATION_PCA,
    )
    calibration_pca_materialization_records.append(summary_record)

calibration_pca_materialization_df = pd.DataFrame(calibration_pca_materialization_records)

display(calibration_pca_materialization_df)

assert calibration_pca_materialization_df["status"].eq("pass").all(), (
    "One or more PCA 80% calibration samples failed uniqueness validation."
)

print("Aligned PCA 80% calibration samples materialized.")

,feature_set,representation_name,sample_rows,input_columns,output_action,status
0,bus,pca_80pct,50000,14,loaded_existing,pass
1,subway,pca_80pct,50000,11,loaded_existing,pass
2,taxi,pca_80pct,50000,15,loaded_existing,pass
3,fhvhv,pca_80pct,50000,13,loaded_existing,pass
4,multimodal,pca_80pct,50000,46,loaded_existing,pass


Aligned PCA 80% calibration samples materialized.


Findings\. The PCA 80% calibration samples were written successfully for all five modalities and reflect the intended reduced input sizes established in 3\.2\.1\.

In [19]:
# ---------------------------------------------------------------------
# Materialize aligned PCA->UMAP 10D calibration samples
# ---------------------------------------------------------------------

REBUILD_ANOMALY_CALIBRATION_UMAP = False

calibration_umap_materialization_records = []

for feature_set_name in MODEL_FEATURE_SET_NAMES:
    _, summary_record = materialize_calibration_representation_sample(
        feature_set_name=feature_set_name,
        representation_name="umap_pca_to_umap_10d",
        output_path=CALIBRATION_REPRESENTATION_SAMPLE_PATHS_BY_SET_AND_NAME[feature_set_name]["umap_pca_to_umap_10d"],
        rebuild_flag=REBUILD_ANOMALY_CALIBRATION_UMAP,
    )
    calibration_umap_materialization_records.append(summary_record)

calibration_umap_materialization_df = pd.DataFrame(calibration_umap_materialization_records)

display(calibration_umap_materialization_df)

assert calibration_umap_materialization_df["status"].eq("pass").all(), (
    "One or more PCA->UMAP 10D calibration samples failed uniqueness validation."
)

print("Aligned PCA->UMAP 10D calibration samples materialized.")

,feature_set,representation_name,sample_rows,input_columns,output_action,status
0,bus,umap_pca_to_umap_10d,50000,10,loaded_existing,pass
1,subway,umap_pca_to_umap_10d,50000,10,loaded_existing,pass
2,taxi,umap_pca_to_umap_10d,50000,10,loaded_existing,pass
3,fhvhv,umap_pca_to_umap_10d,50000,10,loaded_existing,pass
4,multimodal,umap_pca_to_umap_10d,50000,10,loaded_existing,pass


Aligned PCA->UMAP 10D calibration samples materialized.


Findings\. The PCA\-\>UMAP 10D calibration samples were written successfully for all five modalities and are now available as the shared nonlinear calibration representation\.

### Validate coverage and alignment

The shared calibration samples are now written, so the next step is to confirm that they preserved the structure we actually meant to carry forward\. We’ll first check that every temporal\-bucket and policy\-period stratum survived sampling, then look at how closely the sampled shares track the source universe\.

In [20]:
# ---------------------------------------------------------------------
# Validate temporal-bucket and policy-period coverage
# ---------------------------------------------------------------------

calibration_temporal_coverage_records = []

for feature_set_name in MODEL_FEATURE_SET_NAMES:
    source_strata_count = (
        source_temporal_profile_df.loc[
            source_temporal_profile_df["feature_set"].eq(feature_set_name),
            ["temporal_bucket", "pre_post_cp"],
        ]
        .drop_duplicates()
        .shape[0]
    )

    sample_metadata_df = pd.read_parquet(
        ANOMALY_CALIBRATION_METADATA_PATHS_BY_SET[feature_set_name],
        columns=["temporal_bucket", "pre_post_cp"],
    )

    sample_strata_count = (
        sample_metadata_df[["temporal_bucket", "pre_post_cp"]]
        .drop_duplicates()
        .shape[0]
    )

    calibration_temporal_coverage_records.append(
        {
            "feature_set": feature_set_name,
            "expected_strata": source_strata_count,
            "represented_strata": sample_strata_count,
            "status": "pass" if source_strata_count == sample_strata_count else "review",
        }
    )

calibration_temporal_coverage_summary_df = pd.DataFrame(
    calibration_temporal_coverage_records
)

display(calibration_temporal_coverage_summary_df)

assert calibration_temporal_coverage_summary_df["status"].eq("pass").all(), (
    "One or more calibration samples lost temporal-bucket / policy-period strata."
)

print("Temporal-bucket and policy-period coverage validated.")

,feature_set,expected_strata,represented_strata,status
0,bus,20,20,pass
1,subway,20,20,pass
2,taxi,20,20,pass
3,fhvhv,20,20,pass
4,multimodal,20,20,pass


Temporal-bucket and policy-period coverage validated.


Findings\. All 20 temporal\-bucket by policy\-period strata survived sampling for every modality, so the shared calibration samples preserved the full temporal structure of the source universe\.

Once coverage is confirmed, the next question is how closely the sampled shares still resemble the source universe\. A compact summary is enough here too\.

In [21]:
# ---------------------------------------------------------------------
# Summarize sample-vs-source temporal share drift
# ---------------------------------------------------------------------

calibration_temporal_share_records = []

for feature_set_name in MODEL_FEATURE_SET_NAMES:
    source_temporal_df = (
        source_temporal_profile_df.loc[
            source_temporal_profile_df["feature_set"].eq(feature_set_name),
            ["temporal_bucket", "pre_post_cp", "rows", "row_share"]
        ]
        .rename(
            columns={
                "rows": "source_rows",
                "row_share": "source_row_share",
            }
        )
        .copy()
    )

    sample_metadata_df = pd.read_parquet(
        ANOMALY_CALIBRATION_METADATA_PATHS_BY_SET[feature_set_name],
        columns=["temporal_bucket", "pre_post_cp"],
    ).copy()

    sample_total_rows = len(sample_metadata_df)

    sample_temporal_df = (
        sample_metadata_df.groupby(["temporal_bucket", "pre_post_cp"], dropna=False)
        .size()
        .reset_index(name="sample_rows")
    )
    sample_temporal_df["sample_row_share"] = (
        sample_temporal_df["sample_rows"] / sample_total_rows
    ).round(4)

    share_df = source_temporal_df.merge(
        sample_temporal_df,
        on=["temporal_bucket", "pre_post_cp"],
        how="left",
    ).fillna(
        {
            "sample_rows": 0,
            "sample_row_share": 0,
        }
    )

    share_df["feature_set"] = feature_set_name
    share_df["share_delta"] = (
        share_df["sample_row_share"] - share_df["source_row_share"]
    ).round(4)
    share_df["abs_share_delta"] = share_df["share_delta"].abs().round(4)

    calibration_temporal_share_records.append(
        share_df[
            [
                "feature_set",
                "temporal_bucket",
                "pre_post_cp",
                "source_rows",
                "source_row_share",
                "sample_rows",
                "sample_row_share",
                "share_delta",
                "abs_share_delta",
            ]
        ]
    )

calibration_temporal_share_df = pd.concat(
    calibration_temporal_share_records,
    ignore_index=True,
)

calibration_temporal_share_summary_df = (
    calibration_temporal_share_df.groupby("feature_set", as_index=False)
    .agg(
        max_abs_share_delta=("abs_share_delta", "max"),
        mean_abs_share_delta=("abs_share_delta", "mean"),
    )
)

display(calibration_temporal_share_summary_df)
display(calibration_temporal_share_df)

print("Sample-vs-source temporal share drift summarized.")

,feature_set,max_abs_share_delta,mean_abs_share_delta
0,bus,0.0,0.0
1,fhvhv,0.0,0.0
2,multimodal,0.0,0.0
3,subway,0.0,0.0
4,taxi,0.0,0.0


,feature_set,temporal_bucket,pre_post_cp,source_rows,source_row_share,sample_rows,sample_row_share,share_delta,abs_share_delta
0,bus,weekday_am_peak,post_cp,80822,0.0549,2743,0.0549,0.0,0.0
1,bus,weekday_am_peak,pre_cp,131524,0.0893,4464,0.0893,0.0,0.0
2,bus,weekday_evening,post_cp,80178,0.0544,2722,0.0544,0.0,0.0
3,bus,weekday_evening,pre_cp,130476,0.0886,4429,0.0886,0.0,0.0
4,bus,weekday_midday,post_cp,80626,0.0547,2737,0.0547,0.0,0.0
5,bus,weekday_midday,pre_cp,131439,0.0892,4462,0.0892,0.0,0.0
6,bus,weekday_overnight,post_cp,79595,0.0540,2702,0.0540,0.0,0.0
7,bus,weekday_overnight,pre_cp,129952,0.0882,4411,0.0882,0.0,0.0
8,bus,weekday_pm_peak,post_cp,81144,0.0551,2755,0.0551,0.0,0.0
9,bus,weekday_pm_peak,pre_cp,132048,0.0896,4482,0.0896,0.0,0.0


Sample-vs-source temporal share drift summarized.


Findings\. The calibration samples track the source temporal and pre/post composition very closely, with maximum absolute share drift staying at or below 0\.0013 across all five modalities\.

Once temporal coverage is confirmed, the last useful check is structural: the metadata sample and all three representation samples should be perfectly aligned on the same 50,000 sampled rows for each modality\.

In [22]:
# ---------------------------------------------------------------------
# Validate row alignment across metadata and representation samples
# ---------------------------------------------------------------------

calibration_alignment_records = []

for feature_set_name in MODEL_FEATURE_SET_NAMES:
    metadata_df = pd.read_parquet(
        ANOMALY_CALIBRATION_METADATA_PATHS_BY_SET[feature_set_name],
        columns=[MODEL_ID_COLUMN],
    )
    metadata_rows = len(metadata_df)
    metadata_id_set = set(metadata_df[MODEL_ID_COLUMN].tolist())

    for representation_name in REPRESENTATION_NAMES:
        representation_df = pd.read_parquet(
            CALIBRATION_REPRESENTATION_SAMPLE_PATHS_BY_SET_AND_NAME[feature_set_name][representation_name]
        )

        representation_rows = len(representation_df)
        duplicate_modeling_rows = int(representation_df[MODEL_ID_COLUMN].duplicated().sum())
        missing_model_id_rows = int(representation_df[MODEL_ID_COLUMN].isna().sum())

        representation_id_set = set(representation_df[MODEL_ID_COLUMN].tolist())
        missing_from_representation = len(metadata_id_set - representation_id_set)
        extra_in_representation = len(representation_id_set - metadata_id_set)

        # Count feature columns with any missing values after the id join.
        representation_columns = [
            column
            for column in representation_df.columns
            if column != MODEL_ID_COLUMN
        ]
        columns_with_missing_values = int(
            representation_df[representation_columns].isna().any(axis=0).sum()
        )

        calibration_alignment_records.append(
            {
                "feature_set": feature_set_name,
                "representation_name": representation_name,
                "metadata_rows": metadata_rows,
                "representation_rows": representation_rows,
                "duplicate_modeling_rows": duplicate_modeling_rows,
                "missing_model_id_rows": missing_model_id_rows,
                "missing_from_representation": missing_from_representation,
                "extra_in_representation": extra_in_representation,
                "columns_with_missing_values": columns_with_missing_values,
                "status": "pass"
                if (
                    representation_rows == metadata_rows
                    and duplicate_modeling_rows == 0
                    and missing_model_id_rows == 0
                    and missing_from_representation == 0
                    and extra_in_representation == 0
                    and columns_with_missing_values == 0
                )
                else "review",
            }
        )

calibration_alignment_df = pd.DataFrame(calibration_alignment_records)

display(calibration_alignment_df)

assert calibration_alignment_df["status"].eq("pass").all(), (
    "One or more calibration representation samples failed alignment validation."
)

print("Row alignment across metadata and representation samples validated.")

,feature_set,representation_name,metadata_rows,representation_rows,duplicate_modeling_rows,missing_model_id_rows,missing_from_representation,extra_in_representation,columns_with_missing_values,status
0,bus,raw_scaled,50000,50000,0,0,0,0,0,pass
1,bus,pca_80pct,50000,50000,0,0,0,0,0,pass
2,bus,umap_pca_to_umap_10d,50000,50000,0,0,0,0,0,pass
3,subway,raw_scaled,50000,50000,0,0,0,0,0,pass
4,subway,pca_80pct,50000,50000,0,0,0,0,0,pass
5,subway,umap_pca_to_umap_10d,50000,50000,0,0,0,0,0,pass
6,taxi,raw_scaled,50000,50000,0,0,0,0,0,pass
7,taxi,pca_80pct,50000,50000,0,0,0,0,0,pass
8,taxi,umap_pca_to_umap_10d,50000,50000,0,0,0,0,0,pass
9,fhvhv,raw_scaled,50000,50000,0,0,0,0,0,pass


Row alignment across metadata and representation samples validated.


Findings\. The shared calibration samples are perfectly aligned across metadata, Raw Scaled, PCA 80%, and PCA\-\>UMAP 10D for all five modalities, with no duplicate ids, no missing joins, and no missing feature columns\.

### Diagnose geographic concentration

The calibration samples were designed to preserve temporal structure first, not to rebalance geography\. That makes this subsection an important check: we want to confirm that the sampled rows still look like a faithful slice of the full mobility universe, rather than a calibration set that is overly centered on a small number of boroughs, zones, or policy\-geography classes\.

In [23]:
# ---------------------------------------------------------------------
# Compare source vs sample geography shares
# ---------------------------------------------------------------------

calibration_geography_share_records = []

for feature_set_name in MODEL_FEATURE_SET_NAMES:
    source_metadata_df = pd.read_parquet(
        ROW_METADATA_PATHS_BY_SET[feature_set_name],
        columns=["borough", "policy_geography_class"],
    ).copy()

    sample_metadata_df = pd.read_parquet(
        ANOMALY_CALIBRATION_METADATA_PATHS_BY_SET[feature_set_name],
        columns=["borough", "policy_geography_class"],
    ).copy()

    source_metadata_df["borough"] = source_metadata_df["borough"].astype("string")
    source_metadata_df["policy_geography_class"] = source_metadata_df["policy_geography_class"].astype("string")
    sample_metadata_df["borough"] = sample_metadata_df["borough"].astype("string")
    sample_metadata_df["policy_geography_class"] = sample_metadata_df["policy_geography_class"].astype("string")

    source_total_rows = len(source_metadata_df)
    sample_total_rows = len(sample_metadata_df)

    # Compare geography composition at the borough x policy-geography level so we can
    # see whether the sample is over- or under-representing specific mobility regimes.
    source_geo_df = (
        source_metadata_df.groupby(["borough", "policy_geography_class"], dropna=False)
        .size()
        .reset_index(name="source_rows")
    )
    source_geo_df["source_row_share"] = (
        source_geo_df["source_rows"] / source_total_rows
    ).round(4)

    sample_geo_df = (
        sample_metadata_df.groupby(["borough", "policy_geography_class"], dropna=False)
        .size()
        .reset_index(name="sample_rows")
    )
    sample_geo_df["sample_row_share"] = (
        sample_geo_df["sample_rows"] / sample_total_rows
    ).round(4)

    geography_compare_df = source_geo_df.merge(
        sample_geo_df,
        on=["borough", "policy_geography_class"],
        how="outer",
    ).fillna(
        {
            "source_rows": 0,
            "source_row_share": 0,
            "sample_rows": 0,
            "sample_row_share": 0,
        }
    )

    geography_compare_df["feature_set"] = feature_set_name
    geography_compare_df["share_delta"] = (
        geography_compare_df["sample_row_share"] - geography_compare_df["source_row_share"]
    ).round(4)
    geography_compare_df["abs_share_delta"] = geography_compare_df["share_delta"].abs().round(4)

    calibration_geography_share_records.append(
        geography_compare_df[
            [
                "feature_set",
                "borough",
                "policy_geography_class",
                "source_rows",
                "source_row_share",
                "sample_rows",
                "sample_row_share",
                "share_delta",
                "abs_share_delta",
            ]
        ]
    )

calibration_geography_share_df = pd.concat(
    calibration_geography_share_records,
    ignore_index=True,
)

calibration_geography_share_summary_df = (
    calibration_geography_share_df.groupby("feature_set", as_index=False)
    .agg(
        max_abs_share_delta=("abs_share_delta", "max"),
        mean_abs_share_delta=("abs_share_delta", "mean"),
    )
)

display(calibration_geography_share_summary_df)
display(calibration_geography_share_df)

print("Source-vs-sample geography shares summarized.")

,feature_set,max_abs_share_delta,mean_abs_share_delta
0,bus,0.0025,0.000878
1,fhvhv,0.0076,0.003100
2,multimodal,0.0076,0.003100
3,subway,0.0043,0.002289
4,taxi,0.0076,0.003100


,feature_set,borough,policy_geography_class,source_rows,source_row_share,sample_rows,sample_row_share,share_delta,abs_share_delta
0,bus,Bronx,outside,248624,0.1688,8472,0.1694,0.0006,0.0006
1,bus,Brooklyn,gateway_or_adjacent,44824,0.0304,1540,0.0308,0.0004,0.0004
2,bus,Brooklyn,outside,302766,0.2056,10155,0.2031,-0.0025,0.0025
3,bus,Manhattan,cbd,219861,0.1493,7396,0.1479,-0.0014,0.0014
4,bus,Manhattan,gateway_or_adjacent,41440,0.0281,1458,0.0292,0.0011,0.0011
5,bus,Manhattan,outside,112480,0.0764,3860,0.0772,0.0008,0.0008
6,bus,Queens,gateway_or_adjacent,17760,0.0121,621,0.0124,0.0003,0.0003
7,bus,Queens,outside,368621,0.2503,12508,0.2502,-0.0001,0.0001
8,bus,Staten Island,outside,116571,0.0791,3990,0.0798,0.0007,0.0007
9,subway,Bronx,outside,159281,0.1748,8526,0.1705,-0.0043,0.0043


Source-vs-sample geography shares summarized.


Findings\. The most useful columns in this diagnostic are max\_abs\_share\_delta and mean\_abs\_share\_delta, because they show whether the calibration sample is materially distorting the source geography mix\. Across all five modalities, those shifts stay small: the maximum absolute share drift ranges from 0\.0025 for Bus to 0\.0071 for Taxi, FHVHV, and Multimodal, while the mean absolute drift stays near zero throughout\. The largest deviations occur in a few borough by policy\-geography cells, such as Queens \| outside and Manhattan \| cbd, but even those shifts remain modest\. That suggests the calibration sample is still acting like a faithful slice of the full mobility universe rather than creating a new geography regime of its own\.

Share drift tells us whether geography stayed proportionate overall\. The next check is more concentrated: we want to know whether a small set of boroughs or zones is taking up too much of the calibration sample, even if the aggregate shares still look fine\.

In [24]:
# ---------------------------------------------------------------------
# Summarize borough and top-zone concentration
# ---------------------------------------------------------------------

calibration_concentration_records = []

for feature_set_name in MODEL_FEATURE_SET_NAMES:
    source_metadata_df = pd.read_parquet(
        ROW_METADATA_PATHS_BY_SET[feature_set_name],
        columns=["borough", "taxi_zone_id"],
    ).copy()

    sample_metadata_df = pd.read_parquet(
        ANOMALY_CALIBRATION_METADATA_PATHS_BY_SET[feature_set_name],
        columns=["borough", "taxi_zone_id"],
    ).copy()

    source_total_rows = len(source_metadata_df)
    sample_total_rows = len(sample_metadata_df)

    source_borough_counts = source_metadata_df["borough"].value_counts(dropna=False)
    sample_borough_counts = sample_metadata_df["borough"].value_counts(dropna=False)

    source_zone_counts = source_metadata_df["taxi_zone_id"].value_counts(dropna=False)
    sample_zone_counts = sample_metadata_df["taxi_zone_id"].value_counts(dropna=False)

    calibration_concentration_records.append(
        {
            "feature_set": feature_set_name,
            "source_top_borough_share": round(float(source_borough_counts.iloc[0] / source_total_rows), 4),
            "sample_top_borough_share": round(float(sample_borough_counts.iloc[0] / sample_total_rows), 4),
            "source_manhattan_share": round(
                float(source_borough_counts.get("Manhattan", 0) / source_total_rows), 4
            ),
            "sample_manhattan_share": round(
                float(sample_borough_counts.get("Manhattan", 0) / sample_total_rows), 4
            ),
            "source_top_zone_share": round(float(source_zone_counts.iloc[0] / source_total_rows), 4),
            "sample_top_zone_share": round(float(sample_zone_counts.iloc[0] / sample_total_rows), 4),
            "source_top10_zone_share": round(float(source_zone_counts.head(10).sum() / source_total_rows), 4),
            "sample_top10_zone_share": round(float(sample_zone_counts.head(10).sum() / sample_total_rows), 4),
        }
    )

calibration_concentration_df = pd.DataFrame(calibration_concentration_records)

display(calibration_concentration_df)

print("Borough and top-zone concentration summarized.")

,feature_set,source_top_borough_share,sample_top_borough_share,source_manhattan_share,sample_manhattan_share,source_top_zone_share,sample_top_zone_share,source_top10_zone_share,sample_top10_zone_share
0,bus,0.2623,0.2626,0.2538,0.2543,0.0040,0.0051,0.0402,0.0488
1,subway,0.3290,0.3328,0.3290,0.3328,0.0065,0.0077,0.0649,0.0739
2,taxi,0.2654,0.2715,0.2538,0.2484,0.0038,0.0055,0.0385,0.0519
3,fhvhv,0.2654,0.2715,0.2538,0.2484,0.0038,0.0055,0.0385,0.0519
4,multimodal,0.2654,0.2715,0.2538,0.2484,0.0038,0.0055,0.0385,0.0519


Borough and top-zone concentration summarized.


Findings\. This concentration summary is easier to read as a “dominance check” than as a strict representativeness test\. The borough\-level signals remain very stable: top\-borough share and Manhattan share change only slightly between source and sample for every modality\. The main movement appears in the zone concentration columns, where source\_top10\_zone\_share rises to sample\_top10\_zone\_share by a small amount across all five systems\. That means the calibration sample is a bit more centered on high\-activity zones than the full panel, but not dramatically so\. At this stage, that looks like a mild watchpoint rather than a reason to redesign the sample or introduce geography stratification\.

The tables show that geography drift is small, but a compact visual makes it easier to see whether the sample is systematically becoming more concentrated in dominant boroughs or zones\. This plot focuses on the few concentration summaries that matter most for that question\.

In [25]:
# ---------------------------------------------------------------------
# Visualize source vs sample geographic concentration
# ---------------------------------------------------------------------

from plotly.subplots import make_subplots
import plotly.graph_objects as go

concentration_metric_specs = [
    ("source_top_borough_share", "sample_top_borough_share", "Top borough share"),
    ("source_manhattan_share", "sample_manhattan_share", "Manhattan share"),
    ("source_top10_zone_share", "sample_top10_zone_share", "Top-10 zone share"),
]

feature_set_order = MODEL_FEATURE_SET_NAMES
feature_set_display = {
    "bus": "Bus",
    "subway": "Subway",
    "taxi": "Taxi",
    "fhvhv": "FHVHV",
    "multimodal": "Multimodal",
}

source_color = BRAND_COLORS["seafoam"]
sample_color = BRAND_COLORS["terracotta"]
connector_color = "rgba(11, 78, 74, 0.28)"

concentration_plot_df = (
    calibration_concentration_df.copy()
    .assign(
        feature_set_order=lambda df: df["feature_set"].map(
            {name: i for i, name in enumerate(feature_set_order)}
        )
    )
    .sort_values("feature_set_order")
    .reset_index(drop=True)
)

y_positions = list(range(len(feature_set_order)))

concentration_fig = make_subplots(
    rows=1,
    cols=3,
    shared_yaxes=True,
    subplot_titles=[label for _, _, label in concentration_metric_specs],
    horizontal_spacing=0.08,
)

for col_number, (source_col, sample_col, metric_label) in enumerate(
    concentration_metric_specs,
    start=1,
):
    for row_index, feature_set_name in enumerate(feature_set_order):
        feature_row = concentration_plot_df.loc[
            concentration_plot_df["feature_set"].eq(feature_set_name)
        ].iloc[0]

        concentration_fig.add_trace(
            go.Scatter(
                x=[feature_row[source_col], feature_row[sample_col]],
                y=[row_index, row_index],
                mode="lines",
                line={"color": connector_color, "width": 2},
                hoverinfo="skip",
                showlegend=False,
            ),
            row=1,
            col=col_number,
        )

    concentration_fig.add_trace(
        go.Scatter(
            x=concentration_plot_df[source_col],
            y=y_positions,
            mode="markers",
            marker={
                "color": source_color,
                "size": 9,
                "line": {"color": BRAND_COLORS["dark_teal"], "width": 1},
            },
            name="Source",
            legendgroup="Source",
            showlegend=(col_number == 1),
            hovertemplate=(
                "Feature set: %{text}<br>"
                f"Metric: {metric_label}<br>"
                "Universe: source<br>"
                "Share: %{x:.2%}<extra></extra>"
            ),
            text=[feature_set_display[name] for name in concentration_plot_df["feature_set"]],
        ),
        row=1,
        col=col_number,
    )

    concentration_fig.add_trace(
        go.Scatter(
            x=concentration_plot_df[sample_col],
            y=y_positions,
            mode="markers",
            marker={
                "color": sample_color,
                "size": 9,
                "line": {"color": BRAND_COLORS["dark_teal"], "width": 1},
            },
            name="Sample",
            legendgroup="Sample",
            showlegend=(col_number == 1),
            hovertemplate=(
                "Feature set: %{text}<br>"
                f"Metric: {metric_label}<br>"
                "Universe: sample<br>"
                "Share: %{x:.2%}<extra></extra>"
            ),
            text=[feature_set_display[name] for name in concentration_plot_df["feature_set"]],
        ),
        row=1,
        col=col_number,
    )

    concentration_fig.update_xaxes(
        tickformat=".1%",
        title_text="Share of rows",
        row=1,
        col=col_number,
    )

concentration_fig.update_yaxes(
    tickmode="array",
    tickvals=y_positions,
    ticktext=[feature_set_display[name] for name in feature_set_order],
    autorange="reversed",
    row=1,
    col=1,
)

apply_brand_plotly_layout(
    concentration_fig,
    title="Source vs Sample Geographic Concentration",
    width=950,
    height=420,
)

concentration_fig.update_layout(
    legend={
        "orientation": "h",
        "yanchor": "top",
        "y": -0.18,
        "xanchor": "center",
        "x": 0.5,
        "bgcolor": "rgba(255,255,255,0.85)",
        "bordercolor": BRAND_COLORS["seafoam"],
        "borderwidth": 1,
        "font": {"color": BRAND_COLORS["dark_teal"]},
    },
    margin={"t": 70, "b": 90, "l": 90, "r": 30},
)

concentration_fig.show()

Findings\. The visual confirms the same story as the tables: source and sample concentration stay very close across all five modalities, with only small upward drift in sampled top\-zone concentration\. Subway remains the most borough\-concentrated system, while Bus, Taxi, FHVHV, and Multimodal cluster much closer together on the borough and Manhattan\-share panels\. The clearest movement appears in Top\-10 zone share, where every modality shifts slightly upward in the sample, but the change is still modest enough to treat as a watchpoint rather than evidence that the calibration sample has become geographically distorted\.

> 🎯 Decision\. The shared calibration samples will not be geography\-stratified at this stage\. Temporal bucket and policy period are treated as the primary sampling structure, while geography is preserved in its observed proportions and monitored through concentration diagnostics\. This keeps the calibration universe closer to the real citywide mobility distribution and avoids imposing an artificial borough or congestion\-pricing balance before the anomaly methods have had a chance to show whether geography\-aware baselines are actually necessary\.

### Section Summary

> This section established a shared anomaly\-calibration sample for each modality and carried it consistently across Raw Scaled, PCA 80%, and PCA\-\>UMAP 10D representations\. The sampling design preserved the full temporal\-bucket by policy\-period structure, produced exact 50,000\-row aligned samples for all five systems, and kept temporal and geographic drift small relative to the full source universe\. Overall, the calibration samples look large enough, balanced enough, and cleanly reusable for downstream DBSCAN, Isolation Forest, and Gaussian Mixture calibration work, with only a mild top\-zone concentration watchpoint to keep in mind later\.

## 3\.3\.1\.3 Define and Evaluate Anomaly Comparison Universes

An anomaly is only meaningful relative to the comparison universe used to score it\. This section defines the candidate baselines that downstream anomaly notebooks may inherit, evaluates whether they are practical enough to carry forward, and uses a lightweight DBSCAN pilot on the shared calibration samples to decide which universes should advance\.

Temporal\-bucket\-aware scoring remains the preferred hypothesis because weekday AM peak, weekend overnight, and evening mobility states represent meaningfully different operating regimes\. Pre/post congestion\-pricing segmentation should be treated as a targeted sanity check, especially for Taxi given the structural drift observed in 3\.2\.2, while geography\-aware baselines should remain diagnostic unless anomaly behavior suggests that broader universes are being overwhelmed by high\-activity locations\.

This section establishes the comparison\-universe vocabulary, tests baseline feasibility, and makes the shared baseline decision once so that DBSCAN, Isolation Forest, and Gaussian Mixture Model notebooks do not need to re\-litigate the same comparison\-universe question independently\.

### Define candidate comparison universes

We’ll start by naming the candidate universes explicitly and giving each one a clear intended role\. That gives the rest of 3\.3\.x a stable vocabulary, so later notebooks can compare anomaly behavior across methods without redefining what the baseline means each time\.

In [26]:
# ---------------------------------------------------------------------
# Define candidate comparison universe specifications
# ---------------------------------------------------------------------

comparison_universe_spec_records = [
    {
        "comparison_universe": "global_all_rows",
        "scope_type": "global",
        "grouping_columns": "none",
        "shared_default_candidate": True,
        "taxi_policy_sanity_check": False,
        "geography_diagnostic_only": False,
        "intended_role": "simple benchmark that keeps every observation in one citywide comparison universe",
    },
    {
        "comparison_universe": "same_temporal_bucket",
        "scope_type": "temporal",
        "grouping_columns": "temporal_bucket",
        "shared_default_candidate": True,
        "taxi_policy_sanity_check": False,
        "geography_diagnostic_only": False,
        "intended_role": "preferred default candidate that respects major operating-regime differences across time",
    },
    {
        "comparison_universe": "same_temporal_bucket_and_pre_post_cp",
        "scope_type": "temporal_policy",
        "grouping_columns": "temporal_bucket, pre_post_cp",
        "shared_default_candidate": False,
        "taxi_policy_sanity_check": True,
        "geography_diagnostic_only": False,
        "intended_role": "targeted policy-period sanity check, with special attention to Taxi after the 3.2.2 stability results",
    },
    {
        "comparison_universe": "same_temporal_bucket_and_policy_geography",
        "scope_type": "temporal_geography",
        "grouping_columns": "temporal_bucket, policy_geography_class",
        "shared_default_candidate": False,
        "taxi_policy_sanity_check": False,
        "geography_diagnostic_only": True,
        "intended_role": "diagnostic geography-aware baseline kept in reserve unless broader universes look overly concentration-driven",
    },
]

comparison_universe_spec_df = pd.DataFrame(comparison_universe_spec_records)

comparison_universe_spec_df["applies_to_feature_sets"] = ", ".join(MODEL_FEATURE_SET_NAMES)

display(
    comparison_universe_spec_df[
        [
            "comparison_universe",
            "scope_type",
            "grouping_columns",
            "shared_default_candidate",
            "taxi_policy_sanity_check",
            "geography_diagnostic_only",
            "applies_to_feature_sets",
            "intended_role",
        ]
    ]
)

print("Candidate comparison universe specifications defined.")

,comparison_universe,scope_type,grouping_columns,shared_default_candidate,taxi_policy_sanity_check,geography_diagnostic_only,applies_to_feature_sets,intended_role
0,global_all_rows,global,none,True,False,False,"bus, subway, taxi, fhvhv, multimodal",simple benchmark that keeps every observation ...
1,same_temporal_bucket,temporal,temporal_bucket,True,False,False,"bus, subway, taxi, fhvhv, multimodal",preferred default candidate that respects majo...
2,same_temporal_bucket_and_pre_post_cp,temporal_policy,"temporal_bucket, pre_post_cp",False,True,False,"bus, subway, taxi, fhvhv, multimodal","targeted policy-period sanity check, with spec..."
3,same_temporal_bucket_and_policy_geography,temporal_geography,"temporal_bucket, policy_geography_class",False,False,True,"bus, subway, taxi, fhvhv, multimodal",diagnostic geography-aware baseline kept in re...


Candidate comparison universe specifications defined.


Findings\. The candidate baseline set is now intentionally narrow: a shared global benchmark, a shared temporal\-bucket\-aware default candidate, a targeted temporal\-bucket plus pre/post sanity check, and a geography\-aware universe held in reserve as diagnostic only\.

Once the universe vocabulary is in place, the next step is to make the downstream testing scope explicit\. We are not carrying every candidate forward in the same way: some are shared default candidates, some are Taxi\-focused sanity checks, and some are diagnostic\-only branches that should stay narrow unless later evidence promotes them\.

In [27]:
# ---------------------------------------------------------------------
# Expand candidate comparison universes by feature set
# ---------------------------------------------------------------------

comparison_universe_feature_set_records = []

for feature_set_name in MODEL_FEATURE_SET_NAMES:
    for _, universe_row in comparison_universe_spec_df.iterrows():
        comparison_universe_feature_set_records.append(
            {
                "feature_set": feature_set_name,
                "comparison_universe": universe_row["comparison_universe"],
                "scope_type": universe_row["scope_type"],
                "grouping_columns": universe_row["grouping_columns"],
                "shared_default_candidate": universe_row["shared_default_candidate"],
                "diagnostic_status": (
                    "taxi_policy_sanity_check"
                    if (
                        feature_set_name == "taxi"
                        and universe_row["taxi_policy_sanity_check"]
                    )
                    else (
                        "geography_diagnostic_only"
                        if universe_row["geography_diagnostic_only"]
                        else "shared_candidate"
                    )
                ),
            }
        )

comparison_universe_feature_set_df = pd.DataFrame(
    comparison_universe_feature_set_records
)

display(comparison_universe_feature_set_df)

print("Candidate comparison universes expanded across feature sets.")

,feature_set,comparison_universe,scope_type,grouping_columns,shared_default_candidate,diagnostic_status
0,bus,global_all_rows,global,none,True,shared_candidate
1,bus,same_temporal_bucket,temporal,temporal_bucket,True,shared_candidate
2,bus,same_temporal_bucket_and_pre_post_cp,temporal_policy,"temporal_bucket, pre_post_cp",False,shared_candidate
3,bus,same_temporal_bucket_and_policy_geography,temporal_geography,"temporal_bucket, policy_geography_class",False,geography_diagnostic_only
4,subway,global_all_rows,global,none,True,shared_candidate
5,subway,same_temporal_bucket,temporal,temporal_bucket,True,shared_candidate
6,subway,same_temporal_bucket_and_pre_post_cp,temporal_policy,"temporal_bucket, pre_post_cp",False,shared_candidate
7,subway,same_temporal_bucket_and_policy_geography,temporal_geography,"temporal_bucket, policy_geography_class",False,geography_diagnostic_only
8,taxi,global_all_rows,global,none,True,shared_candidate
9,taxi,same_temporal_bucket,temporal,temporal_bucket,True,shared_candidate


Candidate comparison universes expanded across feature sets.


Findings\. The comparison\-universe scope is now explicit across all five modalities, with only Taxi carrying a special policy\-period sanity\-check designation and geography\-aware baselines remaining non\-default for every system\.

Before we fit the pilot, let’s do one narrow sanity check that 3\.3\.1\.2 did not answer directly: once we instantiate each comparison universe, do we still have group sizes that are large enough to support downstream anomaly scoring?

In [28]:
# ---------------------------------------------------------------------
# Evaluate candidate comparison-universe group sizes
# ---------------------------------------------------------------------

should_rebuild_group_sizes = (
    REBUILD_COMPARISON_UNIVERSE_TABLES
    or (not DBSCAN_UNIVERSE_GROUP_SIZE_PATH.exists())
)

if should_rebuild_group_sizes:
    universe_group_size_records = []

    for feature_set_name in MODEL_FEATURE_SET_NAMES:
        metadata_df = pd.read_parquet(
            ANOMALY_CALIBRATION_METADATA_PATHS_BY_SET[feature_set_name],
            columns=[
                MODEL_ID_COLUMN,
                TEMPORAL_BUCKET_COLUMN,
                PRE_POST_CP_COLUMN,
                POLICY_GEOGRAPHY_COLUMN,
            ],
        ).copy()

        for comparison_universe_name, grouping_columns in COMPARISON_UNIVERSE_GROUPING_COLUMNS.items():
            universe_df = metadata_df.copy()

            # Represent the candidate universe explicitly as a group key so we can
            # compare fragmentation across baseline definitions on the same sample.
            if grouping_columns:
                universe_df["comparison_group_key"] = (
                    universe_df[grouping_columns]
                    .astype("string")
                    .fillna("Missing")
                    .agg(" | ".join, axis=1)
                )
            else:
                universe_df["comparison_group_key"] = "__all_rows__"

            group_size_df = (
                universe_df.groupby("comparison_group_key", dropna=False)
                .size()
                .reset_index(name="group_rows")
            )

            universe_group_size_records.append(
                {
                    "feature_set": feature_set_name,
                    "comparison_universe": comparison_universe_name,
                    "group_count": int(len(group_size_df)),
                    "min_group_rows": int(group_size_df["group_rows"].min()),
                    "median_group_rows": float(group_size_df["group_rows"].median()),
                    "max_group_rows": int(group_size_df["group_rows"].max()),
                    "groups_below_pilot_floor": int(
                        group_size_df["group_rows"].lt(DBSCAN_PILOT_MIN_ROWS_PER_GROUP).sum()
                    ),
                    "pilot_feasibility_status": (
                        "review"
                        if group_size_df["group_rows"].lt(DBSCAN_PILOT_MIN_ROWS_PER_GROUP).any()
                        else "pass"
                    ),
                }
            )

    comparison_universe_group_size_df = pd.DataFrame(universe_group_size_records)
    comparison_universe_group_size_df.to_parquet(
        DBSCAN_UNIVERSE_GROUP_SIZE_PATH,
        index=False,
    )
else:
    comparison_universe_group_size_df = pd.read_parquet(DBSCAN_UNIVERSE_GROUP_SIZE_PATH)

comparison_universe_group_size_display_df = (
    comparison_universe_group_size_df
    .sort_values(["feature_set", "comparison_universe"])
    .reset_index(drop=True)
)

display(comparison_universe_group_size_display_df)

print("Candidate comparison-universe group sizes evaluated.")

,feature_set,comparison_universe,group_count,min_group_rows,median_group_rows,max_group_rows,groups_below_pilot_floor,pilot_feasibility_status
0,bus,global_all_rows,1,50000,50000.0,50000,0,pass
1,bus,same_temporal_bucket,10,2768,4973.5,7237,0,pass
2,bus,same_temporal_bucket_and_policy_geography,30,190,811.5,5640,0,pass
3,bus,same_temporal_bucket_and_pre_post_cp,20,1053,2227.0,4482,0,pass
4,fhvhv,global_all_rows,1,50000,50000.0,50000,0,pass
5,fhvhv,same_temporal_bucket,10,2858,5000.0,7142,0,pass
6,fhvhv,same_temporal_bucket_and_policy_geography,30,214,805.0,5532,0,pass
7,fhvhv,same_temporal_bucket_and_pre_post_cp,20,1088,2242.5,4427,0,pass
8,multimodal,global_all_rows,1,50000,50000.0,50000,0,pass
9,multimodal,same_temporal_bucket,10,2858,5000.0,7142,0,pass


Candidate comparison-universe group sizes evaluated.


Findings\. All four candidate comparison universes are operationally feasible on the shared calibration samples: every feature set passes, no universe creates groups below the pilot floor, and even the narrowest geography\-aware baseline still retains workable group sizes, with minimum groups ranging from 182 rows in Bus/Taxi/FHVHV/Multimodal to 253 in Subway\.

### Run a lightweight DBSCAN baseline pilot

This is where we stop treating comparison universes as abstract options and see how they behave under a real anomaly mechanism\. We’ll use a lightweight DBSCAN pilot on the shared calibration samples so we can compare global, temporal\-bucket\-aware, temporal\-plus\-policy\-period, and geography\-aware baselines on the same footing\. The goal is not to tune DBSCAN yet\. It is to learn whether a baseline produces workable group sizes, interpretable anomaly isolation, and sensible behavior across the five mobility systems before we carry that baseline into the method\-specific notebooks\.

To keep this pilot focused on the baseline question rather than the representation question, we’ll run it in the pca\_80pct space\. That gives us a reduced representation that preserved neighborhood structure well in 3\.2\.4 while staying light enough for repeated grouped DBSCAN checks\.

In [29]:
# ---------------------------------------------------------------------
# Define lightweight DBSCAN pilot settings and output paths
# ---------------------------------------------------------------------

# Reuse the real calibration-sample path dictionary if it already exists.
# If it does not, rebuild it from the naming pattern used in 3.3.1.2.
if "ANOMALY_CALIBRATION_REPRESENTATION_PATHS_BY_SET_AND_NAME" not in globals():
    ANOMALY_CALIBRATION_REPRESENTATION_PATHS_BY_SET_AND_NAME = {
        feature_set: {
            representation_name: (
                INTERMEDIATE_331_DIR
                / f"{feature_set}_{representation_name}_anomaly_calibration_sample.parquet"
            )
            for representation_name in REPRESENTATION_NAMES
        }
        for feature_set in MODEL_FEATURE_SET_NAMES
    }

COMPARISON_UNIVERSE_GROUPING_COLUMNS = {
    "global_all_rows": [],
    "same_temporal_bucket": [TEMPORAL_BUCKET_COLUMN],
    "same_temporal_bucket_and_pre_post_cp": [
        TEMPORAL_BUCKET_COLUMN,
        PRE_POST_CP_COLUMN,
    ],
    "same_temporal_bucket_and_policy_geography": [
        TEMPORAL_BUCKET_COLUMN,
        POLICY_GEOGRAPHY_COLUMN,
    ],
}

pilot_input_path_records = []

for feature_set_name in MODEL_FEATURE_SET_NAMES:
    pilot_input_path_records.append(
        {
            "feature_set": feature_set_name,
            "metadata_path": str(ANOMALY_CALIBRATION_METADATA_PATHS_BY_SET[feature_set_name]),
            "pca_80pct_path": str(
                ANOMALY_CALIBRATION_REPRESENTATION_PATHS_BY_SET_AND_NAME[feature_set_name][
                    DBSCAN_PILOT_REPRESENTATION_NAME
                ]
            ),
            "metadata_exists": ANOMALY_CALIBRATION_METADATA_PATHS_BY_SET[feature_set_name].exists(),
            "pca_80pct_exists": ANOMALY_CALIBRATION_REPRESENTATION_PATHS_BY_SET_AND_NAME[
                feature_set_name
            ][DBSCAN_PILOT_REPRESENTATION_NAME].exists(),
        }
    )

pilot_input_path_df = pd.DataFrame(pilot_input_path_records)

display(pilot_input_path_df)

assert pilot_input_path_df["metadata_exists"].all(), (
    "One or more calibration metadata tables is missing."
)
assert pilot_input_path_df["pca_80pct_exists"].all(), (
    "One or more PCA calibration sample tables is missing."
)

print("DBSCAN pilot settings and input paths defined.")

,feature_set,metadata_path,pca_80pct_path,metadata_exists,pca_80pct_exists
0,bus,/datasets/_deepnote_work/pipeline_data/3.3.1.i...,/datasets/_deepnote_work/pipeline_data/3.3.1.i...,True,True
1,subway,/datasets/_deepnote_work/pipeline_data/3.3.1.i...,/datasets/_deepnote_work/pipeline_data/3.3.1.i...,True,True
2,taxi,/datasets/_deepnote_work/pipeline_data/3.3.1.i...,/datasets/_deepnote_work/pipeline_data/3.3.1.i...,True,True
3,fhvhv,/datasets/_deepnote_work/pipeline_data/3.3.1.i...,/datasets/_deepnote_work/pipeline_data/3.3.1.i...,True,True
4,multimodal,/datasets/_deepnote_work/pipeline_data/3.3.1.i...,/datasets/_deepnote_work/pipeline_data/3.3.1.i...,True,True


DBSCAN pilot settings and input paths defined.


Findings\. The lightweight DBSCAN pilot is ready to run across all five modalities in the pca\_80pct space, and the required calibration metadata and PCA sample tables are present for every feature set\. That means the baseline\-universe comparison is being tested on a shared, fidelity\-preserving reduced representation rather than being confounded by missing assets or representation drift\.

This block runs the pilot and writes the reusable outputs to disk\. To keep the notebook readable, we’ll separate computation from interpretation: the pilot itself will only materialize the cached results, and the next blocks will surface the structure and noise summaries in a much more focused way\.

In [30]:

# ---------------------------------------------------------------------
# Run the lightweight DBSCAN baseline pilot
# ---------------------------------------------------------------------

should_rebuild_dbscan_pilot = (
    REBUILD_BASELINE_FEASIBILITY_TABLES
    or (not DBSCAN_PILOT_SUMMARY_PATH.exists())
    or (not DBSCAN_PILOT_GROUP_DETAIL_PATH.exists())
)

if should_rebuild_dbscan_pilot:
    dbscan_pilot_group_records = []
    dbscan_pilot_summary_records = []

    for feature_set_name in MODEL_FEATURE_SET_NAMES:
        metadata_df = pd.read_parquet(
            ANOMALY_CALIBRATION_METADATA_PATHS_BY_SET[feature_set_name],
            columns=[
                MODEL_ID_COLUMN,
                TEMPORAL_BUCKET_COLUMN,
                PRE_POST_CP_COLUMN,
                POLICY_GEOGRAPHY_COLUMN,
                BOROUGH_COLUMN,
            ],
        ).copy()

        pca_df = pd.read_parquet(
            ANOMALY_CALIBRATION_REPRESENTATION_PATHS_BY_SET_AND_NAME[feature_set_name][
                DBSCAN_PILOT_REPRESENTATION_NAME
            ]
        ).copy()

        pca_feature_columns = [
            column
            for column in pca_df.columns
            if column not in DEFAULT_METADATA_COLUMNS
        ]

        # Align metadata and PCA calibration rows so every baseline universe is
        # evaluated on the same shared observation set.
        pilot_df = metadata_df.merge(
            pca_df[[MODEL_ID_COLUMN] + pca_feature_columns],
            on=MODEL_ID_COLUMN,
            how="inner",
            validate="one_to_one",
        )

        for comparison_universe_name, grouping_columns in COMPARISON_UNIVERSE_GROUPING_COLUMNS.items():
            universe_df = pilot_df.copy()

            if grouping_columns:
                universe_df["comparison_group_key"] = (
                    universe_df[grouping_columns]
                    .astype("string")
                    .fillna("Missing")
                    .agg(" | ".join, axis=1)
                )
            else:
                universe_df["comparison_group_key"] = "__all_rows__"

            weighted_noise_numerator = 0.0
            weighted_cluster_numerator = 0.0
            weighted_largest_cluster_numerator = 0.0
            evaluated_rows = 0
            groups_evaluated = 0
            groups_skipped = 0

            for comparison_group_key, group_df in universe_df.groupby("comparison_group_key", dropna=False):
                group_rows = len(group_df)

                if group_rows < DBSCAN_PILOT_MIN_ROWS_PER_GROUP:
                    groups_skipped += 1
                    dbscan_pilot_group_records.append(
                        {
                            "feature_set": feature_set_name,
                            "comparison_universe": comparison_universe_name,
                            "comparison_group_key": comparison_group_key,
                            "group_rows": group_rows,
                            "rows_evaluated": 0,
                            "eps": np.nan,
                            "noise_share": np.nan,
                            "cluster_count": np.nan,
                            "largest_cluster_share": np.nan,
                            "status": "skipped_small_group",
                        }
                    )
                    continue

                eval_group_df = (
                    group_df.sample(
                        n=min(DBSCAN_PILOT_MAX_ROWS_PER_GROUP, group_rows),
                        random_state=ANOMALY_FRAMEWORK_RANDOM_STATE,
                    )
                    .copy()
                )

                X = eval_group_df[pca_feature_columns].to_numpy(dtype="float32", copy=False)

                neighbor_k = min(DBSCAN_PILOT_MIN_SAMPLES, len(eval_group_df) - 1)
                if neighbor_k < 2:
                    groups_skipped += 1
                    dbscan_pilot_group_records.append(
                        {
                            "feature_set": feature_set_name,
                            "comparison_universe": comparison_universe_name,
                            "comparison_group_key": comparison_group_key,
                            "group_rows": group_rows,
                            "rows_evaluated": len(eval_group_df),
                            "eps": np.nan,
                            "noise_share": np.nan,
                            "cluster_count": np.nan,
                            "largest_cluster_share": np.nan,
                            "status": "skipped_too_small_for_knn",
                        }
                    )
                    continue

                # Estimate eps within each baseline group so the pilot reflects the
                # density scale implied by that comparison universe.
                knn = NearestNeighbors(n_neighbors=neighbor_k)
                knn.fit(X)
                neighbor_distances = knn.kneighbors(X)[0][:, -1]
                eps = float(np.quantile(neighbor_distances, DBSCAN_PILOT_EPS_QUANTILE))

                dbscan_model = DBSCAN(
                    eps=eps,
                    min_samples=DBSCAN_PILOT_MIN_SAMPLES,
                    metric="euclidean",
                )
                labels = dbscan_model.fit_predict(X)

                noise_share = float((labels == -1).mean())

                cluster_labels = pd.Series(labels[labels != -1])
                if len(cluster_labels) == 0:
                    cluster_count = 0
                    largest_cluster_share = 0.0
                else:
                    cluster_sizes = cluster_labels.value_counts()
                    cluster_count = int(cluster_sizes.shape[0])
                    largest_cluster_share = float(cluster_sizes.max() / len(labels))

                rows_evaluated = len(eval_group_df)
                groups_evaluated += 1
                evaluated_rows += rows_evaluated
                weighted_noise_numerator += noise_share * rows_evaluated
                weighted_cluster_numerator += cluster_count * rows_evaluated
                weighted_largest_cluster_numerator += largest_cluster_share * rows_evaluated

                dbscan_pilot_group_records.append(
                    {
                        "feature_set": feature_set_name,
                        "comparison_universe": comparison_universe_name,
                        "comparison_group_key": comparison_group_key,
                        "group_rows": group_rows,
                        "rows_evaluated": rows_evaluated,
                        "eps": round(eps, 6),
                        "noise_share": round(noise_share, 4),
                        "cluster_count": cluster_count,
                        "largest_cluster_share": round(largest_cluster_share, 4),
                        "status": "evaluated",
                    }
                )

            dbscan_pilot_summary_records.append(
                {
                    "feature_set": feature_set_name,
                    "pilot_representation": DBSCAN_PILOT_REPRESENTATION_NAME,
                    "comparison_universe": comparison_universe_name,
                    "groups_evaluated": groups_evaluated,
                    "groups_skipped": groups_skipped,
                    "rows_evaluated": evaluated_rows,
                    "weighted_noise_share": round(
                        weighted_noise_numerator / evaluated_rows, 4
                    ) if evaluated_rows else np.nan,
                    "weighted_cluster_count": round(
                        weighted_cluster_numerator / evaluated_rows, 4
                    ) if evaluated_rows else np.nan,
                    "weighted_largest_cluster_share": round(
                        weighted_largest_cluster_numerator / evaluated_rows, 4
                    ) if evaluated_rows else np.nan,
                    "status": "pass" if groups_evaluated > 0 else "review",
                }
            )

    dbscan_pilot_group_detail_df = pd.DataFrame(dbscan_pilot_group_records)
    dbscan_pilot_summary_df = pd.DataFrame(dbscan_pilot_summary_records)

    dbscan_pilot_group_detail_df.to_parquet(
        DBSCAN_PILOT_GROUP_DETAIL_PATH,
        index=False,
    )
    dbscan_pilot_summary_df.to_parquet(
        DBSCAN_PILOT_SUMMARY_PATH,
        index=False,
    )
    pilot_output_action = "written"
else:
    dbscan_pilot_group_detail_df = pd.read_parquet(DBSCAN_PILOT_GROUP_DETAIL_PATH)
    dbscan_pilot_summary_df = pd.read_parquet(DBSCAN_PILOT_SUMMARY_PATH)
    pilot_output_action = "loaded"

pilot_execution_status_df = pd.DataFrame(
    {
        "artifact": ["pilot_summary", "pilot_group_detail"],
        "path": [str(DBSCAN_PILOT_SUMMARY_PATH), str(DBSCAN_PILOT_GROUP_DETAIL_PATH)],
        "output_action": pilot_output_action,
        "path_exists": [
            DBSCAN_PILOT_SUMMARY_PATH.exists(),
            DBSCAN_PILOT_GROUP_DETAIL_PATH.exists(),
        ],
    }
)

display(pilot_execution_status_df)

print("Lightweight DBSCAN baseline pilot completed.")

,artifact,path,output_action,path_exists
0,pilot_summary,/datasets/_deepnote_work/pipeline_data/3.3.1.i...,loaded,True
1,pilot_group_detail,/datasets/_deepnote_work/pipeline_data/3.3.1.i...,loaded,True


Lightweight DBSCAN baseline pilot completed.


Findings\. The DBSCAN pilot completed successfully and cached both the summary\-level and group\-level diagnostic outputs, so we can now inspect baseline behavior without turning the fitting block itself into a wall of results\.

### Analyzing the Initial Results of DBSCAN by Structure

The pilot has now done its first job: it gives us a fast, controlled way to see how baseline choice changes anomaly\-like behavior before we commit to a shared downstream framework\. We can now read the results in layers\. First, we look at structural behavior: whether a candidate universe still collapses observations into one dominant cluster or starts to reveal more local organization\. Then we separate out anomaly\-like volume, so we can tell whether a narrower baseline is genuinely improving the comparison logic or simply labeling more points as unusual\. Finally, we pull those pieces back together into a compact decision summary that tells us which universes are worth carrying forward and which ones still need to justify themselves on more specific grounds, especially geography concentration\.

This first readout focuses only on structure\. The two metrics here are weighted\_cluster\_count, which summarizes how many non\-noise clusters DBSCAN is finding across the evaluated groups, and weighted\_largest\_cluster\_share, which tells us how much of the data is still being absorbed by the single biggest cluster\. Together, they help us see whether a candidate baseline reveals more local organization or whether it still collapses most observations into one dominant “normal” regime\.

In [31]:
# ---------------------------------------------------------------------
# Display DBSCAN pilot structure summary
# ---------------------------------------------------------------------

pilot_structure_summary_df = (
    dbscan_pilot_summary_df[
        [
            "feature_set",
            "comparison_universe",
            "weighted_cluster_count",
            "weighted_largest_cluster_share",
        ]
    ]
    .sort_values(["feature_set", "comparison_universe"])
    .reset_index(drop=True)
)

display(pilot_structure_summary_df)

structure_plot_df = pilot_structure_summary_df.copy()
structure_plot_df["feature_set_display"] = structure_plot_df["feature_set"].str.title()

comparison_universe_display_map = {
    "global_all_rows": "Global",
    "same_temporal_bucket": "Temporal",
    "same_temporal_bucket_and_pre_post_cp": "Temporal +<br>pre/post",
    "same_temporal_bucket_and_policy_geography": "Temporal +<br>geography",
}

structure_plot_df["comparison_universe_display"] = (
    structure_plot_df["comparison_universe"].map(comparison_universe_display_map)
)

feature_set_color_map = {
    feature_set_name: BRAND_COLOR_SEQUENCE[index % len(BRAND_COLOR_SEQUENCE)]
    for index, feature_set_name in enumerate(MODEL_FEATURE_SET_NAMES)
}

structure_fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=[
        "Weighted cluster count",
        "Weighted largest-cluster share",
    ],
    horizontal_spacing=0.10,
)

for feature_set_name in MODEL_FEATURE_SET_NAMES:
    feature_plot_df = (
        structure_plot_df.loc[structure_plot_df["feature_set"].eq(feature_set_name)]
        .copy()
        .reset_index(drop=True)
    )
    feature_color = feature_set_color_map[feature_set_name]

    structure_fig.add_trace(
        go.Scatter(
            x=feature_plot_df["comparison_universe_display"],
            y=feature_plot_df["weighted_cluster_count"],
            mode="lines+markers",
            line={"color": feature_color, "width": 2},
            marker={"color": feature_color, "size": 8},
            name=feature_set_name.title(),
            legendgroup=feature_set_name,
            showlegend=True,
            hovertemplate=(
                "Feature set: %{text}<br>"
                "Baseline: %{x}<br>"
                "Weighted cluster count: %{y:.2f}<extra></extra>"
            ),
            text=feature_plot_df["feature_set_display"],
        ),
        row=1,
        col=1,
    )
    structure_fig.add_trace(
        go.Scatter(
            x=feature_plot_df["comparison_universe_display"],
            y=feature_plot_df["weighted_largest_cluster_share"],
            mode="lines+markers",
            line={"color": feature_color, "width": 2},
            marker={"color": feature_color, "size": 8},
            name=feature_set_name.title(),
            legendgroup=feature_set_name,
            showlegend=False,
            hovertemplate=(
                "Feature set: %{text}<br>"
                "Baseline: %{x}<br>"
                "Weighted largest-cluster share: %{y:.3f}<extra></extra>"
            ),
            text=feature_plot_df["feature_set_display"],
        ),
        row=1,
        col=2,
    )

structure_fig.update_yaxes(title_text="Cluster count", row=1, col=1)
structure_fig.update_yaxes(title_text="Largest-cluster share", row=1, col=2)
structure_fig.update_xaxes(tickangle=0, row=1, col=1)
structure_fig.update_xaxes(tickangle=0, row=1, col=2)

apply_brand_plotly_layout(
    structure_fig,
    title="DBSCAN Pilot Structure by Comparison Universe",
    width=950,
    height=430,
)

structure_fig.update_layout(
    legend={
        "orientation": "h",
        "yanchor": "top",
        "y": -0.18,
        "xanchor": "center",
        "x": 0.5,
        "bgcolor": "rgba(255,255,255,0.85)",
        "bordercolor": BRAND_COLORS["seafoam"],
        "borderwidth": 1,
        "font": {"color": BRAND_COLORS["dark_teal"]},
    },
    margin={"t": 70, "b": 90, "l": 60, "r": 30},
)

structure_fig.show()

print("DBSCAN pilot structure summary displayed.")

,feature_set,comparison_universe,weighted_cluster_count,weighted_largest_cluster_share
0,bus,global_all_rows,2.0000,0.9507
1,bus,same_temporal_bucket,2.1515,0.9521
2,bus,same_temporal_bucket_and_policy_geography,1.6035,0.9432
3,bus,same_temporal_bucket_and_pre_post_cp,2.0600,0.9546
4,fhvhv,global_all_rows,2.0000,0.9520
5,fhvhv,same_temporal_bucket,2.2666,0.9464
6,fhvhv,same_temporal_bucket_and_policy_geography,1.7704,0.9375
7,fhvhv,same_temporal_bucket_and_pre_post_cp,2.6212,0.9464
8,multimodal,global_all_rows,1.0000,0.9720
9,multimodal,same_temporal_bucket,1.3500,0.9713


DBSCAN pilot structure summary displayed.


Findings\. The structural case for temporal bucketing is strongest in Subway, where moving from a global baseline to a same\-temporal\-bucket baseline produces more local cluster structure and noticeably weakens dominant\-cluster behavior\. Taxi shows a smaller version of the same pattern, while Bus, FHVHV, and Multimodal change less\. Taken alone, this does not prove that every modality requires temporal bucketing, but it does show that a global baseline can be too coarse for at least some systems and that temporal segmentation can reveal meaningful operating\-regime structure without obviously destabilizing the pilot\. The added pre/post split, by contrast, does not deliver a comparably useful structural gain, so it does not earn a place as a shared default baseline\.

Noise share is the fraction of evaluated observations that DBSCAN labels as noise, so this readout tells us whether a baseline is changing anomaly volume itself or mainly changing the placement of anomaly\-like points\. The next question is simpler: does a narrower baseline mainly change anomaly structure, or does it just create more anomaly\-like points? This readout isolates the noise\-share effect so we can read it separately from fragmentation\.

In [32]:
# ---------------------------------------------------------------------
# Display DBSCAN pilot noise summary
# ---------------------------------------------------------------------

pilot_noise_summary_df = (
    dbscan_pilot_summary_df[
        [
            "feature_set",
            "comparison_universe",
            "weighted_noise_share",
        ]
    ]
    .sort_values(["feature_set", "comparison_universe"])
    .reset_index(drop=True)
)

display(pilot_noise_summary_df)

noise_plot_df = pilot_noise_summary_df.copy()
noise_plot_df["feature_set_display"] = noise_plot_df["feature_set"].str.title()
noise_plot_df["comparison_universe_display"] = (
    noise_plot_df["comparison_universe"].map(comparison_universe_display_map)
)

feature_set_color_map = {
    feature_set_name: BRAND_COLOR_SEQUENCE[index % len(BRAND_COLOR_SEQUENCE)]
    for index, feature_set_name in enumerate(MODEL_FEATURE_SET_NAMES)
}

noise_fig = go.Figure()

for feature_set_name in MODEL_FEATURE_SET_NAMES:
    feature_plot_df = (
        noise_plot_df.loc[noise_plot_df["feature_set"].eq(feature_set_name)]
        .copy()
        .reset_index(drop=True)
    )
    feature_color = feature_set_color_map[feature_set_name]

    noise_fig.add_trace(
        go.Scatter(
            x=feature_plot_df["comparison_universe_display"],
            y=feature_plot_df["weighted_noise_share"],
            mode="lines+markers",
            line={"color": feature_color, "width": 2},
            marker={"color": feature_color, "size": 8},
            name=feature_set_name.title(),
            hovertemplate=(
                "Feature set: %{text}<br>"
                "Baseline: %{x}<br>"
                "Weighted noise share: %{y:.4f}<extra></extra>"
            ),
            text=feature_plot_df["feature_set_display"],
        )
    )

noise_fig.update_yaxes(title_text="Noise share")

apply_brand_plotly_layout(
    noise_fig,
    title="DBSCAN Pilot Noise Share by Comparison Universe",
    width=850,
    height=420,
)

noise_fig.update_layout(
    legend={
        "orientation": "h",
        "yanchor": "top",
        "y": -0.18,
        "xanchor": "center",
        "x": 0.5,
        "bgcolor": "rgba(255,255,255,0.85)",
        "bordercolor": BRAND_COLORS["seafoam"],
        "borderwidth": 1,
        "font": {"color": BRAND_COLORS["dark_teal"]},
    },
    margin={"t": 70, "b": 90, "l": 60, "r": 30},
)

noise_fig.show()

print("DBSCAN pilot noise summary displayed.")

,feature_set,comparison_universe,weighted_noise_share
0,bus,global_all_rows,0.0203
1,bus,same_temporal_bucket,0.0227
2,bus,same_temporal_bucket_and_policy_geography,0.0227
3,bus,same_temporal_bucket_and_pre_post_cp,0.0225
4,fhvhv,global_all_rows,0.0235
5,fhvhv,same_temporal_bucket,0.0272
6,fhvhv,same_temporal_bucket_and_policy_geography,0.0255
7,fhvhv,same_temporal_bucket_and_pre_post_cp,0.0262
8,multimodal,global_all_rows,0.0280
9,multimodal,same_temporal_bucket,0.0270


DBSCAN pilot noise summary displayed.


Findings\. This block is mainly a sanity check\. The earlier structure results suggested that temporal bucketing helps, especially for Subway, because it reveals more local organization than a global baseline\. Here, noise share stays fairly stable across the candidate universes, which means that temporal bucketing is not simply creating a lot more anomaly\-like points\. Instead, it is mostly changing how observations are being compared\. That makes the structural gains from the previous block easier to trust\.

Summary Scorecard\. Now that we have looked at structure and anomaly volume separately, we can bring them back together in one compact scorecard\. The three columns summarize what we care about at this stage: Structure gain rewards baselines that reveal more local cluster structure, Less dominance rewards baselines that reduce the share absorbed by one oversized cluster, and Stable anomaly volume rewards baselines that keep noise share near the feature\-set norm rather than changing anomaly\-like volume too aggressively\. This does not replace the detailed tables above; it compresses them into a decision\-oriented view so we can see which universes are strongest overall and which ones still need to justify themselves mainly through the geography diagnostics\.

In [33]:
# ---------------------------------------------------------------------
# Summarize the DBSCAN pilot for comparison-universe decisions
# ---------------------------------------------------------------------

pilot_summary_df = pd.read_parquet(DBSCAN_PILOT_SUMMARY_PATH).copy()

baseline_order = [
    "global_all_rows",
    "same_temporal_bucket",
    "same_temporal_bucket_and_policy_geography",
    "same_temporal_bucket_and_pre_post_cp",
]
feature_set_order = ["bus", "subway", "taxi", "fhvhv", "multimodal"]

baseline_label_map = {
    "global_all_rows": "Global",
    "same_temporal_bucket": "Temporal",
    "same_temporal_bucket_and_policy_geography": "Temporal +<br>geography",
    "same_temporal_bucket_and_pre_post_cp": "Temporal +<br>pre/post",
}
metric_order = [
    "structure_gain_score",
    "less_dominance_score",
    "stable_anomaly_volume_score",
]
metric_label_map = {
    "structure_gain_score": "Structure gain",
    "less_dominance_score": "Less dominance",
    "stable_anomaly_volume_score": "Stable anomaly<br>volume",
}

# Higher cluster count is treated as more revealed local structure.
pilot_summary_df["structure_gain_raw"] = pilot_summary_df["weighted_cluster_count"]

# Lower largest-cluster share is better because it means less collapse into one dominant regime.
pilot_summary_df["less_dominance_raw"] = 1 - pilot_summary_df["weighted_largest_cluster_share"]

# Score anomaly-volume stability by rewarding baselines that stay close to the
# feature-set-specific median noise share rather than simply minimizing noise.
pilot_summary_df["feature_set_noise_median"] = (
    pilot_summary_df.groupby("feature_set")["weighted_noise_share"].transform("median")
)
pilot_summary_df["stable_anomaly_volume_raw"] = -(
    pilot_summary_df["weighted_noise_share"] - pilot_summary_df["feature_set_noise_median"]
).abs()

score_columns = [
    ("structure_gain_raw", "structure_gain_score"),
    ("less_dominance_raw", "less_dominance_score"),
    ("stable_anomaly_volume_raw", "stable_anomaly_volume_score"),
]

# Normalize each score within feature set so the comparison stays local to each modality.
for raw_col, score_col in score_columns:
    group_min = pilot_summary_df.groupby("feature_set")[raw_col].transform("min")
    group_max = pilot_summary_df.groupby("feature_set")[raw_col].transform("max")
    denom = (group_max - group_min).replace(0, np.nan)
    pilot_summary_df[score_col] = np.where(
        denom.notna(),
        (pilot_summary_df[raw_col] - group_min) / denom,
        1.0,
    )

comparison_universe_scorecard_df = (
    pilot_summary_df[
        [
            "feature_set",
            "comparison_universe",
            "weighted_cluster_count",
            "weighted_largest_cluster_share",
            "weighted_noise_share",
            "structure_gain_score",
            "less_dominance_score",
            "stable_anomaly_volume_score",
        ]
    ]
    .sort_values(["feature_set", "comparison_universe"])
    .reset_index(drop=True)
)

display(comparison_universe_scorecard_df)

# ---------------------------------------------------------------------
# Visualize faceted comparison-universe scorecards by feature set
# ---------------------------------------------------------------------

feature_set_layout = [
    ("bus", 1, 1),
    ("subway", 1, 2),
    ("taxi", 2, 1),
    ("fhvhv", 2, 2),
    ("multimodal", 2, 3),
]

wrapped_metric_labels = {
    "structure_gain_score": "Structure<br>gain",
    "less_dominance_score": "Less<br>dominance",
    "stable_anomaly_volume_score": "Stable<br>anomaly<br>volume",
}

fig = make_subplots(
    rows=2,
    cols=3,
    subplot_titles=["Bus", "Subway", "", "Taxi", "Fhvhv", "Multimodal"],
    horizontal_spacing=0.07,
    vertical_spacing=0.07,
)

for feature_set_name, row_num, col_num in feature_set_layout:
    panel_df = (
        comparison_universe_scorecard_df.loc[
            comparison_universe_scorecard_df["feature_set"].eq(feature_set_name)
        ]
        .set_index("comparison_universe")
        .loc[baseline_order]
        .reset_index()
    )
    z_matrix = panel_df[metric_order].to_numpy()
    hover_text = []
    for _, row in panel_df.iterrows():
        hover_row = []
        for metric_name in metric_order:
            hover_row.append(
                "<br>".join(
                    [
                        f"Feature set: {feature_set_name}",
                        f"Baseline: {baseline_label_map[row['comparison_universe']].replace('<br>', ' ')}",
                        f"Metric: {metric_label_map[metric_name].replace('<br>', ' ')}",
                        f"Relative score: {row[metric_name]:.2f}",
                    ]
                )
            )
        hover_text.append(hover_row)

    fig.add_trace(
        go.Heatmap(
            z=z_matrix,
            x=[wrapped_metric_labels[m] for m in metric_order],
            y=[baseline_label_map[b] for b in baseline_order],
            zmin=0,
            zmax=1,
            colorscale=BRAND_DIVERGING_SEQUENCE,
            showscale=(feature_set_name == "bus"),
            colorbar={
                "title": {
                    "text": "Relative score",
                    "font": {"color": BRAND_COLORS["dark_teal"]},
                },
                "tickfont": {"color": BRAND_COLORS["dark_teal"]},
                "len": 0.74,
                "y": 0.53,
            }
            if feature_set_name == "bus"
            else None,
            text=np.round(z_matrix, 2),
            texttemplate="%{text:.2f}",
            customdata=hover_text,
            hovertemplate="%{customdata}<extra></extra>",
            xgap=5,
            ygap=3,
        ),
        row=row_num,
        col=col_num,
    )

    fig.update_xaxes(
        title_text=None,
        tickangle=0,
        automargin=True,
        showticklabels=(row_num == 2),
        row=row_num,
        col=col_num,
    )
    fig.update_yaxes(
        title_text=None,
        automargin=True,
        ticklabelstandoff=12,
        showticklabels=(col_num == 1),
        row=row_num,
        col=col_num,
    )

# Hide the empty top-right panel entirely.
fig.update_xaxes(visible=False, row=1, col=3)
fig.update_yaxes(visible=False, row=1, col=3)

apply_brand_plotly_layout(
    fig,
    title="DBSCAN Pilot Comparison-Universe Scorecards by Feature Set",
    width=960,
    height=720,
)

fig.update_layout(
    margin={"l": 190, "r": 95, "t": 80, "b": 120},
    annotations=[
        *fig.layout.annotations,
        {
            "text": "Comparison universe",
            "xref": "paper",
            "yref": "paper",
            "x": -0.19,
            "y": 0.50,
            "showarrow": False,
            "textangle": -90,
            "font": {"size": 13, "color": BRAND_COLORS["dark_teal"]},
        },
        {
            "text": "Score dimension",
            "xref": "paper",
            "yref": "paper",
            "x": 0.52,
            "y": -0.12,
            "showarrow": False,
            "font": {"size": 13, "color": BRAND_COLORS["dark_teal"]},
        },
    ],
)

fig.show()

print("Comparison-universe scorecard table and faceted heatmap prepared.")

,feature_set,comparison_universe,weighted_cluster_count,weighted_largest_cluster_share,weighted_noise_share,structure_gain_score,less_dominance_score,stable_anomaly_volume_score
0,bus,global_all_rows,2.0000,0.9507,0.0203,0.723540,0.342105,0.0000
1,bus,same_temporal_bucket,2.1515,0.9521,0.0227,1.000000,0.219298,1.0000
2,bus,same_temporal_bucket_and_policy_geography,1.6035,0.9432,0.0227,0.000000,1.000000,1.0000
3,bus,same_temporal_bucket_and_pre_post_cp,2.0600,0.9546,0.0225,0.833029,0.000000,1.0000
4,fhvhv,global_all_rows,2.0000,0.9520,0.0235,0.269864,0.000000,0.0000
5,fhvhv,same_temporal_bucket,2.2666,0.9464,0.0272,0.583216,0.386207,0.5000
6,fhvhv,same_temporal_bucket_and_policy_geography,1.7704,0.9375,0.0255,0.000000,1.000000,1.0000
7,fhvhv,same_temporal_bucket_and_pre_post_cp,2.6212,0.9464,0.0262,1.000000,0.386207,1.0000
8,multimodal,global_all_rows,1.0000,0.9720,0.0280,0.000000,0.000000,0.0000
9,multimodal,same_temporal_bucket,1.3500,0.9713,0.0270,0.417313,0.179487,1.0000


Comparison-universe scorecard table and faceted heatmap prepared.


> 💡Note\. This scorecard is normalized within each feature set, so the colors should be read comparatively rather than absolutely\. A value of 1 means that baseline performs best among the four candidate universes for that modality on that specific metric, while a value of 0 means it performs worst on that same metric relative to the other three\. Here, “best” and “worst” depend on what the column is measuring: higher Structure gain means more local cluster structure is revealed, higher Less dominance means the largest cluster absorbs less of the data, and higher Stable anomaly volume means the baseline keeps noise share closer to the feature\-set norm rather than shifting anomaly\-like volume too aggressively\. Green therefore means “best relative performer here,” not “perfect,” and red means “worst relative performer here,” not “invalid\.”

Findings\. The scorecard makes the temporal case much clearer than the earlier tables\. Across most modalities, Temporal is the strongest or near\-strongest baseline on structure gain and usually stays competitive on anomaly\-volume stability as well, which is exactly the pattern we want from a shared default\. Global is rarely the best option and is often the weakest\. Temporal \+ geography looks more mixed here: it sometimes improves dominance behavior, but it does not consistently win the scorecard on its own\. That is why this block supports moving forward with temporal bucketing as the main baseline idea, while leaving the geography question open for the next subsection rather than rejecting it too early\.

> 🎯 Decision\. Temporal\-bucket\-aware anomaly scoring advances downstream as the shared default baseline hypothesis\. The DBSCAN pilot shows that comparing observations within the same temporal bucket reveals more local structure than a global baseline without obviously destabilizing anomaly\-like volume\.

### Decide whether geography\-aware baselines are warranted: Analyze by Outlier Distribution

This scorecard gives us a clear answer on one question: temporal bucketing is worth carrying forward\. It gives a much weaker answer on the geography question\. Temporal \+ geography does not dominate the scorecard the way Temporal does, but it also is not ruled out by these aggregate results\. That leaves one important concern unresolved: if we stop at temporal bucketing, do anomaly\-like points still collapse too heavily onto Manhattan, CBD zones, or the same small set of dominant zones? This subsection tests that directly\.

The earlier pilot summary told us how the candidate universes behave in aggregate\. To answer the geography question, we now need row\-level pilot outputs for the universes that matter most: the shared temporal default candidate, the geography\-aware diagnostic candidate, and the global benchmark for reference\.

In [34]:
# ---------------------------------------------------------------------
# Define geography-baseline diagnostic settings and output paths
# ---------------------------------------------------------------------

GEOGRAPHY_BASELINE_DIAGNOSTIC_UNIVERSES = [
    "global_all_rows",
    "same_temporal_bucket",
    "same_temporal_bucket_and_policy_geography",
]

DBSCAN_GEOGRAPHY_DIAGNOSTIC_ROW_PATH = (
    INTERMEDIATE_331_DIR / "dbscan_geography_baseline_row_diagnostics.parquet"
)
DBSCAN_GEOGRAPHY_DIAGNOSTIC_SUMMARY_PATH = (
    INTERMEDIATE_331_DIR / "dbscan_geography_baseline_summary.parquet"
)
DBSCAN_GEOGRAPHY_ZONE_PERSISTENCE_PATH = (
    INTERMEDIATE_331_DIR / "dbscan_geography_zone_persistence.parquet"
)

geography_diagnostic_spec_df = pd.DataFrame(
    {
        "comparison_universe": GEOGRAPHY_BASELINE_DIAGNOSTIC_UNIVERSES,
        "included_in_geography_diagnostic": True,
        "pilot_representation": DBSCAN_PILOT_REPRESENTATION_NAME,
        "min_samples": DBSCAN_PILOT_MIN_SAMPLES,
        "eps_quantile": DBSCAN_PILOT_EPS_QUANTILE,
        "max_rows_per_group": DBSCAN_PILOT_MAX_ROWS_PER_GROUP,
    }
)

display(geography_diagnostic_spec_df)

print("Geography-baseline diagnostic settings defined.")

,comparison_universe,included_in_geography_diagnostic,pilot_representation,min_samples,eps_quantile,max_rows_per_group
0,global_all_rows,True,pca_80pct,15,0.95,4000
1,same_temporal_bucket,True,pca_80pct,15,0.95,4000
2,same_temporal_bucket_and_policy_geography,True,pca_80pct,15,0.95,4000


Geography-baseline diagnostic settings defined.


Findings\. The geography diagnostic keeps the pilot deliberately narrow and comparable: all three candidate universes are evaluated with the same PCA representation, DBSCAN calibration rule, and per\-group row cap, so any differences we see downstream are coming from the baseline definition rather than from a moving\-method setup\.

Materialize DBSCAN GEO\-Baselines\. This block just creates the row\-level pilot output we need for the geography test\. We are not interpreting geography yet; we are caching the anomaly\-like labels and group metadata so the next blocks can ask cleaner questions about volume, concentration, and repetition\.

In [35]:
# ---------------------------------------------------------------------
# Materialize row-level DBSCAN geography-baseline diagnostics
# ---------------------------------------------------------------------

should_rebuild_geography_rows = (
    REBUILD_BASELINE_FEASIBILITY_TABLES
    or (not DBSCAN_GEOGRAPHY_DIAGNOSTIC_ROW_PATH.exists())
)

if should_rebuild_geography_rows:
    geography_row_records = []

    for feature_set_name in MODEL_FEATURE_SET_NAMES:
        metadata_df = pd.read_parquet(
            ANOMALY_CALIBRATION_METADATA_PATHS_BY_SET[feature_set_name],
            columns=[
                MODEL_ID_COLUMN,
                DATE_COLUMN,
                TEMPORAL_BUCKET_COLUMN,
                PRE_POST_CP_COLUMN,
                POLICY_GEOGRAPHY_COLUMN,
                BOROUGH_COLUMN,
                ZONE_ID_COLUMN,
                ZONE_COLUMN,
            ],
        ).copy()

        pca_df = pd.read_parquet(
            ANOMALY_CALIBRATION_REPRESENTATION_PATHS_BY_SET_AND_NAME[feature_set_name][
                DBSCAN_PILOT_REPRESENTATION_NAME
            ]
        ).copy()

        pca_feature_columns = [
            column
            for column in pca_df.columns
            if column not in DEFAULT_METADATA_COLUMNS
        ]

        pilot_df = metadata_df.merge(
            pca_df[[MODEL_ID_COLUMN] + pca_feature_columns],
            on=MODEL_ID_COLUMN,
            how="inner",
            validate="one_to_one",
        )

        for comparison_universe_name in GEOGRAPHY_BASELINE_DIAGNOSTIC_UNIVERSES:
            grouping_columns = COMPARISON_UNIVERSE_GROUPING_COLUMNS[comparison_universe_name]
            universe_df = pilot_df.copy()

            if grouping_columns:
                universe_df["comparison_group_key"] = (
                    universe_df[grouping_columns]
                    .astype("string")
                    .fillna("Missing")
                    .agg(" | ".join, axis=1)
                )
            else:
                universe_df["comparison_group_key"] = "__all_rows__"

            for comparison_group_key, group_df in universe_df.groupby("comparison_group_key", dropna=False):
                if len(group_df) < DBSCAN_PILOT_MIN_ROWS_PER_GROUP:
                    continue

                eval_group_df = (
                    group_df.sample(
                        n=min(DBSCAN_PILOT_MAX_ROWS_PER_GROUP, len(group_df)),
                        random_state=ANOMALY_FRAMEWORK_RANDOM_STATE,
                    )
                    .copy()
                )

                X = eval_group_df[pca_feature_columns].to_numpy(dtype="float32", copy=False)

                neighbor_k = min(DBSCAN_PILOT_MIN_SAMPLES, len(eval_group_df) - 1)
                if neighbor_k < 2:
                    continue

                # Estimate eps within each comparison-universe group so the pilot
                # reflects the local density scale implied by that baseline choice.
                knn = NearestNeighbors(n_neighbors=neighbor_k)
                knn.fit(X)
                neighbor_distances = knn.kneighbors(X)[0][:, -1]
                eps = float(np.quantile(neighbor_distances, DBSCAN_PILOT_EPS_QUANTILE))

                dbscan_model = DBSCAN(
                    eps=eps,
                    min_samples=DBSCAN_PILOT_MIN_SAMPLES,
                    metric="euclidean",
                )
                labels = dbscan_model.fit_predict(X)

                eval_group_df["feature_set"] = feature_set_name
                eval_group_df["comparison_universe"] = comparison_universe_name
                eval_group_df["comparison_group_key"] = comparison_group_key
                eval_group_df["pilot_eps"] = round(eps, 6)
                eval_group_df["dbscan_label"] = labels
                eval_group_df["is_noise"] = labels == -1

                geography_row_records.append(
                    eval_group_df[
                        [
                            "feature_set",
                            MODEL_ID_COLUMN,
                            DATE_COLUMN,
                            TEMPORAL_BUCKET_COLUMN,
                            PRE_POST_CP_COLUMN,
                            POLICY_GEOGRAPHY_COLUMN,
                            BOROUGH_COLUMN,
                            ZONE_ID_COLUMN,
                            ZONE_COLUMN,
                            "comparison_universe",
                            "comparison_group_key",
                            "pilot_eps",
                            "dbscan_label",
                            "is_noise",
                        ]
                    ]
                )

    dbscan_geography_row_df = pd.concat(
        geography_row_records,
        ignore_index=True,
    )
    dbscan_geography_row_df.to_parquet(
        DBSCAN_GEOGRAPHY_DIAGNOSTIC_ROW_PATH,
        index=False,
    )
    output_action = "written"
else:
    dbscan_geography_row_df = pd.read_parquet(DBSCAN_GEOGRAPHY_DIAGNOSTIC_ROW_PATH)
    output_action = "loaded_existing"

row_materialization_status_df = pd.DataFrame(
    [
        {
            "output_path": str(DBSCAN_GEOGRAPHY_DIAGNOSTIC_ROW_PATH),
            "rows": len(dbscan_geography_row_df),
            "columns": len(dbscan_geography_row_df.columns),
            "output_action": output_action,
            "status": "pass" if len(dbscan_geography_row_df) > 0 else "review",
        }
    ]
)

display(row_materialization_status_df)

print("Row-level DBSCAN geography-baseline diagnostics materialized.")

,output_path,rows,columns,output_action,status
0,/datasets/_deepnote_work/pipeline_data/3.3.1.i...,404805,14,loaded_existing,pass


Row-level DBSCAN geography-baseline diagnostics materialized.


Findings\. The row\-level DBSCAN pilot output was materialized successfully and is now cached as the shared geography\-diagnostic asset for the next baseline analyses\.

Overall Noise Share\. With the row\-level pilot output in place, we can start with the simplest sanity check: anomaly\-like volume\. The main metric here is noise\_share, the fraction of evaluated rows that DBSCAN labels as noise\. Before asking where the noise points land, we want to know whether geography\-aware scoring is dramatically changing how much anomaly\-like mass the pilot produces at all\.

In [36]:
# ---------------------------------------------------------------------
# Summarize anomaly-like volume across candidate baselines
# ---------------------------------------------------------------------

dbscan_geography_volume_df = (
    dbscan_geography_row_df.groupby(
        ["feature_set", "comparison_universe"],
        as_index=False,
    )
    .agg(
        evaluated_rows=(MODEL_ID_COLUMN, "size"),
        noise_rows=("is_noise", "sum"),
    )
)

dbscan_geography_volume_df["noise_share"] = (
    dbscan_geography_volume_df["noise_rows"]
    / dbscan_geography_volume_df["evaluated_rows"]
).round(4)

display(
    dbscan_geography_volume_df
    .sort_values(["feature_set", "comparison_universe"])
    .reset_index(drop=True)
)

noise_share_fig = px.line(
    dbscan_geography_volume_df,
    x="comparison_universe",
    y="noise_share",
    color="feature_set",
    markers=True,
    category_orders={
        "comparison_universe": [
            "global_all_rows",
            "same_temporal_bucket",
            "same_temporal_bucket_and_policy_geography",
        ],
        "feature_set": ["bus", "subway", "taxi", "fhvhv", "multimodal"],
    },
    labels={
        "comparison_universe": "Comparison universe",
        "noise_share": "Noise share",
        "feature_set": "Feature set",
    },
    title="DBSCAN Pilot Noise Share by Comparison Universe",
)

noise_share_fig.update_layout(
    width=900,
    height=420,
    template="plotly_white",
    legend=dict(
        orientation="h",
        yanchor="top",
        y=-0.22,
        xanchor="center",
        x=0.5,
    ),
    margin=dict(t=70, b=90, l=70, r=30),
)

noise_share_fig.update_xaxes(
    tickmode="array",
    tickvals=[
        "global_all_rows",
        "same_temporal_bucket",
        "same_temporal_bucket_and_policy_geography",
    ],
    ticktext=[
        "Global",
        "Temporal",
        "Temporal +<br>geography",
    ],
)

noise_share_fig.show()

print("Anomaly-like volume across candidate baselines summarized.")

,feature_set,comparison_universe,evaluated_rows,noise_rows,noise_share
0,bus,global_all_rows,4000,81,0.0202
1,bus,same_temporal_bucket,34093,774,0.0227
2,bus,same_temporal_bucket_and_policy_geography,42114,956,0.0227
3,fhvhv,global_all_rows,4000,94,0.0235
4,fhvhv,same_temporal_bucket,34290,931,0.0272
5,fhvhv,same_temporal_bucket_and_policy_geography,42340,1081,0.0255
6,multimodal,global_all_rows,4000,112,0.0280
7,multimodal,same_temporal_bucket,34290,926,0.0270
8,multimodal,same_temporal_bucket_and_policy_geography,42340,1129,0.0267
9,subway,global_all_rows,4000,134,0.0335


Anomaly-like volume across candidate baselines summarized.


Findings\. The volume check shows that changing the comparison universe usually does not radically change how much anomaly\-like mass DBSCAN produces, but the effect is not perfectly uniform across modalities\. Bus, Taxi, FHVHV, and Multimodal move only modestly across the three baselines, while Subway shows a more noticeable decline in noise share as the baseline becomes more conditioned\. Even there, though, the change is still relatively small in absolute terms, which suggests that the geography\-aware baseline is not simply flooding or starving the system of anomalies\. That makes the next geography\-focused diagnostics more credible, because the main story is about where anomaly\-like points land, not just how many there are\.

Concentration Analysis on Core Geos\. Now we move from anomaly volume to anomaly concentration\. The key metrics here are the geography\-specific noise\_lift values, which compare a geography’s share of anomaly\-like points with its share of the evaluated rows\. A lift near 1\.0 means proportional representation, while higher values mean that geography is absorbing more anomaly\-like points than its baseline prevalence would suggest\.

In [37]:
# ---------------------------------------------------------------------
# Summarize anomaly-like geography concentration relative to source prevalence
# ---------------------------------------------------------------------

should_rebuild_geography_summary = (
    REBUILD_BASELINE_FEASIBILITY_TABLES
    or (not DBSCAN_GEOGRAPHY_DIAGNOSTIC_SUMMARY_PATH.exists())
)

if should_rebuild_geography_summary:
    geography_summary_records = []
    for feature_set_name in MODEL_FEATURE_SET_NAMES:
        feature_rows_df = dbscan_geography_row_df.loc[
            dbscan_geography_row_df["feature_set"].eq(feature_set_name)
        ].copy()
        for comparison_universe_name in GEOGRAPHY_BASELINE_DIAGNOSTIC_UNIVERSES:
            universe_rows_df = feature_rows_df.loc[
                feature_rows_df["comparison_universe"].eq(comparison_universe_name)
            ].copy()
            source_rows = len(universe_rows_df)
            noise_rows_df = universe_rows_df.loc[universe_rows_df["is_noise"]].copy()
            noise_rows = len(noise_rows_df)
            source_zone_counts = universe_rows_df[ZONE_ID_COLUMN].value_counts(dropna=False)
            noise_zone_counts = noise_rows_df[ZONE_ID_COLUMN].value_counts(dropna=False)
            source_manhattan_share = float(
                universe_rows_df[BOROUGH_COLUMN].eq("Manhattan").mean()
            )
            noise_manhattan_share = (
                float(noise_rows_df[BOROUGH_COLUMN].eq("Manhattan").mean())
                if noise_rows
                else np.nan
            )
            source_cbd_share = float(
                universe_rows_df[POLICY_GEOGRAPHY_COLUMN].eq("cbd").mean()
            )
            noise_cbd_share = (
                float(noise_rows_df[POLICY_GEOGRAPHY_COLUMN].eq("cbd").mean())
                if noise_rows
                else np.nan
            )
            source_top10_zone_share = float(source_zone_counts.head(10).sum() / source_rows)
            noise_top10_zone_share = (
                float(noise_zone_counts.head(10).sum() / noise_rows)
                if noise_rows
                else np.nan
            )
            geography_summary_records.append(
                {
                    "feature_set": feature_set_name,
                    "comparison_universe": comparison_universe_name,
                    "evaluated_rows": source_rows,
                    "noise_rows": noise_rows,
                    "noise_share": round(noise_rows / source_rows, 4) if source_rows else np.nan,
                    "source_manhattan_share": round(source_manhattan_share, 4),
                    "noise_manhattan_share": round(noise_manhattan_share, 4)
                    if pd.notna(noise_manhattan_share)
                    else np.nan,
                    "manhattan_noise_lift": round(
                        noise_manhattan_share / source_manhattan_share, 3
                    )
                    if source_manhattan_share > 0 and pd.notna(noise_manhattan_share)
                    else np.nan,
                    "source_cbd_share": round(source_cbd_share, 4),
                    "noise_cbd_share": round(noise_cbd_share, 4)
                    if pd.notna(noise_cbd_share)
                    else np.nan,
                    "cbd_noise_lift": round(noise_cbd_share / source_cbd_share, 3)
                    if source_cbd_share > 0 and pd.notna(noise_cbd_share)
                    else np.nan,
                    "source_top10_zone_share": round(source_top10_zone_share, 4),
                    "noise_top10_zone_share": round(noise_top10_zone_share, 4)
                    if pd.notna(noise_top10_zone_share)
                    else np.nan,
                    "top10_zone_noise_lift": round(
                        noise_top10_zone_share / source_top10_zone_share, 3
                    )
                    if source_top10_zone_share > 0 and pd.notna(noise_top10_zone_share)
                    else np.nan,
                }
            )
    dbscan_geography_summary_df = pd.DataFrame(geography_summary_records)
    dbscan_geography_summary_df.to_parquet(
        DBSCAN_GEOGRAPHY_DIAGNOSTIC_SUMMARY_PATH,
        index=False,
    )
else:
    dbscan_geography_summary_df = pd.read_parquet(
        DBSCAN_GEOGRAPHY_DIAGNOSTIC_SUMMARY_PATH
    )

display(
    dbscan_geography_summary_df
    .sort_values(["feature_set", "comparison_universe"])
    .reset_index(drop=True)
)

# ---------------------------------------------------------------------
# Visualize geography-baseline concentration tradeoffs
# ---------------------------------------------------------------------

comparison_plot_df = (
    dbscan_geography_summary_df
    .loc[
        dbscan_geography_summary_df["comparison_universe"].isin(
            [
                "same_temporal_bucket",
                "same_temporal_bucket_and_policy_geography",
            ]
        )
    ][
        [
            "feature_set",
            "comparison_universe",
            "manhattan_noise_lift",
            "cbd_noise_lift",
            "top10_zone_noise_lift",
        ]
    ]
    .copy()
)

metric_specs = [
    ("manhattan_noise_lift", "Manhattan noise lift"),
    ("cbd_noise_lift", "CBD noise lift"),
    ("top10_zone_noise_lift", "Top-10 zone noise lift"),
]

universe_label_map = {
    "same_temporal_bucket": "Temporal only",
    "same_temporal_bucket_and_policy_geography": "Temporal + geography",
}

feature_set_display_map = {
    "bus": "Bus",
    "subway": "Subway",
    "taxi": "Taxi",
    "fhvhv": "FHVHV",
    "multimodal": "Multimodal",
}

temporal_color = BRAND_COLORS["seafoam"]
geography_color = BRAND_COLORS["terracotta"]
connector_color = "rgba(11, 78, 74, 0.28)"

geography_comparison_fig = make_subplots(
    rows=1,
    cols=3,
    shared_yaxes=True,
    subplot_titles=[label for _, label in metric_specs],
    horizontal_spacing=0.08,
)

for col_number, (metric_column, metric_label) in enumerate(metric_specs, start=1):
    temporal_df = (
        comparison_plot_df
        .loc[comparison_plot_df["comparison_universe"].eq("same_temporal_bucket")]
        [["feature_set", metric_column]]
        .rename(columns={metric_column: "temporal_value"})
    )
    geography_df = (
        comparison_plot_df
        .loc[
            comparison_plot_df["comparison_universe"].eq(
                "same_temporal_bucket_and_policy_geography"
            )
        ][["feature_set", metric_column]]
        .rename(columns={metric_column: "geography_value"})
    )
    paired_df = (
        temporal_df
        .merge(geography_df, on="feature_set", how="inner", validate="one_to_one")
        .sort_values("feature_set")
        .reset_index(drop=True)
    )
    paired_df["feature_set_display"] = paired_df["feature_set"].map(feature_set_display_map)

    for _, row in paired_df.iterrows():
        geography_comparison_fig.add_trace(
            go.Scatter(
                x=[row["temporal_value"], row["geography_value"]],
                y=[row["feature_set_display"], row["feature_set_display"]],
                mode="lines",
                line={"color": connector_color, "width": 2},
                hoverinfo="skip",
                showlegend=False,
            ),
            row=1,
            col=col_number,
        )

    geography_comparison_fig.add_trace(
        go.Scatter(
            x=paired_df["temporal_value"],
            y=paired_df["feature_set_display"],
            mode="markers",
            marker={
                "color": temporal_color,
                "size": 10,
                "line": {"color": BRAND_COLORS["dark_teal"], "width": 1},
            },
            name="Temporal only",
            legendgroup="Temporal only",
            showlegend=(col_number == 1),
            hovertemplate=(
                "Feature set: %{y}<br>"
                f"Baseline: {universe_label_map['same_temporal_bucket']}<br>"
                f"{metric_label}: %{{x:.2f}}<extra></extra>"
            ),
        ),
        row=1,
        col=col_number,
    )
    geography_comparison_fig.add_trace(
        go.Scatter(
            x=paired_df["geography_value"],
            y=paired_df["feature_set_display"],
            mode="markers",
            marker={
                "color": geography_color,
                "size": 10,
                "line": {"color": BRAND_COLORS["dark_teal"], "width": 1},
            },
            name="Temporal + geography",
            legendgroup="Temporal + geography",
            showlegend=(col_number == 1),
            hovertemplate=(
                "Feature set: %{y}<br>"
                f"Baseline: {universe_label_map['same_temporal_bucket_and_policy_geography']}<br>"
                f"{metric_label}: %{{x:.2f}}<extra></extra>"
            ),
        ),
        row=1,
        col=col_number,
    )
    geography_comparison_fig.update_xaxes(
        title_text="Lift relative to source prevalence",
        row=1,
        col=col_number,
    )

apply_brand_plotly_layout(
    geography_comparison_fig,
    title="Geography-Aware Baseline Diagnostic",
    width=950,
    height=430,
)

geography_comparison_fig.update_layout(
    legend={
        "orientation": "h",
        "yanchor": "top",
        "y": -0.18,
        "xanchor": "center",
        "x": 0.5,
        "bgcolor": "rgba(255,255,255,0.85)",
        "bordercolor": BRAND_COLORS["seafoam"],
        "borderwidth": 1,
        "font": {"color": BRAND_COLORS["dark_teal"]},
    },
    margin={"t": 70, "b": 90, "l": 90, "r": 30},
)

geography_comparison_fig.show()

print("Geography concentration relative to source prevalence summarized.")

,feature_set,comparison_universe,evaluated_rows,noise_rows,noise_share,source_manhattan_share,noise_manhattan_share,manhattan_noise_lift,source_cbd_share,noise_cbd_share,cbd_noise_lift,source_top10_zone_share,noise_top10_zone_share,top10_zone_noise_lift
0,bus,global_all_rows,4000,81,0.0203,0.2707,0.3580,1.322,0.1520,0.1728,1.137,0.0630,0.5802,9.210
1,bus,same_temporal_bucket,34093,774,0.0227,0.2536,0.2610,1.029,0.1461,0.1654,1.132,0.0507,0.5568,10.980
2,bus,same_temporal_bucket_and_policy_geography,42114,956,0.0227,0.2829,0.2197,0.777,0.1756,0.1464,0.834,0.0549,0.5858,10.661
3,fhvhv,global_all_rows,4000,94,0.0235,0.2405,0.2340,0.973,0.1455,0.1702,1.170,0.0625,0.5745,9.191
4,fhvhv,same_temporal_bucket,34290,931,0.0272,0.2511,0.1697,0.676,0.1521,0.1257,0.826,0.0535,0.6187,11.561
5,fhvhv,same_temporal_bucket_and_policy_geography,42340,1081,0.0255,0.2741,0.1795,0.655,0.1735,0.1397,0.805,0.0568,0.6050,10.651
6,multimodal,global_all_rows,4000,112,0.0280,0.2405,0.4107,1.708,0.1455,0.3482,2.393,0.0625,0.6429,10.286
7,multimodal,same_temporal_bucket,34290,926,0.0270,0.2511,0.2916,1.161,0.1521,0.2376,1.562,0.0535,0.6037,11.281
8,multimodal,same_temporal_bucket_and_policy_geography,42340,1129,0.0267,0.2741,0.1922,0.701,0.1735,0.1143,0.659,0.0568,0.6103,10.744
9,subway,global_all_rows,4000,134,0.0335,0.3415,0.8284,2.426,0.1995,0.7463,3.741,0.0938,0.7537,8.040


Geography concentration relative to source prevalence summarized.


Findings\. This is the core concentration test\. The three lift columns compare a geography’s share of anomaly\-like points with its share of the evaluated rows: values near 1\.0 mean roughly proportional representation, while larger values mean that geography is absorbing an outsized share of noise\. On that test, the strongest geography problem appears in Subway and Taxi, where the temporal\-only baseline still leaves Manhattan and especially CBD substantially overrepresented, but adding policy geography pulls those lifts much closer to parity\. Multimodal also improves meaningfully on Manhattan and CBD lift, while Bus shifts more modestly and FHVHV was not strongly core\-dominated to begin with\. The chart makes that pattern easy to see through the leftward shifts\. The important caveat is the top\-10\-zone panel: even after adding geography, dominant zones still absorb far more anomaly\-like points than their baseline prevalence would predict\. So geography\-aware scoring does not eliminate concentration altogether, but it does materially reduce the specific Midtown/CBD distortion that motivated this diagnostic\.

Persistence / Repetition analysis\. Concentration alone is not the whole story\. Even if Manhattan or CBD lift improves, anomaly\-like points may still keep recurring in the same small set of zones over and over\. This block therefore looks at persistence: how many distinct zones receive anomaly\-like points, how much of the anomaly mass stays inside the top zones, and how many dates those dominant zones continue to appear\.

In [38]:
# ---------------------------------------------------------------------
# Summarize repeated-zone persistence of anomaly-like outputs
# ---------------------------------------------------------------------

should_rebuild_zone_persistence = (
    REBUILD_BASELINE_FEASIBILITY_TABLES
    or (not DBSCAN_GEOGRAPHY_ZONE_PERSISTENCE_PATH.exists())
)

if should_rebuild_zone_persistence:
    zone_persistence_records = []
    noise_rows_df = dbscan_geography_row_df.loc[dbscan_geography_row_df["is_noise"]].copy()
    for feature_set_name in MODEL_FEATURE_SET_NAMES:
        for comparison_universe_name in GEOGRAPHY_BASELINE_DIAGNOSTIC_UNIVERSES:
            universe_noise_df = noise_rows_df.loc[
                noise_rows_df["feature_set"].eq(feature_set_name)
                & noise_rows_df["comparison_universe"].eq(comparison_universe_name)
            ].copy()
            if universe_noise_df.empty:
                zone_persistence_records.append(
                    {
                        "feature_set": feature_set_name,
                        "comparison_universe": comparison_universe_name,
                        "distinct_noise_zones": 0,
                        "top_zone_noise_share": np.nan,
                        "top10_zone_noise_share": np.nan,
                        "top_zone_distinct_dates": np.nan,
                        "top10_zone_distinct_dates": np.nan,
                    }
                )
                continue
            zone_counts = universe_noise_df[ZONE_ID_COLUMN].value_counts(dropna=False)
            top_zone_ids = zone_counts.head(10).index.tolist()
            top_zone_rows_df = universe_noise_df.loc[
                universe_noise_df[ZONE_ID_COLUMN].eq(zone_counts.index[0])
            ]
            top10_zone_rows_df = universe_noise_df.loc[
                universe_noise_df[ZONE_ID_COLUMN].isin(top_zone_ids)
            ]
            zone_persistence_records.append(
                {
                    "feature_set": feature_set_name,
                    "comparison_universe": comparison_universe_name,
                    "distinct_noise_zones": int(universe_noise_df[ZONE_ID_COLUMN].nunique(dropna=True)),
                    "top_zone_noise_share": round(float(zone_counts.iloc[0] / len(universe_noise_df)), 4),
                    "top10_zone_noise_share": round(float(zone_counts.head(10).sum() / len(universe_noise_df)), 4),
                    "top_zone_distinct_dates": int(top_zone_rows_df[DATE_COLUMN].nunique(dropna=True)),
                    "top10_zone_distinct_dates": int(top10_zone_rows_df[DATE_COLUMN].nunique(dropna=True)),
                }
            )
    dbscan_zone_persistence_df = pd.DataFrame(zone_persistence_records)
    dbscan_zone_persistence_df.to_parquet(
        DBSCAN_GEOGRAPHY_ZONE_PERSISTENCE_PATH,
        index=False,
    )
else:
    dbscan_zone_persistence_df = pd.read_parquet(DBSCAN_GEOGRAPHY_ZONE_PERSISTENCE_PATH)

display(
    dbscan_zone_persistence_df
    .sort_values(["feature_set", "comparison_universe"])
    .reset_index(drop=True)
)

# ---------------------------------------------------------------------
# Visualize repeated-zone persistence of anomaly-like outputs
# ---------------------------------------------------------------------

persistence_plot_df = (
    dbscan_zone_persistence_df
    .loc[
        dbscan_zone_persistence_df["comparison_universe"].isin(
            [
                "same_temporal_bucket",
                "same_temporal_bucket_and_policy_geography",
            ]
        )
    ][
        [
            "feature_set",
            "comparison_universe",
            "distinct_noise_zones",
            "top10_zone_noise_share",
            "top10_zone_distinct_dates",
        ]
    ]
    .copy()
)

persistence_metric_specs = [
    ("distinct_noise_zones", "Distinct noise zones"),
    ("top10_zone_noise_share", "Top-10 zone noise share"),
    ("top10_zone_distinct_dates", "Top-10 zone distinct dates"),
]

temporal_color = BRAND_COLORS["seafoam"]
geography_color = BRAND_COLORS["terracotta"]
connector_color = "rgba(11, 78, 74, 0.28)"

persistence_fig = make_subplots(
    rows=1,
    cols=3,
    shared_yaxes=True,
    subplot_titles=[label for _, label in persistence_metric_specs],
    horizontal_spacing=0.08,
)

for col_number, (metric_column, metric_label) in enumerate(
    persistence_metric_specs,
    start=1,
):
    temporal_df = (
        persistence_plot_df
        .loc[persistence_plot_df["comparison_universe"].eq("same_temporal_bucket")]
        [["feature_set", metric_column]]
        .rename(columns={metric_column: "temporal_value"})
    )
    geography_df = (
        persistence_plot_df
        .loc[
            persistence_plot_df["comparison_universe"].eq(
                "same_temporal_bucket_and_policy_geography"
            )
        ][["feature_set", metric_column]]
        .rename(columns={metric_column: "geography_value"})
    )
    paired_df = (
        temporal_df
        .merge(geography_df, on="feature_set", how="inner", validate="one_to_one")
        .sort_values("feature_set")
        .reset_index(drop=True)
    )
    paired_df["feature_set_display"] = paired_df["feature_set"].map(feature_set_display_map)

    for _, row in paired_df.iterrows():
        persistence_fig.add_trace(
            go.Scatter(
                x=[row["temporal_value"], row["geography_value"]],
                y=[row["feature_set_display"], row["feature_set_display"]],
                mode="lines",
                line={"color": connector_color, "width": 2},
                hoverinfo="skip",
                showlegend=False,
            ),
            row=1,
            col=col_number,
        )

    persistence_fig.add_trace(
        go.Scatter(
            x=paired_df["temporal_value"],
            y=paired_df["feature_set_display"],
            mode="markers",
            marker={
                "color": temporal_color,
                "size": 10,
                "line": {"color": BRAND_COLORS["dark_teal"], "width": 1},
            },
            name="Temporal only",
            legendgroup="Temporal only",
            showlegend=(col_number == 1),
            hovertemplate=(
                "Feature set: %{y}<br>"
                "Baseline: Temporal only<br>"
                f"{metric_label}: %{{x}}<extra></extra>"
            ),
        ),
        row=1,
        col=col_number,
    )
    persistence_fig.add_trace(
        go.Scatter(
            x=paired_df["geography_value"],
            y=paired_df["feature_set_display"],
            mode="markers",
            marker={
                "color": geography_color,
                "size": 10,
                "line": {"color": BRAND_COLORS["dark_teal"], "width": 1},
            },
            name="Temporal + geography",
            legendgroup="Temporal + geography",
            showlegend=(col_number == 1),
            hovertemplate=(
                "Feature set: %{y}<br>"
                "Baseline: Temporal + geography<br>"
                f"{metric_label}: %{{x}}<extra></extra>"
            ),
        ),
        row=1,
        col=col_number,
    )
    persistence_fig.update_xaxes(title_text=metric_label, row=1, col=col_number)

apply_brand_plotly_layout(
    persistence_fig,
    title="Repeated-Zone Persistence Diagnostic",
    width=950,
    height=430,
)

persistence_fig.update_layout(
    legend={
        "orientation": "h",
        "yanchor": "top",
        "y": -0.18,
        "xanchor": "center",
        "x": 0.5,
        "bgcolor": "rgba(255,255,255,0.85)",
        "bordercolor": BRAND_COLORS["seafoam"],
        "borderwidth": 1,
        "font": {"color": BRAND_COLORS["dark_teal"]},
    },
    margin={"t": 70, "b": 90, "l": 90, "r": 30},
)

persistence_fig.show()

print("Repeated-zone persistence of anomaly-like outputs summarized.")

,feature_set,comparison_universe,distinct_noise_zones,top_zone_noise_share,top10_zone_noise_share,top_zone_distinct_dates,top10_zone_distinct_dates
0,bus,global_all_rows,40,0.0988,0.5802,8,44
1,bus,same_temporal_bucket,114,0.2067,0.5568,124,306
2,bus,same_temporal_bucket_and_policy_geography,109,0.1883,0.5858,143,375
3,fhvhv,global_all_rows,43,0.1170,0.5745,11,50
4,fhvhv,same_temporal_bucket,117,0.1182,0.6187,24,206
5,fhvhv,same_temporal_bucket_and_policy_geography,141,0.1230,0.6050,29,263
6,multimodal,global_all_rows,37,0.1250,0.6429,13,68
7,multimodal,same_temporal_bucket,115,0.0961,0.6037,29,197
8,multimodal,same_temporal_bucket_and_policy_geography,132,0.1151,0.6103,34,251
9,subway,global_all_rows,37,0.1716,0.7537,23,99


Repeated-zone persistence of anomaly-like outputs summarized.


Findings\. The repetition analysis asks a different but complementary question: even if concentration improves, are anomaly\-like points still coming back to the same few zones over and over? The clearest improvement again appears in Subway, where adding policy geography increases the number of distinct noise zones and sharply reduces the share absorbed by the top 10 zones, suggesting that anomaly\-like outputs are becoming less trapped in the same dominant core locations\. Taxi shows a milder version of the same pattern, with broader zone coverage and slightly lower top\-10 concentration\. Bus also broadens somewhat, while FHVHV changes only marginally\. Multimodal remains the mixed case: anomaly\-like points spread across more zones and dates, but top\-10\-zone concentration stays stubbornly high\. Taken together, this says geography\-aware scoring is most valuable where repeated core\-zone dominance is the real problem, especially in Subway and Taxi\.

In [39]:
# ---------------------------------------------------------------------
# Build a geography-baseline diagnostic scorecard
# ---------------------------------------------------------------------

# Start from the two baselines that matter for the geography decision.
scorecard_base_df = (
    dbscan_geography_summary_df
    .loc[
        dbscan_geography_summary_df["comparison_universe"].isin(
            [
                "same_temporal_bucket",
                "same_temporal_bucket_and_policy_geography",
            ]
        )
    ][
        [
            "feature_set",
            "comparison_universe",
            "noise_share",
            "manhattan_noise_lift",
            "cbd_noise_lift",
            "top10_zone_noise_lift",
        ]
    ]
    .merge(
        dbscan_zone_persistence_df[
            [
                "feature_set",
                "comparison_universe",
                "distinct_noise_zones",
                "top10_zone_noise_share",
                "top10_zone_distinct_dates",
            ]
        ],
        on=["feature_set", "comparison_universe"],
        how="inner",
        validate="one_to_one",
    )
    .copy()
)

# 1) Stable anomaly volume: closer to the feature-set temporal/geography median is better.
scorecard_base_df["feature_set_noise_median"] = (
    scorecard_base_df.groupby("feature_set")["noise_share"].transform("median")
)
scorecard_base_df["stable_anomaly_volume_raw"] = -(
    scorecard_base_df["noise_share"] - scorecard_base_df["feature_set_noise_median"]
).abs()

# 2) Less core-geo concentration: lower average lift across Manhattan/CBD/top10 is better.
scorecard_base_df["core_geo_concentration_raw"] = -(
    scorecard_base_df[
        ["manhattan_noise_lift", "cbd_noise_lift", "top10_zone_noise_lift"]
    ].mean(axis=1)
)

# 3) Less repeated-zone dominance: lower top-10 share and broader zone/date spread is better.
scorecard_base_df["repeated_zone_dominance_raw"] = (
    -scorecard_base_df["top10_zone_noise_share"]
    + 0.002 * scorecard_base_df["distinct_noise_zones"]
    + 0.0005 * scorecard_base_df["top10_zone_distinct_dates"]
)

score_columns = [
    ("stable_anomaly_volume_raw", "stable_anomaly_volume_score"),
    ("core_geo_concentration_raw", "less_core_geo_concentration_score"),
    ("repeated_zone_dominance_raw", "less_repeated_zone_dominance_score"),
]

for raw_col, score_col in score_columns:
    group_min = scorecard_base_df.groupby("feature_set")[raw_col].transform("min")
    group_max = scorecard_base_df.groupby("feature_set")[raw_col].transform("max")
    denom = (group_max - group_min).replace(0, np.nan)
    scorecard_base_df[score_col] = np.where(
        denom.notna(),
        (scorecard_base_df[raw_col] - group_min) / denom,
        1.0,
    )

geography_decision_scorecard_df = (
    scorecard_base_df[
        [
            "feature_set",
            "comparison_universe",
            "noise_share",
            "manhattan_noise_lift",
            "cbd_noise_lift",
            "top10_zone_noise_lift",
            "top10_zone_noise_share",
            "distinct_noise_zones",
            "top10_zone_distinct_dates",
            "stable_anomaly_volume_score",
            "less_core_geo_concentration_score",
            "less_repeated_zone_dominance_score",
        ]
    ]
    .sort_values(["feature_set", "comparison_universe"])
    .reset_index(drop=True)
)

display(geography_decision_scorecard_df)

metric_order = [
    "stable_anomaly_volume_score",
    "less_core_geo_concentration_score",
    "less_repeated_zone_dominance_score",
]
metric_label_map = {
    "stable_anomaly_volume_score": "Stable<br>anomaly<br>volume",
    "less_core_geo_concentration_score": "Less<br>core-geo<br>concentration",
    "less_repeated_zone_dominance_score": "Less<br>repeated-zone<br>dominance",
}

fig = make_subplots(
    rows=2,
    cols=3,
    subplot_titles=["Bus", "Subway", "", "Taxi", "FHVHV", "Multimodal"],
    horizontal_spacing=0.01,
    vertical_spacing=0.1,
)

feature_set_layout = [
    ("bus", 1, 1),
    ("subway", 1, 2),
    ("taxi", 2, 1),
    ("fhvhv", 2, 2),
    ("multimodal", 2, 3),
]

for feature_set_name, row_num, col_num in feature_set_layout:
    panel_df = (
        geography_decision_scorecard_df.loc[
            geography_decision_scorecard_df["feature_set"].eq(feature_set_name)
        ]
        .set_index("comparison_universe")
        .loc[
            [
                "same_temporal_bucket",
                "same_temporal_bucket_and_policy_geography",
            ]
        ]
        .reset_index()
    )
    z_matrix = panel_df[metric_order].to_numpy()

    fig.add_trace(
        go.Heatmap(
            z=z_matrix,
            x=[metric_label_map[m] for m in metric_order],
            y=["Temporal", "Temporal +<br>geography"],
            zmin=0,
            zmax=1,
            colorscale=BRAND_DIVERGING_SEQUENCE,
            showscale=(feature_set_name == "bus"),
            colorbar={
                "title": {
                    "text": "Relative score",
                    "font": {"color": BRAND_COLORS["dark_teal"]},
                },
                "tickfont": {"color": BRAND_COLORS["dark_teal"]},
                "len": 0.70,
                "y": 0.54,
            }
            if feature_set_name == "bus"
            else None,
            text=np.round(z_matrix, 2),
            texttemplate="%{text:.2f}",
            xgap=5,
            ygap=3,
            hovertemplate="Relative score: %{z:.2f}<extra></extra>",
        ),
        row=row_num,
        col=col_num,
    )
    fig.update_xaxes(
        showticklabels=(row_num == 2),
        tickangle=0,
        row=row_num,
        col=col_num,
    )
    fig.update_yaxes(
        showticklabels=(col_num == 1),
        ticklabelstandoff=10,
        row=row_num,
        col=col_num,
    )

fig.update_xaxes(visible=False, row=1, col=3)
fig.update_yaxes(visible=False, row=1, col=3)

apply_brand_plotly_layout(
    fig,
    title="Geography-Aware Baseline Decision Scorecards by Feature Set",
    width=1000,
    height=500,
)

fig.update_layout(
    margin={"l": 180, "r": 95, "t": 80, "b": 120},
    annotations=[
        *fig.layout.annotations,
        {
            "text": "Comparison universe",
            "xref": "paper",
            "yref": "paper",
            "x": -0.19,
            "y": 0.50,
            "showarrow": False,
            "textangle": -90,
            "font": {"size": 13, "color": BRAND_COLORS["dark_teal"]},
        },
        {
            "text": "Score dimension",
            "xref": "paper",
            "yref": "paper",
            "x": 0.52,
            "y": -0.27,
            "showarrow": False,
            "font": {"size": 13, "color": BRAND_COLORS["dark_teal"]},
        },
    ],
)

fig.show()

print("Geography decision scorecard prepared.")

,feature_set,comparison_universe,noise_share,manhattan_noise_lift,cbd_noise_lift,top10_zone_noise_lift,top10_zone_noise_share,distinct_noise_zones,top10_zone_distinct_dates,stable_anomaly_volume_score,less_core_geo_concentration_score,less_repeated_zone_dominance_score
0,bus,same_temporal_bucket,0.0227,1.029,1.132,10.980,0.5568,114,306,1.0,0.0,1.0
1,bus,same_temporal_bucket_and_policy_geography,0.0227,0.777,0.834,10.661,0.5858,109,375,1.0,1.0,0.0
2,fhvhv,same_temporal_bucket,0.0272,0.676,0.826,11.561,0.6187,117,206,1.0,0.0,0.0
3,fhvhv,same_temporal_bucket_and_policy_geography,0.0255,0.655,0.805,10.651,0.6050,141,263,1.0,1.0,1.0
4,multimodal,same_temporal_bucket,0.0270,1.161,1.562,11.281,0.6037,115,197,0.0,0.0,0.0
5,multimodal,same_temporal_bucket_and_policy_geography,0.0267,0.701,0.659,10.744,0.6103,132,251,1.0,1.0,1.0
6,subway,same_temporal_bucket,0.0261,2.039,3.129,7.711,0.5951,125,430,1.0,0.0,0.0
7,subway,same_temporal_bucket_and_policy_geography,0.0241,1.008,0.917,4.442,0.3514,145,294,1.0,1.0,1.0
8,taxi,same_temporal_bucket,0.0255,1.788,1.665,8.455,0.4525,164,150,1.0,0.0,0.0
9,taxi,same_temporal_bucket_and_policy_geography,0.0235,1.179,0.888,7.889,0.4481,192,162,0.0,1.0,1.0


Geography decision scorecard prepared.


Findings\. The final scorecard brings the three geography tests together: stable anomaly volume, less core\-geography concentration, and less repeated\-zone dominance\. Because this scorecard compares only two baselines within each modality, the coloring is binary: 1 means that baseline performs better on that metric for that feature set, and 0 means it performs worse\. On that framing, Temporal \+ geography is the stronger option almost everywhere that matters for the geography question\. It clearly wins all three criteria for Bus, Taxi, and Multimodal, and it wins the two geography\-specific criteria for Subway while giving up only a small amount on anomaly\-volume stability\. FHVHV is the mild exception: geography helps concentration and repetition, but not volume\. Overall, the scorecard makes the downstream decision much easier to justify: temporal bucketing remains the baseline backbone, and adding policy\_geography\_class is worth it because it consistently reduces geography\-driven distortion without causing unstable anomaly volume\.

> 🎯 Decision\. The geography\-aware baseline will use policy\_geography\_class, not borough or individual zone IDs, because it is the simplest segmentation that directly targets the Midtown/CBD concentration problem while preserving workable group sizes and a consistent interpretation across all modalities\.

🎯 Decision\. Downstream anomaly generation will use comparison universes partitioned by both temporal\_bucket and policy\_geography\_class across all five modalities\. The earlier pilot showed that temporal segmentation was broadly useful, and the geography\-aware diagnostic showed that adding policy geography materially reduced Midtown/CBD dominance in the modalities where that problem was most pronounced, especially Subway and Taxi\. Even though the gain was weaker for Bus and FHVHV, using one shared baseline rule across all modalities keeps the anomaly pipeline simpler, more consistent, and easier to compare across methods\.

### Record the comparison universes advancing downstream

We’ve finished the baseline\-universe decision, so this last step is just to record it cleanly for the downstream anomaly notebooks\. At this point, we are no longer comparing candidate universes on equal footing: we are identifying the one shared default that advances, plus the few targeted exceptions or diagnostics that still need to be preserved\.

This block writes the comparison\-universe decision into a small handoff table that later notebooks can read directly\. It keeps only the universes that still serve a real downstream purpose and drops the ones that were useful during analysis but are no longer part of the active framework\.

In [40]:
# ---------------------------------------------------------------------
# Record the comparison universes advancing downstream
# ---------------------------------------------------------------------

COMPARISON_UNIVERSE_HANDOFF_PATH = (
    FINAL_331_DIR / "comparison_universe_handoff.parquet"
)

should_rebuild_comparison_handoff = (
    REBUILD_COMPARISON_UNIVERSE_TABLES
    or (not COMPARISON_UNIVERSE_HANDOFF_PATH.exists())
)

if should_rebuild_comparison_handoff:
    comparison_universe_handoff_records = []

    for feature_set_name in MODEL_FEATURE_SET_NAMES:
        for _, spec_row in comparison_universe_spec_df.iterrows():
            comparison_universe_name = spec_row["comparison_universe"]

            if comparison_universe_name == "same_temporal_bucket_and_policy_geography":
                downstream_status = "shared_default"
                downstream_role = "primary_scoring_universe"
                rationale = (
                    "Advances as the shared downstream default because it preserves "
                    "temporal operating-regime context while reducing core-geography "
                    "dominance where that problem is most pronounced."
                )

            elif comparison_universe_name == "same_temporal_bucket_and_pre_post_cp":
                if feature_set_name == "taxi":
                    downstream_status = "taxi_policy_sanity_check"
                    downstream_role = "targeted_taxi_check"
                    rationale = (
                        "Retained only for Taxi as a targeted policy-period sanity check "
                        "because Taxi showed the strongest pre/post structural drift in 3.2.2."
                    )
                else:
                    downstream_status = "not_advanced"
                    downstream_role = "analysis_only"
                    rationale = (
                        "Analyzed during framework design but not advanced because the extra "
                        "pre/post split did not earn a shared downstream role."
                    )

            elif comparison_universe_name == "same_temporal_bucket":
                downstream_status = "not_advanced"
                downstream_role = "analysis_only"
                rationale = (
                    "Useful as an intermediate analytical baseline, but superseded by "
                    "temporal plus policy geography once the geography diagnostics showed "
                    "that the narrower universe added real downstream value."
                )

            elif comparison_universe_name == "global_all_rows":
                downstream_status = "not_advanced"
                downstream_role = "analysis_only"
                rationale = (
                    "Rejected for downstream scoring because the global baseline is too "
                    "coarse and hides meaningful temporal and geographic regime differences."
                )

            else:
                downstream_status = "review"
                downstream_role = "unassigned"
                rationale = "Unexpected comparison universe; review required."

            comparison_universe_handoff_records.append(
                {
                    "feature_set": feature_set_name,
                    "comparison_universe": comparison_universe_name,
                    "scope_type": spec_row["scope_type"],
                    "grouping_columns": spec_row["grouping_columns"],
                    "downstream_status": downstream_status,
                    "downstream_role": downstream_role,
                    "rationale": rationale,
                }
            )

    comparison_universe_handoff_df = pd.DataFrame(comparison_universe_handoff_records)
    comparison_universe_handoff_df.to_parquet(
        COMPARISON_UNIVERSE_HANDOFF_PATH,
        index=False,
    )
    output_action = "written"
else:
    comparison_universe_handoff_df = pd.read_parquet(COMPARISON_UNIVERSE_HANDOFF_PATH)
    output_action = "loaded_existing"

comparison_universe_handoff_display_df = (
    comparison_universe_handoff_df
    .sort_values(["feature_set", "downstream_status", "comparison_universe"])
    .reset_index(drop=True)
)

display(comparison_universe_handoff_display_df)

comparison_universe_handoff_summary_df = (
    comparison_universe_handoff_df.groupby(
        ["comparison_universe", "downstream_status", "downstream_role"],
        as_index=False,
    ).agg(
        feature_sets=("feature_set", lambda values: ", ".join(sorted(values))),
    )
)

display(comparison_universe_handoff_summary_df)

print(f"Comparison-universe handoff recorded ({output_action}).")

,feature_set,comparison_universe,scope_type,grouping_columns,downstream_status,downstream_role,rationale
0,bus,global_all_rows,global,none,not_advanced,analysis_only,Rejected for downstream scoring because the gl...
1,bus,same_temporal_bucket,temporal,temporal_bucket,not_advanced,analysis_only,"Useful as an intermediate analytical baseline,..."
2,bus,same_temporal_bucket_and_pre_post_cp,temporal_policy,"temporal_bucket, pre_post_cp",not_advanced,analysis_only,Analyzed during framework design but not advan...
3,bus,same_temporal_bucket_and_policy_geography,temporal_geography,"temporal_bucket, policy_geography_class",shared_default,primary_scoring_universe,Advances as the shared downstream default beca...
4,fhvhv,global_all_rows,global,none,not_advanced,analysis_only,Rejected for downstream scoring because the gl...
5,fhvhv,same_temporal_bucket,temporal,temporal_bucket,not_advanced,analysis_only,"Useful as an intermediate analytical baseline,..."
6,fhvhv,same_temporal_bucket_and_pre_post_cp,temporal_policy,"temporal_bucket, pre_post_cp",not_advanced,analysis_only,Analyzed during framework design but not advan...
7,fhvhv,same_temporal_bucket_and_policy_geography,temporal_geography,"temporal_bucket, policy_geography_class",shared_default,primary_scoring_universe,Advances as the shared downstream default beca...
8,multimodal,global_all_rows,global,none,not_advanced,analysis_only,Rejected for downstream scoring because the gl...
9,multimodal,same_temporal_bucket,temporal,temporal_bucket,not_advanced,analysis_only,"Useful as an intermediate analytical baseline,..."


,comparison_universe,downstream_status,downstream_role,feature_sets
0,global_all_rows,not_advanced,analysis_only,"bus, fhvhv, multimodal, subway, taxi"
1,same_temporal_bucket,not_advanced,analysis_only,"bus, fhvhv, multimodal, subway, taxi"
2,same_temporal_bucket_and_policy_geography,shared_default,primary_scoring_universe,"bus, fhvhv, multimodal, subway, taxi"
3,same_temporal_bucket_and_pre_post_cp,not_advanced,analysis_only,"bus, fhvhv, multimodal, subway"
4,same_temporal_bucket_and_pre_post_cp,taxi_policy_sanity_check,targeted_taxi_check,taxi


Comparison-universe handoff recorded (loaded_existing).


Findings\. The comparison\-universe decision is now explicit: same\_temporal\_bucket\_and\_policy\_geography advances as the shared downstream scoring universe across all five modalities, while same\_temporal\_bucket and global\_all\_rows are retained only as analysis\-stage reference universes and do not advance into active anomaly generation\. The only extra exception is same\_temporal\_bucket\_and\_pre\_post\_cp, which remains available solely as a Taxi\-specific policy sanity check\.

### Section Summary

This section settled the baseline\-universe question for downstream anomaly detection\. The DBSCAN pilot showed that a global baseline is too coarse and that temporal bucketing improves local anomaly structure without destabilizing anomaly\-like volume too aggressively\. The geography diagnostics then answered the more important question: in the systems where anomaly\-like points were still collapsing onto core geographies, especially Subway and Taxi, adding policy\_geography\_class materially reduced Midtown/CBD dominance and repeated\-zone concentration\. Taken together, those results justify a shared downstream comparison universe defined by temporal\_bucket \+ policy\_geography\_class, with only a limited Taxi\-specific pre/post policy sanity check retained outside that default\.

## 3\.3\.1\.4 Select the Shared Downstream Anomaly Representation

The comparison\-universe question is now settled: downstream anomaly scoring will compare observations within the same temporal\_bucket and policy\_geography\_class\. With that baseline fixed, the next decision is which representation should actually carry the anomaly\-detection work\. This section compares the three candidate representations inherited from 3\.2\.4, raw\_scaled, pca\_80pct, and umap\_pca\_to\_umap\_10d, under the shared scoring universe we just selected\.

The central question is not which representation looks best as a visualization, but which one behaves best as an anomaly\-detection space\. We want a representation that preserves meaningful anomaly structure without collapsing into a simple detector of high activity, centrality, or system scale\. The section therefore uses lightweight anomaly\-behavior diagnostics to compare whether anomaly\-like outputs are stable, interpretable, and not overly dominated by obvious high\-intensity regimes\.

The output of this section should be a single shared downstream anomaly representation\. Once that choice is made, the notebook can handle any remaining representation\-specific follow\-up, such as the Taxi PCA caveat or the raw\-feature absolute\-scale question, only if those issues are still relevant\.

### Define the representation comparison setup

The baseline question is now settled, so this subsection just fixes the comparison contract for the representation bakeoff\. We will keep one scoring universe, one pilot method, and one shared set of diagnostics so that Raw, PCA, and UMAP are being judged on the same terms\.

In [41]:
# ---------------------------------------------------------------------
# Define the shared anomaly-representation comparison setup
# ---------------------------------------------------------------------

REPRESENTATION_BAKEOFF_NAMES = [
    "raw_scaled",
    "pca_80pct",
    "umap_pca_to_umap_10d",
]

SHARED_ANOMALY_COMPARISON_UNIVERSE = "same_temporal_bucket_and_policy_geography"

# Keep the pilot method lightweight and consistent across representations.
REPRESENTATION_BAKEOFF_METHOD = "dbscan"
REPRESENTATION_BAKEOFF_MIN_SAMPLES = 15
REPRESENTATION_BAKEOFF_EPS_QUANTILE = 0.95
REPRESENTATION_BAKEOFF_MAX_ROWS_PER_GROUP = 4_000
REPRESENTATION_BAKEOFF_MIN_ROWS_PER_GROUP = 100

REPRESENTATION_BAKEOFF_SUMMARY_PATH = (
    INTERMEDIATE_331_DIR / "representation_bakeoff_summary.parquet"
)
REPRESENTATION_BAKEOFF_GROUP_DETAIL_PATH = (
    INTERMEDIATE_331_DIR / "representation_bakeoff_group_detail.parquet"
)
REPRESENTATION_BAKEOFF_ACTIVITY_DIAGNOSTIC_PATH = (
    INTERMEDIATE_331_DIR / "representation_bakeoff_activity_diagnostic.parquet"
)

representation_bakeoff_setup_df = pd.DataFrame(
    [
        {
            "comparison_universe": SHARED_ANOMALY_COMPARISON_UNIVERSE,
            "candidate_representations": ", ".join(REPRESENTATION_BAKEOFF_NAMES),
            "pilot_method": REPRESENTATION_BAKEOFF_METHOD,
            "min_samples": REPRESENTATION_BAKEOFF_MIN_SAMPLES,
            "eps_quantile": REPRESENTATION_BAKEOFF_EPS_QUANTILE,
            "max_rows_per_group": REPRESENTATION_BAKEOFF_MAX_ROWS_PER_GROUP,
            "min_rows_per_group": REPRESENTATION_BAKEOFF_MIN_ROWS_PER_GROUP,
            "random_state": ANOMALY_FRAMEWORK_RANDOM_STATE,
        }
    ]
)

representation_bakeoff_path_df = pd.DataFrame(
    [
        {
            "artifact": "representation_bakeoff_summary",
            "path": str(REPRESENTATION_BAKEOFF_SUMMARY_PATH),
        },
        {
            "artifact": "representation_bakeoff_group_detail",
            "path": str(REPRESENTATION_BAKEOFF_GROUP_DETAIL_PATH),
        },
        {
            "artifact": "representation_bakeoff_activity_diagnostic",
            "path": str(REPRESENTATION_BAKEOFF_ACTIVITY_DIAGNOSTIC_PATH),
        },
    ]
)

display(representation_bakeoff_setup_df)
display(representation_bakeoff_path_df)

print("Shared anomaly-representation comparison setup defined.")

,comparison_universe,candidate_representations,pilot_method,min_samples,eps_quantile,max_rows_per_group,min_rows_per_group,random_state
0,same_temporal_bucket_and_policy_geography,"raw_scaled, pca_80pct, umap_pca_to_umap_10d",dbscan,15,0.95,4000,100,42


,artifact,path
0,representation_bakeoff_summary,/datasets/_deepnote_work/pipeline_data/3.3.1.i...
1,representation_bakeoff_group_detail,/datasets/_deepnote_work/pipeline_data/3.3.1.i...
2,representation_bakeoff_activity_diagnostic,/datasets/_deepnote_work/pipeline_data/3.3.1.i...


Shared anomaly-representation comparison setup defined.


Findings\. The representation bakeoff is now fixed to one shared scoring universe, same\_temporal\_bucket\_and\_policy\_geography, and one shared lightweight DBSCAN pilot, so Raw, PCA, and UMAP will be compared under the same anomaly framework rather than under slightly different conditions\.

### Run a lightweight anomaly\-behavior pilot across Raw, PCA, and UMAP

This block applies the same lightweight DBSCAN pilot to all three candidate representations under the shared downstream comparison universe\. The point is not to generate final anomaly labels; it is to create a fair, reusable first readout of how raw\_scaled, pca\_80pct, and umap\_pca\_to\_umap\_10d behave when they are asked to separate anomaly\-like points inside the same temporal\-and\-geography regime\.

In [42]:
# ---------------------------------------------------------------------
# Run a lightweight anomaly pilot across Raw, PCA, and UMAP
# ---------------------------------------------------------------------

if "ANOMALY_CALIBRATION_REPRESENTATION_PATHS_BY_SET_AND_NAME" not in globals():
    raise NameError(
        "ANOMALY_CALIBRATION_REPRESENTATION_PATHS_BY_SET_AND_NAME is not defined. "
        "Run 3.3.1.2 Build Shared Anomaly Calibration Samples first."
    )

should_rebuild_representation_bakeoff = (
    REBUILD_SELF_NORMALIZATION_DIAGNOSTICS
    or (not REPRESENTATION_BAKEOFF_SUMMARY_PATH.exists())
    or (not REPRESENTATION_BAKEOFF_GROUP_DETAIL_PATH.exists())
)

if should_rebuild_representation_bakeoff:
    bakeoff_summary_records = []
    bakeoff_group_detail_records = []

    for feature_set_name in MODEL_FEATURE_SET_NAMES:
        metadata_df = pd.read_parquet(
            ANOMALY_CALIBRATION_METADATA_PATHS_BY_SET[feature_set_name],
            columns=[
                MODEL_ID_COLUMN,
                TEMPORAL_BUCKET_COLUMN,
                POLICY_GEOGRAPHY_COLUMN,
            ],
        ).copy()

        metadata_df["comparison_group_key"] = (
            metadata_df[[TEMPORAL_BUCKET_COLUMN, POLICY_GEOGRAPHY_COLUMN]]
            .astype("string")
            .fillna("Missing")
            .agg(" | ".join, axis=1)
        )

        for representation_name in REPRESENTATION_BAKEOFF_NAMES:
            representation_df = pd.read_parquet(
                ANOMALY_CALIBRATION_REPRESENTATION_PATHS_BY_SET_AND_NAME[feature_set_name][representation_name]
            ).copy()

            representation_feature_columns = [
                column
                for column in representation_df.columns
                if column not in DEFAULT_METADATA_COLUMNS
            ]

            pilot_df = metadata_df.merge(
                representation_df[[MODEL_ID_COLUMN] + representation_feature_columns],
                on=MODEL_ID_COLUMN,
                how="inner",
                validate="one_to_one",
            )

            group_result_records = []

            for comparison_group_key, group_df in pilot_df.groupby("comparison_group_key", dropna=False):
                group_rows = len(group_df)

                if group_rows < REPRESENTATION_BAKEOFF_MIN_ROWS_PER_GROUP:
                    group_result_records.append(
                        {
                            "feature_set": feature_set_name,
                            "representation_name": representation_name,
                            "comparison_universe": SHARED_ANOMALY_COMPARISON_UNIVERSE,
                            "comparison_group_key": comparison_group_key,
                            "group_rows": group_rows,
                            "rows_evaluated": 0,
                            "eps": np.nan,
                            "noise_share": np.nan,
                            "cluster_count": np.nan,
                            "largest_cluster_share": np.nan,
                            "status": "skipped_small_group",
                        }
                    )
                    continue

                eval_group_df = (
                    group_df.sample(
                        n=min(REPRESENTATION_BAKEOFF_MAX_ROWS_PER_GROUP, group_rows),
                        random_state=ANOMALY_FRAMEWORK_RANDOM_STATE,
                    )
                    .copy()
                )

                X = eval_group_df[representation_feature_columns].to_numpy(
                    dtype="float32",
                    copy=False,
                )

                neighbor_k = min(REPRESENTATION_BAKEOFF_MIN_SAMPLES, len(eval_group_df) - 1)
                if neighbor_k < 2:
                    group_result_records.append(
                        {
                            "feature_set": feature_set_name,
                            "representation_name": representation_name,
                            "comparison_universe": SHARED_ANOMALY_COMPARISON_UNIVERSE,
                            "comparison_group_key": comparison_group_key,
                            "group_rows": group_rows,
                            "rows_evaluated": len(eval_group_df),
                            "eps": np.nan,
                            "noise_share": np.nan,
                            "cluster_count": np.nan,
                            "largest_cluster_share": np.nan,
                            "status": "skipped_too_few_neighbors",
                        }
                    )
                    continue

                # Estimate eps separately inside each temporal+geography group so
                # each representation is judged against the same local-density logic.
                knn = NearestNeighbors(n_neighbors=neighbor_k)
                knn.fit(X)
                neighbor_distances = knn.kneighbors(X)[0][:, -1]
                eps = float(np.quantile(neighbor_distances, REPRESENTATION_BAKEOFF_EPS_QUANTILE))

                dbscan_model = DBSCAN(
                    eps=eps,
                    min_samples=REPRESENTATION_BAKEOFF_MIN_SAMPLES,
                    metric="euclidean",
                )
                labels = dbscan_model.fit_predict(X)

                noise_mask = labels == -1
                non_noise_labels = labels[~noise_mask]

                cluster_count = len(set(non_noise_labels)) if len(non_noise_labels) else 0
                largest_cluster_share = (
                    pd.Series(non_noise_labels).value_counts(normalize=True).iloc[0]
                    if cluster_count > 0
                    else 0.0
                )

                group_result_records.append(
                    {
                        "feature_set": feature_set_name,
                        "representation_name": representation_name,
                        "comparison_universe": SHARED_ANOMALY_COMPARISON_UNIVERSE,
                        "comparison_group_key": comparison_group_key,
                        "group_rows": group_rows,
                        "rows_evaluated": len(eval_group_df),
                        "eps": round(eps, 6),
                        "noise_share": round(float(noise_mask.mean()), 4),
                        "cluster_count": int(cluster_count),
                        "largest_cluster_share": round(float(largest_cluster_share), 4),
                        "status": "evaluated",
                    }
                )

            group_result_df = pd.DataFrame(group_result_records)
            bakeoff_group_detail_records.append(group_result_df)

            evaluated_groups_df = group_result_df.loc[group_result_df["status"].eq("evaluated")].copy()

            rows_evaluated = int(evaluated_groups_df["rows_evaluated"].sum())
            groups_evaluated = int(len(evaluated_groups_df))
            groups_skipped = int(len(group_result_df) - groups_evaluated)

            if rows_evaluated > 0:
                weight_vector = evaluated_groups_df["rows_evaluated"].to_numpy(dtype="float64")
                weighted_noise_share = float(
                    np.average(evaluated_groups_df["noise_share"], weights=weight_vector)
                )
                weighted_cluster_count = float(
                    np.average(evaluated_groups_df["cluster_count"], weights=weight_vector)
                )
                weighted_largest_cluster_share = float(
                    np.average(evaluated_groups_df["largest_cluster_share"], weights=weight_vector)
                )
            else:
                weighted_noise_share = np.nan
                weighted_cluster_count = np.nan
                weighted_largest_cluster_share = np.nan

            bakeoff_summary_records.append(
                {
                    "feature_set": feature_set_name,
                    "representation_name": representation_name,
                    "comparison_universe": SHARED_ANOMALY_COMPARISON_UNIVERSE,
                    "groups_evaluated": groups_evaluated,
                    "groups_skipped": groups_skipped,
                    "rows_evaluated": rows_evaluated,
                    "weighted_noise_share": round(weighted_noise_share, 4) if pd.notna(weighted_noise_share) else np.nan,
                    "weighted_cluster_count": round(weighted_cluster_count, 4) if pd.notna(weighted_cluster_count) else np.nan,
                    "weighted_largest_cluster_share": round(weighted_largest_cluster_share, 4) if pd.notna(weighted_largest_cluster_share) else np.nan,
                    "status": "pass" if groups_evaluated > 0 else "review",
                }
            )

    representation_bakeoff_summary_df = pd.DataFrame(bakeoff_summary_records)
    representation_bakeoff_group_detail_df = pd.concat(
        bakeoff_group_detail_records,
        ignore_index=True,
    )

    representation_bakeoff_summary_df.to_parquet(
        REPRESENTATION_BAKEOFF_SUMMARY_PATH,
        index=False,
    )
    representation_bakeoff_group_detail_df.to_parquet(
        REPRESENTATION_BAKEOFF_GROUP_DETAIL_PATH,
        index=False,
    )
    output_action = "written"
else:
    representation_bakeoff_summary_df = pd.read_parquet(REPRESENTATION_BAKEOFF_SUMMARY_PATH)
    representation_bakeoff_group_detail_df = pd.read_parquet(REPRESENTATION_BAKEOFF_GROUP_DETAIL_PATH)
    output_action = "loaded_existing"

representation_bakeoff_status_df = pd.DataFrame(
    [
        {
            "artifact": "representation_bakeoff_summary",
            "path": str(REPRESENTATION_BAKEOFF_SUMMARY_PATH),
            "rows": len(representation_bakeoff_summary_df),
            "output_action": output_action,
            "status": "pass" if len(representation_bakeoff_summary_df) > 0 else "review",
        },
        {
            "artifact": "representation_bakeoff_group_detail",
            "path": str(REPRESENTATION_BAKEOFF_GROUP_DETAIL_PATH),
            "rows": len(representation_bakeoff_group_detail_df),
            "output_action": output_action,
            "status": "pass" if len(representation_bakeoff_group_detail_df) > 0 else "review",
        },
    ]
)

display(representation_bakeoff_status_df)

print("Lightweight anomaly pilot across Raw, PCA, and UMAP completed.")

,artifact,path,rows,output_action,status
0,representation_bakeoff_summary,/datasets/_deepnote_work/pipeline_data/3.3.1.i...,15,loaded_existing,pass
1,representation_bakeoff_group_detail,/datasets/_deepnote_work/pipeline_data/3.3.1.i...,450,loaded_existing,pass


Lightweight anomaly pilot across Raw, PCA, and UMAP completed.


Findings\. The shared representation bakeoff is now materialized and cached, so Raw, PCA, and UMAP can be compared under the same temporal\-plus\-policy\-geography scoring universe rather than through mismatched pilot conditions\.

In [43]:
# ---------------------------------------------------------------------
# Validate representation bakeoff coverage
# ---------------------------------------------------------------------

representation_bakeoff_validation_df = (
    representation_bakeoff_summary_df[
        [
            "feature_set",
            "representation_name",
            "groups_evaluated",
            "groups_skipped",
            "rows_evaluated",
            "weighted_noise_share",
            "weighted_cluster_count",
            "weighted_largest_cluster_share",
            "status",
        ]
    ]
    .sort_values(["feature_set", "representation_name"])
    .reset_index(drop=True)
)

representation_bakeoff_coverage_df = (
    representation_bakeoff_group_detail_df.groupby(
        ["feature_set", "representation_name", "status"],
        as_index=False,
    )
    .size()
    .rename(columns={"size": "group_count"})
    .sort_values(["feature_set", "representation_name", "status"])
    .reset_index(drop=True)
)

display(representation_bakeoff_validation_df)
display(representation_bakeoff_coverage_df)

assert representation_bakeoff_validation_df["status"].eq("pass").all(), (
    "One or more representation bakeoff runs did not complete successfully."
)

print("Representation bakeoff coverage validated.")

,feature_set,representation_name,groups_evaluated,groups_skipped,rows_evaluated,weighted_noise_share,weighted_cluster_count,weighted_largest_cluster_share,status
0,bus,pca_80pct,30,0,42114,0.0227,1.6035,0.9655,pass
1,bus,raw_scaled,30,0,42114,0.0243,1.3873,0.9674,pass
2,bus,umap_pca_to_umap_10d,30,0,42114,0.0214,8.0194,0.8752,pass
3,fhvhv,pca_80pct,30,0,42340,0.0255,1.7704,0.9623,pass
4,fhvhv,raw_scaled,30,0,42340,0.0299,1.5652,0.9661,pass
5,fhvhv,umap_pca_to_umap_10d,30,0,42340,0.0178,3.6194,0.9322,pass
6,multimodal,pca_80pct,30,0,42340,0.0267,1.5363,0.9949,pass
7,multimodal,raw_scaled,30,0,42340,0.0283,1.3603,0.9949,pass
8,multimodal,umap_pca_to_umap_10d,30,0,42340,0.0271,10.6960,0.8636,pass
9,subway,pca_80pct,30,0,44474,0.0241,2.1335,0.9420,pass


,feature_set,representation_name,status,group_count
0,bus,pca_80pct,evaluated,30
1,bus,raw_scaled,evaluated,30
2,bus,umap_pca_to_umap_10d,evaluated,30
3,fhvhv,pca_80pct,evaluated,30
4,fhvhv,raw_scaled,evaluated,30
5,fhvhv,umap_pca_to_umap_10d,evaluated,30
6,multimodal,pca_80pct,evaluated,30
7,multimodal,raw_scaled,evaluated,30
8,multimodal,umap_pca_to_umap_10d,evaluated,30
9,subway,pca_80pct,evaluated,30


Representation bakeoff coverage validated.


Findings\. The representation bakeoff ran across the full modality\-by\-representation grid and successfully evaluated the expected temporal\-plus\-policy\-geography groups, so the downstream comparison is based on the local anomaly framework we just selected rather than on one global citywide run\.

This next check opens up the bakeoff one level further\. Instead of only showing the rolled\-up totals, it shows the actual temporal\-by\-policy\-geography group sizes and the capped row contribution from each group, which makes the fixed rows\_evaluated totals much easier to interpret\.

In [44]:
# ---------------------------------------------------------------------
# Validate per-group bakeoff coverage and row contributions
# ---------------------------------------------------------------------

representation_bakeoff_group_coverage_df = (
    representation_bakeoff_group_detail_df[
        [
            "feature_set",
            "representation_name",
            "comparison_group_key",
            "group_rows",
            "rows_evaluated",
            "status",
        ]
    ]
    .sort_values(
        ["feature_set", "representation_name", "comparison_group_key"]
    )
    .reset_index(drop=True)
)

representation_bakeoff_group_rollup_df = (
    representation_bakeoff_group_coverage_df.groupby(
        ["feature_set", "representation_name"],
        as_index=False,
    ).agg(
        groups_evaluated=("comparison_group_key", "count"),
        min_group_rows=("group_rows", "min"),
        median_group_rows=("group_rows", "median"),
        max_group_rows=("group_rows", "max"),
        rows_evaluated=("rows_evaluated", "sum"),
        groups_hit_row_cap=("rows_evaluated", lambda s: int((s == REPRESENTATION_BAKEOFF_MAX_ROWS_PER_GROUP).sum())),
        groups_below_row_cap=("rows_evaluated", lambda s: int((s < REPRESENTATION_BAKEOFF_MAX_ROWS_PER_GROUP).sum())),
    )
    .sort_values(["feature_set", "representation_name"])
    .reset_index(drop=True)
)

display(representation_bakeoff_group_rollup_df)
display(representation_bakeoff_group_coverage_df)

print("Per-group bakeoff coverage and row contributions validated.")

,feature_set,representation_name,groups_evaluated,min_group_rows,median_group_rows,max_group_rows,rows_evaluated,groups_hit_row_cap,groups_below_row_cap
0,bus,pca_80pct,30,190,811.5,5640,42114,5,25
1,bus,raw_scaled,30,190,811.5,5640,42114,5,25
2,bus,umap_pca_to_umap_10d,30,190,811.5,5640,42114,5,25
3,fhvhv,pca_80pct,30,214,805.0,5532,42340,5,25
4,fhvhv,raw_scaled,30,214,805.0,5532,42340,5,25
5,fhvhv,umap_pca_to_umap_10d,30,214,805.0,5532,42340,5,25
6,multimodal,pca_80pct,30,214,805.0,5532,42340,5,25
7,multimodal,raw_scaled,30,214,805.0,5532,42340,5,25
8,multimodal,umap_pca_to_umap_10d,30,214,805.0,5532,42340,5,25
9,subway,pca_80pct,30,254,1017.0,5137,44474,5,25


,feature_set,representation_name,comparison_group_key,group_rows,rows_evaluated,status
0,bus,pca_80pct,weekday_am_peak | cbd,1080,1080,evaluated
1,bus,pca_80pct,weekday_am_peak | gateway_or_adjacent,487,487,evaluated
2,bus,pca_80pct,weekday_am_peak | outside,5640,4000,evaluated
3,bus,pca_80pct,weekday_evening | cbd,1097,1097,evaluated
4,bus,pca_80pct,weekday_evening | gateway_or_adjacent,471,471,evaluated
5,bus,pca_80pct,weekday_evening | outside,5583,4000,evaluated
6,bus,pca_80pct,weekday_midday | cbd,1090,1090,evaluated
7,bus,pca_80pct,weekday_midday | gateway_or_adjacent,544,544,evaluated
8,bus,pca_80pct,weekday_midday | outside,5565,4000,evaluated
9,bus,pca_80pct,weekday_overnight | cbd,1079,1079,evaluated


Per-group bakeoff coverage and row contributions validated.


Findings\. This validation makes the row totals easier to trust\. The candidate representations are being compared on the same temporal\-by\-policy\-geography groups within each modality, so the total rows\_evaluated naturally matches across Raw, PCA, and UMAP\. The unevenness is still there, but it appears at the group level through group\_rows; the rolled\-up total is simply the sum of each group’s capped contribution\.

### Compare Anomaly Volume and Structure Across Representations

The shared scoring universe is now fixed, so the next question is simpler: when Raw, PCA, and UMAP are all asked to separate anomaly\-like points inside the same temporal\-and\-geography regime, which one behaves best as an anomaly\-detection space? This section focuses on the behavior of the anomaly outputs themselves rather than on abstract representation quality\. We want to see which representation produces the most credible anomaly\-like volume, the clearest local structure, and the least collapse into one dominant notion of “normal\.”

The comparison starts with volume and structure because those are the most immediate signs of whether a representation is behaving sensibly under the chosen baseline\. A good anomaly space should not explode into brittle fragments, collapse nearly everything into one oversized cluster, or generate anomaly\-like mass only because the geometry itself is poorly scaled\. Each block below therefore pairs a compact table with a matching chart so we can see both the exact values and the overall pattern\.

Anomaly\-like Volume\. We’ll start with anomaly\-like volume\. The key metric here is weighted\_noise\_share, which tells us what fraction of evaluated rows DBSCAN labels as noise after the temporal\-plus\-policy\-geography grouping is respected\. If one representation produces a much larger or smaller anomaly\-like volume than the others, that is an early sign that its geometry may be distorting the scoring process rather than simply surfacing unusual mobility states\.

In [45]:
# ---------------------------------------------------------------------
# Compare anomaly-like volume across representations
# ---------------------------------------------------------------------

representation_label_map = {
    "raw_scaled": "Raw",
    "pca_80pct": "PCA",
    "umap_pca_to_umap_10d": "UMAP",
}
feature_set_order = ["bus", "subway", "taxi", "fhvhv", "multimodal"]
representation_order = ["raw_scaled", "pca_80pct", "umap_pca_to_umap_10d"]
feature_set_color_map = {
    feature_set_name: BRAND_COLOR_SEQUENCE[index % len(BRAND_COLOR_SEQUENCE)]
    for index, feature_set_name in enumerate(feature_set_order)
}

volume_comparison_df = (
    representation_bakeoff_summary_df[
        [
            "feature_set",
            "representation_name",
            "rows_evaluated",
            "weighted_noise_share",
            "status",
        ]
    ]
    .copy()
)

volume_comparison_df["representation_label"] = (
    volume_comparison_df["representation_name"].map(representation_label_map)
)

volume_comparison_display_df = (
    volume_comparison_df
    .sort_values(
        ["feature_set", "representation_name"],
        key=lambda series: series.map(
            {
                **{name: i for i, name in enumerate(feature_set_order)},
                **{name: i for i, name in enumerate(representation_order)},
            }
        ) if series.name in {"feature_set", "representation_name"} else series,
    )
    .reset_index(drop=True)
)

display(volume_comparison_display_df)

volume_fig = px.line(
    volume_comparison_df,
    x="representation_label",
    y="weighted_noise_share",
    color="feature_set",
    markers=True,
    category_orders={
        "representation_label": ["Raw", "PCA", "UMAP"],
        "feature_set": feature_set_order,
    },
    color_discrete_map=feature_set_color_map,
    labels={
        "representation_label": "Representation",
        "weighted_noise_share": "Weighted noise share",
        "feature_set": "Feature set",
    },
    title="Anomaly-Like Volume Across Raw, PCA, and UMAP",
)

volume_fig.update_traces(line={"width": 2}, marker={"size": 8})

apply_brand_plotly_layout(
    volume_fig,
    title="Anomaly-Like Volume Across Raw, PCA, and UMAP",
    width=900,
    height=430,
    legend_title="Feature set",
)

volume_fig.update_layout(
    legend={
        "title": {"text": "Feature set"},
        "orientation": "h",
        "yanchor": "top",
        "y": -0.18,
        "xanchor": "center",
        "x": 0.5,
        "bgcolor": "rgba(255,255,255,0.85)",
        "bordercolor": BRAND_COLORS["seafoam"],
        "borderwidth": 1,
        "font": {"color": BRAND_COLORS["dark_teal"]},
    },
    margin={"t": 70, "b": 90, "l": 70, "r": 30},
)

volume_fig.show()

print("Anomaly-like volume across representations summarized.")

,feature_set,representation_name,rows_evaluated,weighted_noise_share,status,representation_label
0,bus,raw_scaled,42114,0.0243,pass,Raw
1,bus,pca_80pct,42114,0.0227,pass,PCA
2,bus,umap_pca_to_umap_10d,42114,0.0214,pass,UMAP
3,subway,raw_scaled,44474,0.0239,pass,Raw
4,subway,pca_80pct,44474,0.0241,pass,PCA
5,subway,umap_pca_to_umap_10d,44474,0.0162,pass,UMAP
6,taxi,raw_scaled,42340,0.0239,pass,Raw
7,taxi,pca_80pct,42340,0.0235,pass,PCA
8,taxi,umap_pca_to_umap_10d,42340,0.0208,pass,UMAP
9,fhvhv,raw_scaled,42340,0.0299,pass,Raw


Anomaly-like volume across representations summarized.


Findings\. The volume comparison is fairly gentle for Raw and PCA, but more uneven once UMAP enters the picture\. Bus and Taxi stay close across all three representations, and Subway is also nearly unchanged between Raw and PCA, which suggests those two spaces are producing similar anomaly\-like volume under the shared baseline\. PCA is especially stable overall: in every modality, its noise share stays close to Raw while avoiding the larger swings seen in UMAP\. UMAP is the most variable representation, driving anomaly\-like volume sharply downward in Subway and FHVHV, modestly downward in Taxi, barely changing Bus, and slightly increasing Multimodal\. So at the volume level alone, PCA looks like the most conservative and stable reduced representation, while UMAP appears to be changing the anomaly surface more aggressively\.

Anomaly Structure\. Volume alone is not enough\. We also want to know whether each representation produces a sensible local structure once DBSCAN is run inside the chosen comparison universe\. The two main metrics here are weighted\_cluster\_count, which tells us how much local structure is being revealed, and weighted\_largest\_cluster\_share, which tells us how much of the evaluated mass is still being absorbed by one dominant cluster\.

In [46]:
# ---------------------------------------------------------------------
# Compare anomaly structure across representations
# ---------------------------------------------------------------------

structure_comparison_df = (
    representation_bakeoff_summary_df[
        [
            "feature_set",
            "representation_name",
            "rows_evaluated",
            "weighted_cluster_count",
            "weighted_largest_cluster_share",
            "status",
        ]
    ]
    .copy()
)

structure_comparison_df["representation_label"] = (
    structure_comparison_df["representation_name"].map(representation_label_map)
)

structure_comparison_display_df = (
    structure_comparison_df
    .sort_values(
        ["feature_set", "representation_name"],
        key=lambda series: series.map(
            {
                **{name: i for i, name in enumerate(feature_set_order)},
                **{name: i for i, name in enumerate(representation_order)},
            }
        ) if series.name in {"feature_set", "representation_name"} else series,
    )
    .reset_index(drop=True)
)

display(structure_comparison_display_df)

feature_set_color_map = {
    feature_set_name: BRAND_COLOR_SEQUENCE[index % len(BRAND_COLOR_SEQUENCE)]
    for index, feature_set_name in enumerate(feature_set_order)
}

structure_fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=["Weighted cluster count", "Weighted largest-cluster share"],
    horizontal_spacing=0.10,
)

for feature_set_name in feature_set_order:
    feature_df = structure_comparison_df.loc[
        structure_comparison_df["feature_set"].eq(feature_set_name)
    ].copy()
    feature_color = feature_set_color_map[feature_set_name]

    structure_fig.add_trace(
        go.Scatter(
            x=feature_df["representation_label"],
            y=feature_df["weighted_cluster_count"],
            mode="lines+markers",
            line={"color": feature_color, "width": 2},
            marker={"color": feature_color, "size": 8},
            name=feature_set_name,
            legendgroup=feature_set_name,
            showlegend=True,
            hovertemplate=(
                f"Feature set: {feature_set_name}<br>"
                "Representation: %{x}<br>"
                "Weighted cluster count: %{y:.2f}<extra></extra>"
            ),
        ),
        row=1,
        col=1,
    )
    structure_fig.add_trace(
        go.Scatter(
            x=feature_df["representation_label"],
            y=feature_df["weighted_largest_cluster_share"],
            mode="lines+markers",
            line={"color": feature_color, "width": 2},
            marker={"color": feature_color, "size": 8},
            name=feature_set_name,
            legendgroup=feature_set_name,
            showlegend=False,
            hovertemplate=(
                f"Feature set: {feature_set_name}<br>"
                "Representation: %{x}<br>"
                "Weighted largest-cluster share: %{y:.3f}<extra></extra>"
            ),
        ),
        row=1,
        col=2,
    )

structure_fig.update_xaxes(title_text="Representation", row=1, col=1)
structure_fig.update_xaxes(title_text="Representation", row=1, col=2)
structure_fig.update_yaxes(title_text="Cluster count", row=1, col=1)
structure_fig.update_yaxes(title_text="Largest-cluster share", row=1, col=2)

apply_brand_plotly_layout(
    structure_fig,
    title="Anomaly Structure Across Raw, PCA, and UMAP",
    width=950,
    height=430,
)

structure_fig.update_layout(
    legend={
        "orientation": "h",
        "yanchor": "top",
        "y": -0.18,
        "xanchor": "center",
        "x": 0.5,
        "bgcolor": "rgba(255,255,255,0.85)",
        "bordercolor": BRAND_COLORS["seafoam"],
        "borderwidth": 1,
        "font": {"color": BRAND_COLORS["dark_teal"]},
    },
    margin={"t": 70, "b": 90, "l": 70, "r": 30},
)

structure_fig.show()

print("Anomaly structure across representations summarized.")

,feature_set,representation_name,rows_evaluated,weighted_cluster_count,weighted_largest_cluster_share,status,representation_label
0,bus,raw_scaled,42114,1.3873,0.9674,pass,Raw
1,bus,pca_80pct,42114,1.6035,0.9655,pass,PCA
2,bus,umap_pca_to_umap_10d,42114,8.0194,0.8752,pass,UMAP
3,subway,raw_scaled,44474,1.7428,0.9468,pass,Raw
4,subway,pca_80pct,44474,2.1335,0.9420,pass,PCA
5,subway,umap_pca_to_umap_10d,44474,16.2103,0.3098,pass,UMAP
6,taxi,raw_scaled,42340,1.6618,0.9847,pass,Raw
7,taxi,pca_80pct,42340,1.7057,0.9678,pass,PCA
8,taxi,umap_pca_to_umap_10d,42340,9.0507,0.9011,pass,UMAP
9,fhvhv,raw_scaled,42340,1.5652,0.9661,pass,Raw


Anomaly structure across representations summarized.


Findings\. The structure comparison makes the tradeoff much sharper\. Raw and PCA behave similarly in every modality: both usually produce around one to two clusters per temporal\-and\-geography group, and both leave most evaluated rows inside one dominant cluster\. PCA is slightly better than Raw in Bus, Subway, and FHVHV, and roughly comparable in Taxi, but the overall pattern is still conservative\. UMAP, by contrast, changes the anomaly geometry dramatically\. It produces far more clusters in every modality, especially Subway, Multimodal, Taxi, and Bus, while also reducing the share absorbed by the largest cluster\. That means UMAP is clearly revealing much more local structure, but it is doing so in a far more aggressive way than either Raw or PCA\. So this block cuts both ways: if we want a representation that stays close to the original anomaly structure, PCA looks safer; if we want to force much stronger local separation, UMAP is the most assertive option by a wide margin\.

### Compare Anomaly Overlap Across Representations

Volume and structure told us how differently Raw, PCA, and UMAP behave in aggregate\. The next question is more concrete: are they actually flagging the same observations, or is UMAP carving out a meaningfully different anomaly\-like set?

We’ll start by materializing one row\-level DBSCAN pilot output for each representation under the shared downstream baseline\. That gives us a stable per\-row foundation for the overlap checks that follow, rather than trying to infer row\-level agreement from group summaries\.

Build Row\-level Pilot\. Before we compare agreement directly, we need one row\-level pilot artifact that records the DBSCAN assignment for each sampled observation under Raw, PCA, and UMAP\. This gives the overlap analysis a stable per\-row foundation instead of forcing us to infer agreement from group summaries\.

In [47]:
# ---------------------------------------------------------------------
# Materialize row-level representation pilot assignments
# ---------------------------------------------------------------------

REPRESENTATION_OVERLAP_ROW_LEVEL_PATH = (
    INTERMEDIATE_331_DIR / "representation_overlap_row_level.parquet"
)

should_rebuild_representation_overlap_row_level = (
    REBUILD_SELF_NORMALIZATION_DIAGNOSTICS
    or (not REPRESENTATION_OVERLAP_ROW_LEVEL_PATH.exists())
)

if should_rebuild_representation_overlap_row_level:
    if "ANOMALY_CALIBRATION_REPRESENTATION_PATHS_BY_SET_AND_NAME" not in globals():
        raise NameError(
            "ANOMALY_CALIBRATION_REPRESENTATION_PATHS_BY_SET_AND_NAME is not defined. "
            "Please rerun the shared calibration-sample materialization section first."
        )

    overlap_row_level_records = []
    overlap_row_level_write_records = []

    for feature_set_name in MODEL_FEATURE_SET_NAMES:
        metadata_df = pd.read_parquet(
            ANOMALY_CALIBRATION_METADATA_PATHS_BY_SET[feature_set_name],
            columns=[
                MODEL_ID_COLUMN,
                TEMPORAL_BUCKET_COLUMN,
                POLICY_GEOGRAPHY_COLUMN,
            ],
        ).copy()

        metadata_df["comparison_group_key"] = (
            metadata_df[TEMPORAL_BUCKET_COLUMN].astype("string")
            + " | "
            + metadata_df[POLICY_GEOGRAPHY_COLUMN].astype("string")
        )

        for representation_name in REPRESENTATION_NAMES:
            representation_df = pd.read_parquet(
                ANOMALY_CALIBRATION_REPRESENTATION_PATHS_BY_SET_AND_NAME[
                    feature_set_name
                ][representation_name]
            ).copy()

            input_columns = [
                column_name
                for column_name in representation_df.columns
                if column_name != MODEL_ID_COLUMN
            ]

            non_numeric_input_columns = [
                column_name
                for column_name in input_columns
                if not pd.api.types.is_numeric_dtype(representation_df[column_name])
            ]

            assert not non_numeric_input_columns, (
                f"{feature_set_name} / {representation_name} contains non-numeric "
                f"representation columns: {non_numeric_input_columns}"
            )

            working_df = metadata_df.merge(
                representation_df,
                on=MODEL_ID_COLUMN,
                how="inner",
            )

            feature_representation_frames = []

            for comparison_group_key, group_df in working_df.groupby(
                "comparison_group_key",
                dropna=False,
            ):
                group_rows = len(group_df)

                if group_rows < DBSCAN_PILOT_MIN_ROWS_PER_GROUP:
                    skipped_df = group_df.loc[:, [MODEL_ID_COLUMN]].copy()
                    skipped_df["feature_set"] = feature_set_name
                    skipped_df["representation_name"] = representation_name
                    skipped_df["comparison_universe"] = (
                        "same_temporal_bucket_and_policy_geography"
                    )
                    skipped_df["comparison_group_key"] = comparison_group_key
                    skipped_df["group_rows"] = group_rows
                    skipped_df["rows_evaluated"] = 0
                    skipped_df["dbscan_eps"] = np.nan
                    skipped_df["cluster_label"] = np.nan
                    skipped_df["is_anomaly_like"] = False
                    skipped_df["group_status"] = "skipped_small_group"
                    feature_representation_frames.append(skipped_df)
                    continue

                if group_rows > DBSCAN_PILOT_MAX_ROWS_PER_GROUP:
                    group_eval_df = group_df.sample(
                        n=DBSCAN_PILOT_MAX_ROWS_PER_GROUP,
                        random_state=ANOMALY_FRAMEWORK_RANDOM_STATE,
                    ).copy()
                else:
                    group_eval_df = group_df.copy()

                X = group_eval_df[input_columns].to_numpy(dtype="float32", copy=False)

                knn = NearestNeighbors(
                    n_neighbors=DBSCAN_PILOT_MIN_SAMPLES,
                    metric="euclidean",
                )
                knn.fit(X)

                distances, _ = knn.kneighbors(X)
                kth_distances = distances[:, DBSCAN_PILOT_MIN_SAMPLES - 1]
                eps_value = float(np.quantile(kth_distances, DBSCAN_PILOT_EPS_QUANTILE))

                cluster_labels = DBSCAN(
                    eps=eps_value,
                    min_samples=DBSCAN_PILOT_MIN_SAMPLES,
                    metric="euclidean",
                ).fit_predict(X)

                group_result_df = group_eval_df.loc[:, [MODEL_ID_COLUMN]].copy()
                group_result_df["feature_set"] = feature_set_name
                group_result_df["representation_name"] = representation_name
                group_result_df["comparison_universe"] = (
                    "same_temporal_bucket_and_policy_geography"
                )
                group_result_df["comparison_group_key"] = comparison_group_key
                group_result_df["group_rows"] = group_rows
                group_result_df["rows_evaluated"] = len(group_eval_df)
                group_result_df["dbscan_eps"] = eps_value
                group_result_df["cluster_label"] = cluster_labels
                group_result_df["is_anomaly_like"] = group_result_df["cluster_label"].eq(-1)
                group_result_df["group_status"] = "evaluated"

                feature_representation_frames.append(group_result_df)

            feature_representation_df = pd.concat(
                feature_representation_frames,
                ignore_index=True,
            )
            overlap_row_level_records.append(feature_representation_df)

            overlap_row_level_write_records.append(
                {
                    "feature_set": feature_set_name,
                    "representation_name": representation_name,
                    "rows_written": int(len(feature_representation_df)),
                    "groups_present": int(
                        feature_representation_df["comparison_group_key"].nunique(dropna=False)
                    ),
                    "anomaly_like_rows": int(
                        feature_representation_df["is_anomaly_like"].sum()
                    ),
                    "status": "pass",
                }
            )

    representation_overlap_row_level_df = pd.concat(
        overlap_row_level_records,
        ignore_index=True,
    )
    representation_overlap_row_level_df.to_parquet(
        REPRESENTATION_OVERLAP_ROW_LEVEL_PATH,
        index=False,
    )

    representation_overlap_row_level_write_df = pd.DataFrame(
        overlap_row_level_write_records
    )

else:
    representation_overlap_row_level_df = pd.read_parquet(
        REPRESENTATION_OVERLAP_ROW_LEVEL_PATH
    )

    representation_overlap_row_level_write_df = (
        representation_overlap_row_level_df.groupby(
            ["feature_set", "representation_name"],
            dropna=False,
        )
        .agg(
            rows_written=(MODEL_ID_COLUMN, "size"),
            groups_present=("comparison_group_key", "nunique"),
            anomaly_like_rows=("is_anomaly_like", "sum"),
        )
        .reset_index()
    )
    representation_overlap_row_level_write_df["status"] = "pass"

display(representation_overlap_row_level_write_df)

assert representation_overlap_row_level_write_df["status"].eq("pass").all(), (
    "One or more row-level representation pilot outputs failed to materialize."
)

print("Row-level representation pilot assignments are ready.")

,feature_set,representation_name,rows_written,groups_present,anomaly_like_rows,status
0,bus,pca_80pct,42114,30,956,pass
1,bus,raw_scaled,42114,30,1024,pass
2,bus,umap_pca_to_umap_10d,42114,30,901,pass
3,fhvhv,pca_80pct,42340,30,1081,pass
4,fhvhv,raw_scaled,42340,30,1265,pass
5,fhvhv,umap_pca_to_umap_10d,42340,30,754,pass
6,multimodal,pca_80pct,42340,30,1129,pass
7,multimodal,raw_scaled,42340,30,1200,pass
8,multimodal,umap_pca_to_umap_10d,42340,30,1149,pass
9,subway,pca_80pct,44474,30,1073,pass


Row-level representation pilot assignments are ready.


Findings\. The row\-level pilot assignments are now in place for all five modalities and all three representation spaces, giving us a shared per\-row foundation for overlap analysis\. The evaluated row counts are consistent within each modality across Raw, PCA, and UMAP, while the anomaly\-like row totals already suggest representation differences worth inspecting more closely: Raw and PCA stay fairly close in most systems, whereas UMAP flags noticeably fewer rows in Subway, Taxi, and especially FHVHV\.

Once that row\-level artifact exists, the next step is a compact QA check\. We want to confirm that every representation produced the expected rows and groups under the shared downstream baseline before we start comparing agreement or uniqueness\.

In [48]:
# ---------------------------------------------------------------------
# Validate row-level representation pilot assignment coverage
# ---------------------------------------------------------------------

if "representation_overlap_row_level_df" not in globals():
    representation_overlap_row_level_df = pd.read_parquet(
        REPRESENTATION_OVERLAP_ROW_LEVEL_PATH
    )

required_overlap_columns = [
    MODEL_ID_COLUMN,
    "feature_set",
    "representation_name",
    "comparison_universe",
    "comparison_group_key",
    "group_rows",
    "rows_evaluated",
    "dbscan_eps",
    "cluster_label",
    "is_anomaly_like",
    "group_status",
]

missing_overlap_columns = [
    column_name
    for column_name in required_overlap_columns
    if column_name not in representation_overlap_row_level_df.columns
]

assert not missing_overlap_columns, (
    f"representation_overlap_row_level_df is missing required columns: "
    f"{missing_overlap_columns}"
)

row_level_overlap_validation_df = (
    representation_overlap_row_level_df.groupby(
        ["feature_set", "representation_name"],
        dropna=False,
    )
    .agg(
        row_level_rows=(MODEL_ID_COLUMN, "size"),
        distinct_modeling_rows=(MODEL_ID_COLUMN, "nunique"),
        groups_present=("comparison_group_key", "nunique"),
        unexpected_universe_rows=(
            "comparison_universe",
            lambda s: int((~s.eq("same_temporal_bucket_and_policy_geography")).sum()),
        ),
        skipped_rows=(
            "group_status",
            lambda s: int((~s.eq("evaluated")).sum()),
        ),
    )
    .reset_index()
)

if "representation_bakeoff_summary_df" in globals():
    expected_overlap_df = representation_bakeoff_summary_df.loc[
        :,
        [
            "feature_set",
            "representation_name",
            "groups_evaluated",
            "rows_evaluated",
        ],
    ].rename(
        columns={
            "groups_evaluated": "expected_groups_evaluated",
            "rows_evaluated": "expected_rows_evaluated",
        }
    )

    row_level_overlap_validation_df = row_level_overlap_validation_df.merge(
        expected_overlap_df,
        on=["feature_set", "representation_name"],
        how="left",
    )
else:
    row_level_overlap_validation_df["expected_groups_evaluated"] = np.nan
    row_level_overlap_validation_df["expected_rows_evaluated"] = np.nan

row_level_overlap_validation_df["status"] = np.where(
    (
        row_level_overlap_validation_df["row_level_rows"]
        == row_level_overlap_validation_df["distinct_modeling_rows"]
    )
    & (
        row_level_overlap_validation_df["unexpected_universe_rows"] == 0
    )
    & (
        row_level_overlap_validation_df["skipped_rows"] == 0
    )
    & (
        row_level_overlap_validation_df["row_level_rows"]
        == row_level_overlap_validation_df["expected_rows_evaluated"]
    )
    & (
        row_level_overlap_validation_df["groups_present"]
        == row_level_overlap_validation_df["expected_groups_evaluated"]
    ),
    "pass",
    "review",
)

display(row_level_overlap_validation_df)


assert row_level_overlap_validation_df["status"].eq("pass").all(), (
    "One or more row-level representation pilot outputs failed coverage validation."
)

print("Row-level representation pilot coverage validated.")

,feature_set,representation_name,row_level_rows,distinct_modeling_rows,groups_present,unexpected_universe_rows,skipped_rows,expected_groups_evaluated,expected_rows_evaluated,status
0,bus,pca_80pct,42114,42114,30,0,0,30,42114,pass
1,bus,raw_scaled,42114,42114,30,0,0,30,42114,pass
2,bus,umap_pca_to_umap_10d,42114,42114,30,0,0,30,42114,pass
3,fhvhv,pca_80pct,42340,42340,30,0,0,30,42340,pass
4,fhvhv,raw_scaled,42340,42340,30,0,0,30,42340,pass
5,fhvhv,umap_pca_to_umap_10d,42340,42340,30,0,0,30,42340,pass
6,multimodal,pca_80pct,42340,42340,30,0,0,30,42340,pass
7,multimodal,raw_scaled,42340,42340,30,0,0,30,42340,pass
8,multimodal,umap_pca_to_umap_10d,42340,42340,30,0,0,30,42340,pass
9,subway,pca_80pct,44474,44474,30,0,0,30,44474,pass


Row-level representation pilot coverage validated.


Findings\. The row\-level anomaly pilot outputs passed validation for all five modalities and all three representations\. Every representation preserved one row per evaluated observation, all 30 temporal\-plus\-policy\-geography groups were present, no universe leakage or skipped rows appeared, and the realized row counts matched the expected evaluated totals exactly\. That gives us a clean foundation for comparing Raw, PCA, and UMAP on the same anomaly\-scoring universe rather than worrying about coverage or alignment artifacts\.

Shared vs Unique Anomalies\. Now that the row\-level pilot outputs are validated, we can compare the anomaly\-like flags directly across Raw, PCA, and UMAP\. This first overlap readout stays close to the underlying counts: how many rows each representation flags, how many of those rows are shared, and how many are unique to one representation alone\. That gives us a more intuitive foundation before we collapse agreement into a similarity score such as Jaccard\.

In [49]:
# ---------------------------------------------------------------------
# Summarize shared and unique anomaly-like flags across representations
# ---------------------------------------------------------------------

representation_overlap_records = []

for feature_set_name in MODEL_FEATURE_SET_NAMES:
    feature_df = pd.read_parquet(REPRESENTATION_OVERLAP_ROW_LEVEL_PATH).loc[
        lambda df: (
            df["feature_set"].eq(feature_set_name)
            & df["group_status"].eq("evaluated")
            & df["comparison_universe"].eq("same_temporal_bucket_and_policy_geography")
        )
    ].copy()
    wide_flag_df = (
        feature_df.pivot_table(
            index=MODEL_ID_COLUMN,
            columns="representation_name",
            values="is_anomaly_like",
            aggfunc="first",
        )
        .reset_index()
        .rename_axis(None, axis=1)
    )
    for representation_name in REPRESENTATION_NAMES:
        if representation_name not in wide_flag_df.columns:
            wide_flag_df[representation_name] = False
    wide_flag_df["raw_scaled"] = wide_flag_df["raw_scaled"].fillna(False).astype(bool)
    wide_flag_df["pca_80pct"] = wide_flag_df["pca_80pct"].fillna(False).astype(bool)
    wide_flag_df["umap_pca_to_umap_10d"] = (
        wide_flag_df["umap_pca_to_umap_10d"].fillna(False).astype(bool)
    )

    raw_flag_count = int(wide_flag_df["raw_scaled"].sum())
    pca_flag_count = int(wide_flag_df["pca_80pct"].sum())
    umap_flag_count = int(wide_flag_df["umap_pca_to_umap_10d"].sum())
    raw_pca_overlap = int((wide_flag_df["raw_scaled"] & wide_flag_df["pca_80pct"]).sum())
    raw_umap_overlap = int(
        (wide_flag_df["raw_scaled"] & wide_flag_df["umap_pca_to_umap_10d"]).sum()
    )
    pca_umap_overlap = int(
        (wide_flag_df["pca_80pct"] & wide_flag_df["umap_pca_to_umap_10d"]).sum()
    )
    all_three_overlap_rows = int(
        (
            wide_flag_df["raw_scaled"]
            & wide_flag_df["pca_80pct"]
            & wide_flag_df["umap_pca_to_umap_10d"]
        ).sum()
    )
    raw_only_rows = int(
        (
            wide_flag_df["raw_scaled"]
            & ~wide_flag_df["pca_80pct"]
            & ~wide_flag_df["umap_pca_to_umap_10d"]
        ).sum()
    )
    pca_only_rows = int(
        (
            wide_flag_df["pca_80pct"]
            & ~wide_flag_df["raw_scaled"]
            & ~wide_flag_df["umap_pca_to_umap_10d"]
        ).sum()
    )
    umap_only_rows = int(
        (
            wide_flag_df["umap_pca_to_umap_10d"]
            & ~wide_flag_df["raw_scaled"]
            & ~wide_flag_df["pca_80pct"]
        ).sum()
    )
    any_flag_rows = int(
        (
            wide_flag_df["raw_scaled"]
            | wide_flag_df["pca_80pct"]
            | wide_flag_df["umap_pca_to_umap_10d"]
        ).sum()
    )
    representation_overlap_records.append(
        {
            "feature_set": feature_set_name,
            "rows_compared": int(len(wide_flag_df)),
            "raw_flag_count": raw_flag_count,
            "pca_flag_count": pca_flag_count,
            "umap_flag_count": umap_flag_count,
            "raw_pca_overlap": raw_pca_overlap,
            "raw_umap_overlap": raw_umap_overlap,
            "pca_umap_overlap": pca_umap_overlap,
            "all_three_overlap_rows": all_three_overlap_rows,
            "raw_only_rows": raw_only_rows,
            "pca_only_rows": pca_only_rows,
            "umap_only_rows": umap_only_rows,
            "any_flag_rows": any_flag_rows,
            "status": "pass",
        }
    )

representation_overlap_df = pd.DataFrame(representation_overlap_records)

display(representation_overlap_df)

# ---------------------------------------------------------------------
# Visualize shared and unique anomaly-like flags across representations
# ---------------------------------------------------------------------

overlap_stack_plot_records = []

for _, row in representation_overlap_df.iterrows():
    overlap_stack_plot_records.extend(
        [
            {
                "feature_set": row["feature_set"],
                "segment": "All three",
                "rows": row["all_three_overlap_rows"],
            },
            {
                "feature_set": row["feature_set"],
                "segment": "Raw + PCA only",
                "rows": max(row["raw_pca_overlap"] - row["all_three_overlap_rows"], 0),
            },
            {
                "feature_set": row["feature_set"],
                "segment": "Raw + UMAP only",
                "rows": max(row["raw_umap_overlap"] - row["all_three_overlap_rows"], 0),
            },
            {
                "feature_set": row["feature_set"],
                "segment": "PCA + UMAP only",
                "rows": max(row["pca_umap_overlap"] - row["all_three_overlap_rows"], 0),
            },
            {
                "feature_set": row["feature_set"],
                "segment": "Raw only",
                "rows": row["raw_only_rows"],
            },
            {
                "feature_set": row["feature_set"],
                "segment": "PCA only",
                "rows": row["pca_only_rows"],
            },
            {
                "feature_set": row["feature_set"],
                "segment": "UMAP only",
                "rows": row["umap_only_rows"],
            },
        ]
    )

overlap_stack_plot_df = pd.DataFrame(overlap_stack_plot_records)

segment_order = [
    "All three",
    "Raw + PCA only",
    "Raw + UMAP only",
    "PCA + UMAP only",
    "Raw only",
    "PCA only",
    "UMAP only",
]

segment_color_map = {
    "All three": BRAND_COLORS["dark_teal"],
    "Raw + PCA only": BRAND_COLORS["seafoam"],
    "Raw + UMAP only": BRAND_COLORS["terracotta"],
    "PCA + UMAP only": BRAND_COLORS["pale_peach"],
    "Raw only": "rgba(11, 78, 74, 0.62)",
    "PCA only": "rgba(190, 91, 72, 0.62)",
    "UMAP only": "rgba(102, 170, 153, 0.62)",
}

segment_pattern_map = {
    "All three": "",
    "Raw + PCA only": "",
    "Raw + UMAP only": "",
    "PCA + UMAP only": "",
    "Raw only": "/",
    "PCA only": "\\",
    "UMAP only": "x",
}

overlap_stack_plot_df["segment"] = pd.Categorical(
    overlap_stack_plot_df["segment"],
    categories=segment_order,
    ordered=True,
)

overlap_stack_fig = px.bar(
    overlap_stack_plot_df.sort_values(["feature_set", "segment"]),
    x="feature_set",
    y="rows",
    color="segment",
    pattern_shape="segment",
    category_orders={
        "feature_set": MODEL_FEATURE_SET_NAMES,
        "segment": segment_order,
    },
    color_discrete_map=segment_color_map,
    pattern_shape_map=segment_pattern_map,
    title="Shared and Unique Anomaly-Like Flags Across Representations",
    labels={
        "feature_set": "Feature set",
        "rows": "Anomaly-like rows",
        "segment": "Flag category",
    },
)

overlap_stack_fig.update_traces(
    marker_line_color=BRAND_COLORS["dark_teal"],
    marker_line_width=0.5,
)

apply_brand_plotly_layout(
    overlap_stack_fig,
    title="Shared and Unique Anomaly-Like Flags Across Representations",
    width=980,
    height=560,
    legend_title="Flag category",
)

overlap_stack_fig.update_layout(
    barmode="stack",
    margin={"t": 70, "b": 70, "l": 70, "r": 30},
    legend={
        "title": {"text": "Flag category"},
        "orientation": "h",
        "yanchor": "top",
        "y": -0.18,
        "xanchor": "center",
        "x": 0.5,
        "bgcolor": "rgba(255,255,255,0.85)",
        "bordercolor": BRAND_COLORS["seafoam"],
        "borderwidth": 1,
        "font": {"color": BRAND_COLORS["dark_teal"]},
    },
)

overlap_stack_fig.update_xaxes(title_text="Feature set")
overlap_stack_fig.update_yaxes(title_text="Anomaly-like rows")

overlap_stack_fig.show()

assert representation_overlap_df["status"].eq("pass").all(), (
    "Shared and unique anomaly-flag overlap summary failed."
)

print("Shared and unique anomaly-like flags summarized.")

,feature_set,rows_compared,raw_flag_count,pca_flag_count,umap_flag_count,raw_pca_overlap,raw_umap_overlap,pca_umap_overlap,all_three_overlap_rows,raw_only_rows,pca_only_rows,umap_only_rows,any_flag_rows,status
0,bus,42114,1024,956,901,861,277,286,251,137,60,589,1708,pass
1,subway,44474,1061,1073,719,882,262,273,249,166,167,433,1685,pass
2,taxi,42340,1011,993,880,855,214,230,201,143,109,637,1786,pass
3,fhvhv,42340,1265,1081,754,982,183,190,165,265,74,546,1910,pass
4,multimodal,42340,1200,1129,1149,1027,198,203,179,154,78,927,2229,pass


Shared and unique anomaly-like flags summarized.


Findings\. Raw and PCA still look like the closest pair, but the picture is a little more nuanced than “UMAP is completely separate\.” In every modality, the largest shared segment is Raw \+ PCA only, which confirms that the two linear representations are surfacing a substantial common anomaly\-like core\. At the same time, UMAP contributes a large unique segment everywhere, especially in Multimodal, Taxi, Bus, and FHVHV, so it is clearly not just reproducing the Raw/PCA view\. The smaller Raw \+ UMAP only and PCA \+ UMAP only bands show that UMAP shares some signal with each linear representation, but much less than Raw and PCA share with each other\. So the clean takeaway is that Raw and PCA are partly redundant, while UMAP appears to add a genuinely different anomaly surface on top of that shared linear core\.

Jaccard Similarity\. The overlap counts already tell us the broad story, but they are still scale\-dependent: a modality with more flagged rows will naturally have bigger intersections\. Jaccard similarity gives us a cleaner normalization by comparing the overlap to the full union of flagged rows\. That makes it easier to judge whether Raw and PCA are genuinely close, and whether UMAP is adding a distinct anomaly\-like signal or mostly rediscovering the same rows\.

In [50]:
# ---------------------------------------------------------------------
# Compare pairwise Jaccard similarity across representations
# ---------------------------------------------------------------------

jaccard_records = []

for _, row in representation_overlap_df.iterrows():
    raw_flag_count = int(row["raw_flag_count"])
    pca_flag_count = int(row["pca_flag_count"])
    umap_flag_count = int(row["umap_flag_count"])
    raw_pca_overlap = int(row["raw_pca_overlap"])
    raw_umap_overlap = int(row["raw_umap_overlap"])
    pca_umap_overlap = int(row["pca_umap_overlap"])

    def safe_jaccard(intersection_count: int, left_count: int, right_count: int) -> float:
        union_count = left_count + right_count - intersection_count
        if union_count == 0:
            return np.nan
        return intersection_count / union_count

    jaccard_records.extend(
        [
            {
                "feature_set": row["feature_set"],
                "representation_pair": "raw_vs_pca",
                "left_representation": "Raw",
                "right_representation": "PCA",
                "intersection_rows": raw_pca_overlap,
                "union_rows": raw_flag_count + pca_flag_count - raw_pca_overlap,
                "jaccard_similarity": safe_jaccard(
                    raw_pca_overlap,
                    raw_flag_count,
                    pca_flag_count,
                ),
                "status": "pass",
            },
            {
                "feature_set": row["feature_set"],
                "representation_pair": "raw_vs_umap",
                "left_representation": "Raw",
                "right_representation": "UMAP",
                "intersection_rows": raw_umap_overlap,
                "union_rows": raw_flag_count + umap_flag_count - raw_umap_overlap,
                "jaccard_similarity": safe_jaccard(
                    raw_umap_overlap,
                    raw_flag_count,
                    umap_flag_count,
                ),
                "status": "pass",
            },
            {
                "feature_set": row["feature_set"],
                "representation_pair": "pca_vs_umap",
                "left_representation": "PCA",
                "right_representation": "UMAP",
                "intersection_rows": pca_umap_overlap,
                "union_rows": pca_flag_count + umap_flag_count - pca_umap_overlap,
                "jaccard_similarity": safe_jaccard(
                    pca_umap_overlap,
                    pca_flag_count,
                    umap_flag_count,
                ),
                "status": "pass",
            },
        ]
    )

representation_jaccard_df = pd.DataFrame(jaccard_records)
representation_jaccard_df["jaccard_similarity"] = (
    representation_jaccard_df["jaccard_similarity"].round(4)
)

display(representation_jaccard_df)

# Prepare plotting labels in a separate dataframe so the chart code stays clean.
pair_order = ["raw_vs_pca", "raw_vs_umap", "pca_vs_umap"]
pair_label_map = {
    "raw_vs_pca": "Raw vs PCA",
    "raw_vs_umap": "Raw vs UMAP",
    "pca_vs_umap": "PCA vs UMAP",
}
feature_set_color_map = {
    feature_set_name: BRAND_COLOR_SEQUENCE[index % len(BRAND_COLOR_SEQUENCE)]
    for index, feature_set_name in enumerate(MODEL_FEATURE_SET_NAMES)
}

jaccard_plot_df = representation_jaccard_df.copy()
jaccard_plot_df["pair_label"] = (
    jaccard_plot_df["representation_pair"].map(pair_label_map)
)

# ---------------------------------------------------------------------
# Visualize via Bar Chart
# ---------------------------------------------------------------------

jaccard_fig = px.bar(
    jaccard_plot_df,
    x="pair_label",
    y="jaccard_similarity",
    color="feature_set",
    barmode="group",
    category_orders={
        "pair_label": [pair_label_map[pair] for pair in pair_order],
        "feature_set": MODEL_FEATURE_SET_NAMES,
    },
    color_discrete_map=feature_set_color_map,
    labels={
        "pair_label": "Representation pair",
        "jaccard_similarity": "Jaccard similarity",
        "feature_set": "Feature set",
    },
    title="Pairwise Anomaly-Flag Jaccard Similarity Across Representations",
)

jaccard_fig.update_traces(
    marker_line_color=BRAND_COLORS["dark_teal"],
    marker_line_width=0.5,
)

apply_brand_plotly_layout(
    jaccard_fig,
    title="Pairwise Anomaly-Flag Jaccard Similarity Across Representations",
    width=920,
    height=430,
    legend_title="Feature set",
)

jaccard_fig.update_layout(
    legend={
        "title": {"text": "Feature set"},
        "orientation": "h",
        "yanchor": "top",
        "y": -0.20,
        "xanchor": "center",
        "x": 0.5,
        "bgcolor": "rgba(255,255,255,0.85)",
        "bordercolor": BRAND_COLORS["seafoam"],
        "borderwidth": 1,
        "font": {"color": BRAND_COLORS["dark_teal"]},
    },
    margin={"t": 70, "b": 95, "l": 70, "r": 30},
)

jaccard_fig.show()

assert representation_jaccard_df["status"].eq("pass").all(), (
    "Pairwise Jaccard similarity comparison failed."
)

print("Pairwise Jaccard similarity across representations summarized.")

,feature_set,representation_pair,left_representation,right_representation,intersection_rows,union_rows,jaccard_similarity,status
0,bus,raw_vs_pca,Raw,PCA,861,1119,0.7694,pass
1,bus,raw_vs_umap,Raw,UMAP,277,1648,0.1681,pass
2,bus,pca_vs_umap,PCA,UMAP,286,1571,0.1820,pass
3,subway,raw_vs_pca,Raw,PCA,882,1252,0.7045,pass
4,subway,raw_vs_umap,Raw,UMAP,262,1518,0.1726,pass
5,subway,pca_vs_umap,PCA,UMAP,273,1519,0.1797,pass
6,taxi,raw_vs_pca,Raw,PCA,855,1149,0.7441,pass
7,taxi,raw_vs_umap,Raw,UMAP,214,1677,0.1276,pass
8,taxi,pca_vs_umap,PCA,UMAP,230,1643,0.1400,pass
9,fhvhv,raw_vs_pca,Raw,PCA,982,1364,0.7199,pass


Pairwise Jaccard similarity across representations summarized.


Findings\. The Jaccard view makes the overlap story much sharper: Raw and PCA are highly similar, while UMAP is not\. Raw vs PCA stays strong across all five modalities, from about 0\.70 in Subway to 0\.79 in Multimodal, which means those two representations are flagging largely the same rows\. By contrast, both UMAP pairings are low everywhere: Raw vs UMAP ranges from about 0\.09 to 0\.17, and PCA vs UMAP from about 0\.10 to 0\.18\. Bus and Subway are the least divergent from UMAP, but even there the overlap remains modest\. So if we are asking which representations are redundant, the evidence points squarely at Raw and PCA; UMAP is contributing a substantially different anomaly\-like set rather than a lightly perturbed version of the linear results\.

UMAP Coherence Check\. UMAP has already shown that it does not simply reproduce the Raw/PCA anomaly surface\. The next question is whether that difference is useful\. Here we compare the UMAP\-only anomaly\-like rows against the shared Raw \+ PCA anomaly core and check how concentrated each one is across temporal\-policy groups and zones\. If UMAP\-only rows stay organized into recurring regimes rather than dissolving into a diffuse tail, that gives the nonlinear representation a stronger claim on the downstream workflow\.

This block treats the shared Raw \+ PCA anomaly core as the linear reference point, then asks whether UMAP\-only anomaly\-like rows behave like a coherent supplementary layer\. The main quantities to watch are concentration and recurrence: how much of each anomaly surface sits in its top group, how much accumulates in its top five groups, and how much falls into its top ten zones\.

In [51]:
# ---------------------------------------------------------------------
# Check whether UMAP-only anomaly-like rows form a coherent layer
# ---------------------------------------------------------------------

from project_branding import BRAND_COLORS, BRAND_COLOR_SEQUENCE

px.defaults.color_discrete_sequence = BRAND_COLOR_SEQUENCE
px.defaults.template = "plotly_white"


def apply_brand_plotly_layout(
    fig,
    *,
    title=None,
    width=None,
    height=None,
    legend_title=None,
):
    fig.update_layout(
        template="plotly_white",
        paper_bgcolor="white",
        plot_bgcolor=BRAND_COLORS["ice"],
        font={"color": BRAND_COLORS["dark_teal"]},
        title={
            "text": title,
            "font": {"color": BRAND_COLORS["dark_teal"]},
        } if title else None,
        width=width,
        height=height,
        legend={
            "title": {"text": legend_title} if legend_title else None,
            "bgcolor": "rgba(255,255,255,0.85)",
            "bordercolor": BRAND_COLORS["seafoam"],
            "borderwidth": 1,
            "font": {"color": BRAND_COLORS["dark_teal"]},
        },
    )

    fig.update_xaxes(
        gridcolor="rgba(11, 78, 74, 0.14)",
        zerolinecolor="rgba(11, 78, 74, 0.20)",
        tickfont={"color": BRAND_COLORS["dark_teal"]},
        title_font={"color": BRAND_COLORS["dark_teal"]},
    )
    fig.update_yaxes(
        gridcolor="rgba(11, 78, 74, 0.14)",
        zerolinecolor="rgba(11, 78, 74, 0.20)",
        tickfont={"color": BRAND_COLORS["dark_teal"]},
        title_font={"color": BRAND_COLORS["dark_teal"]},
    )

    return fig


if "representation_overlap_row_level_df" not in globals():
    representation_overlap_row_level_df = pd.read_parquet(
        REPRESENTATION_OVERLAP_ROW_LEVEL_PATH
    )

required_overlap_columns = [
    "feature_set",
    "representation_name",
    MODEL_ID_COLUMN,
    "comparison_group_key",
]

missing_overlap_columns = [
    column_name
    for column_name in required_overlap_columns
    if column_name not in representation_overlap_row_level_df.columns
]

assert not missing_overlap_columns, (
    "representation_overlap_row_level_df is missing required columns for the "
    f"UMAP coherence check: {missing_overlap_columns}"
)

if "anomaly_like_flag" in representation_overlap_row_level_df.columns:
    overlap_row_level_flags_df = representation_overlap_row_level_df.copy()
    overlap_row_level_flags_df["anomaly_like_flag"] = (
        overlap_row_level_flags_df["anomaly_like_flag"].astype(bool)
    )
elif "dbscan_label" in representation_overlap_row_level_df.columns:
    overlap_row_level_flags_df = representation_overlap_row_level_df.copy()
    overlap_row_level_flags_df["anomaly_like_flag"] = (
        overlap_row_level_flags_df["dbscan_label"].eq(-1)
    )
elif "cluster_label" in representation_overlap_row_level_df.columns:
    overlap_row_level_flags_df = representation_overlap_row_level_df.copy()
    overlap_row_level_flags_df["anomaly_like_flag"] = (
        overlap_row_level_flags_df["cluster_label"].eq(-1)
    )
else:
    raise AssertionError(
        "representation_overlap_row_level_df must contain anomaly_like_flag, "
        "dbscan_label, or cluster_label."
    )

umap_coherence_records = []

for feature_set_name in MODEL_FEATURE_SET_NAMES:
    feature_row_df = overlap_row_level_flags_df.loc[
        overlap_row_level_flags_df["feature_set"].eq(feature_set_name)
    ].copy()

    metadata_df = pd.read_parquet(
        ANOMALY_CALIBRATION_METADATA_PATHS_BY_SET[feature_set_name],
        columns=[MODEL_ID_COLUMN, ZONE_ID_COLUMN, DATE_COLUMN],
    ).drop_duplicates(subset=[MODEL_ID_COLUMN])

    feature_row_df = feature_row_df.merge(
        metadata_df,
        on=MODEL_ID_COLUMN,
        how="left",
        validate="many_to_one",
    )

    wide_flag_df = (
        feature_row_df.pivot_table(
            index=[
                MODEL_ID_COLUMN,
                "comparison_group_key",
                ZONE_ID_COLUMN,
                DATE_COLUMN,
            ],
            columns="representation_name",
            values="anomaly_like_flag",
            aggfunc="max",
            fill_value=False,
        )
        .reset_index()
    )

    for representation_name in REPRESENTATION_NAMES:
        if representation_name not in wide_flag_df.columns:
            wide_flag_df[representation_name] = False

    wide_flag_df["raw_scaled"] = wide_flag_df["raw_scaled"].astype(bool)
    wide_flag_df["pca_80pct"] = wide_flag_df["pca_80pct"].astype(bool)
    wide_flag_df["umap_pca_to_umap_10d"] = (
        wide_flag_df["umap_pca_to_umap_10d"].astype(bool)
    )

    category_masks = {
        "Shared Raw + PCA core": (
            wide_flag_df["raw_scaled"] & wide_flag_df["pca_80pct"]
        ),
        "UMAP only": (
            wide_flag_df["umap_pca_to_umap_10d"]
            & ~wide_flag_df["raw_scaled"]
            & ~wide_flag_df["pca_80pct"]
        ),
    }

    for anomaly_surface_name, anomaly_surface_mask in category_masks.items():
        anomaly_surface_df = wide_flag_df.loc[anomaly_surface_mask].copy()
        flagged_rows = len(anomaly_surface_df)

        if flagged_rows == 0:
            umap_coherence_records.append(
                {
                    "feature_set": feature_set_name,
                    "anomaly_surface": anomaly_surface_name,
                    "flagged_rows": 0,
                    "active_groups": 0,
                    "top_group_share": np.nan,
                    "top5_group_share": np.nan,
                    "active_zones": 0,
                    "top10_zone_share": np.nan,
                    "top10_zone_distinct_dates": 0,
                    "status": "review",
                }
            )
            continue

        group_counts = (
            anomaly_surface_df.groupby("comparison_group_key")
            .size()
            .sort_values(ascending=False)
        )

        zone_counts = (
            anomaly_surface_df.groupby(ZONE_ID_COLUMN)
            .size()
            .sort_values(ascending=False)
        )

        top10_zone_ids = zone_counts.head(10).index.tolist()

        umap_coherence_records.append(
            {
                "feature_set": feature_set_name,
                "anomaly_surface": anomaly_surface_name,
                "flagged_rows": int(flagged_rows),
                "active_groups": int(group_counts.shape[0]),
                "top_group_share": float(group_counts.iloc[0] / flagged_rows),
                "top5_group_share": float(group_counts.head(5).sum() / flagged_rows),
                "active_zones": int(zone_counts.shape[0]),
                "top10_zone_share": float(zone_counts.head(10).sum() / flagged_rows),
                "top10_zone_distinct_dates": int(
                    anomaly_surface_df.loc[
                        anomaly_surface_df[ZONE_ID_COLUMN].isin(top10_zone_ids),
                        DATE_COLUMN,
                    ].nunique()
                ),
                "status": "pass",
            }
        )

umap_coherence_df = pd.DataFrame(umap_coherence_records)

display(umap_coherence_df)

assert umap_coherence_df["status"].eq("pass").all(), (
    "UMAP coherence check did not produce a complete comparison."
)

coherence_plot_df = umap_coherence_df.melt(
    id_vars=["feature_set", "anomaly_surface"],
    value_vars=["top_group_share", "top5_group_share", "top10_zone_share"],
    var_name="metric_name",
    value_name="metric_value",
)

metric_label_map = {
    "top_group_share": "Top group share",
    "top5_group_share": "Top-5 group share",
    "top10_zone_share": "Top-10 zone share",
}

surface_color_map = {
    "Shared Raw + PCA core": BRAND_COLORS["dark_teal"],
    "UMAP only": BRAND_COLORS["terracotta"],
}

coherence_plot_df["metric_label"] = coherence_plot_df["metric_name"].map(
    metric_label_map
)

coherence_fig = px.scatter(
    coherence_plot_df,
    x="metric_value",
    y="feature_set",
    color="anomaly_surface",
    symbol="anomaly_surface",
    facet_col="metric_label",
    category_orders={
        "feature_set": MODEL_FEATURE_SET_NAMES[::-1],
        "metric_label": [
            "Top group share",
            "Top-5 group share",
            "Top-10 zone share",
        ],
    },
    color_discrete_map=surface_color_map,
    labels={
        "metric_value": "Share of anomaly-like rows",
        "feature_set": "Feature set",
        "anomaly_surface": "Anomaly surface",
        "metric_label": "",
    },
)

for metric_label in coherence_plot_df["metric_label"].dropna().unique():
    metric_col_index = [
        "Top group share",
        "Top-5 group share",
        "Top-10 zone share",
    ].index(metric_label) + 1

    panel_df = coherence_plot_df.loc[
        coherence_plot_df["metric_label"].eq(metric_label)
    ]

    for feature_set_name in MODEL_FEATURE_SET_NAMES:
        feature_metric_df = (
            panel_df.loc[panel_df["feature_set"].eq(feature_set_name)]
            .set_index("anomaly_surface")
        )

        if {"Shared Raw + PCA core", "UMAP only"}.issubset(feature_metric_df.index):
            coherence_fig.add_trace(
                go.Scatter(
                    x=[
                        feature_metric_df.loc[
                            "Shared Raw + PCA core", "metric_value"
                        ],
                        feature_metric_df.loc["UMAP only", "metric_value"],
                    ],
                    y=[feature_set_name, feature_set_name],
                    mode="lines",
                    line={
                        "color": "rgba(11, 78, 74, 0.35)",
                        "width": 2,
                    },
                    showlegend=False,
                    hoverinfo="skip",
                ),
                row=1,
                col=metric_col_index,
            )

coherence_fig.update_traces(marker={"size": 9})

coherence_fig.for_each_annotation(
    lambda annotation: annotation.update(text=annotation.text.split("=")[-1])
)

coherence_fig.update_xaxes(tickformat=".0%")
coherence_fig.update_yaxes(title_text="")

apply_brand_plotly_layout(
    coherence_fig,
    title="UMAP-Only Coherence Relative to the Shared Raw + PCA Core",
    width=980,
    height=430,
    legend_title="Anomaly surface",
)

coherence_fig.update_layout(
    margin={"t": 70, "b": 80, "l": 70, "r": 20},
    legend={
        "orientation": "h",
        "yanchor": "top",
        "y": -0.18,
        "xanchor": "center",
        "x": 0.5,
        "title": {"text": "Anomaly surface"},
    },
)

coherence_fig.show()

print("UMAP-only coherence relative to the shared Raw + PCA core summarized.")

,feature_set,anomaly_surface,flagged_rows,active_groups,top_group_share,top5_group_share,active_zones,top10_zone_share,top10_zone_distinct_dates,status
0,bus,Shared Raw + PCA core,861,28,0.113821,0.513357,99,0.626016,364,pass
1,bus,UMAP only,589,28,0.100170,0.448217,127,0.478778,217,pass
2,subway,Shared Raw + PCA core,882,30,0.096372,0.451247,136,0.379819,262,pass
3,subway,UMAP only,433,28,0.103926,0.445727,120,0.371824,132,pass
4,taxi,Shared Raw + PCA core,855,30,0.111111,0.480702,180,0.471345,158,pass
5,taxi,UMAP only,637,30,0.128728,0.491366,125,0.403454,169,pass
6,fhvhv,Shared Raw + PCA core,982,30,0.125255,0.500000,123,0.653768,260,pass
7,fhvhv,UMAP only,546,28,0.186813,0.467033,164,0.335165,123,pass
8,multimodal,Shared Raw + PCA core,1027,30,0.118793,0.513145,121,0.641675,243,pass
9,multimodal,UMAP only,927,29,0.094930,0.421791,170,0.423948,149,pass


UMAP-only coherence relative to the shared Raw + PCA core summarized.


Findings\. The UMAP\-only anomaly layer does look coherent enough to take seriously, and in most modalities it is less concentrated than the anomaly core shared by Raw and PCA\. Bus, FHVHV, and Multimodal show the clearest pattern: UMAP\-only rows are spread across more zones and a smaller top\-10\-zone share than the shared linear core, which suggests that UMAP is not just rediscovering the same dominant hotspot locations\. Subway is more neutral, with very similar concentration between the two surfaces, while Taxi is mixed: UMAP\-only anomalies are somewhat more concentrated by group but less concentrated by top zones\. Overall, the picture supports the idea that UMAP is contributing a distinct nonlinear anomaly surface, but not a random or incoherent one\.

> 🎯 Decision\. We should advance PCA and UMAP as the shared downstream anomaly representations\. PCA remains the primary linear representation because it preserves most of the anomaly\-like structure that Raw surfaced while removing unnecessary dimensional bulk\. UMAP also advances because it contributes a coherent nonlinear anomaly surface that is meaningfully different from the shared Raw/PCA core, rather than simply rediscovering the same rows\. Raw Scaled will not move forward as a separate downstream anomaly representation\. We are also not defining the final anomaly set as a PCA–UMAP union here; that combination rule should be evaluated later, after the downstream anomaly notebooks show how the two representations behave under the full method\-specific generation workflow\.

### Select and record the shared downstream representation

At this point, the representation comparison has done its job\. Raw and PCA overlap heavily, while UMAP contributes a genuinely different anomaly\-like surface that still appears coherent rather than random\. Bus, FHVHV, and Multimodal show the clearest evidence that UMAP\-only anomaly\-like rows are less tied to the same dominant hotspot pattern as the shared Raw/PCA core; Subway is more neutral, and Taxi is mixed but still distinct\. That leaves us with a cleaner downstream handoff: keep PCA as the primary linear representation, keep UMAP as the nonlinear complement, and retire Raw as a separate anomaly\-generation path\.

This block turns the comparison result into a compact downstream handoff\. We are not deciding how future notebooks should combine PCA and UMAP anomaly outputs; we are only recording which representation spaces should advance into that next stage\.

In [52]:
# ---------------------------------------------------------------------
# Record the shared downstream anomaly representations
# ---------------------------------------------------------------------

shared_representation_selection_records = [
    {
        "representation_name": "pca_80pct",
        "representation_family": "pca",
        "downstream_status": "advance",
        "downstream_role": "primary_linear_representation",
        "selection_rationale": (
            "Preserves the core anomaly-like structure surfaced by Raw while "
            "removing unnecessary dimensional bulk."
        ),
    },
    {
        "representation_name": "umap_pca_to_umap_10d",
        "representation_family": "umap",
        "downstream_status": "advance",
        "downstream_role": "nonlinear_complement_representation",
        "selection_rationale": (
            "Adds a coherent anomaly-like surface that is meaningfully distinct "
            "from the shared Raw/PCA core."
        ),
    },
    {
        "representation_name": "raw_scaled",
        "representation_family": "raw",
        "downstream_status": "retire",
        "downstream_role": "comparison_baseline_only",
        "selection_rationale": (
            "Largely overlaps with PCA and does not justify a separate downstream "
            "anomaly-generation path."
        ),
    },
]

shared_representation_selection_df = pd.DataFrame(
    shared_representation_selection_records
)

display(shared_representation_selection_df)

print("Shared downstream anomaly representation set recorded.")

,representation_name,representation_family,downstream_status,downstream_role,selection_rationale
0,pca_80pct,pca,advance,primary_linear_representation,Preserves the core anomaly-like structure surf...
1,umap_pca_to_umap_10d,umap,advance,nonlinear_complement_representation,Adds a coherent anomaly-like surface that is m...
2,raw_scaled,raw,retire,comparison_baseline_only,Largely overlaps with PCA and does not justify...


Shared downstream anomaly representation set recorded.


Findings\. The downstream anomaly workflow will advance with two representation spaces, not three\. PCA is retained as the primary linear representation because it preserves the main anomaly\-like signal that Raw was surfacing without carrying the full raw feature\-space burden, while UMAP is retained as the nonlinear complement because it contributes a distinct and coherent anomaly surface\. Raw Scaled is therefore retired as a separate downstream anomaly\-generation path and kept only as an upstream comparison reference in the narrative\.

### Section Summary

> This section selected the shared downstream anomaly representations by comparing how Raw, PCA, and UMAP behave under the same temporal\-bucket\-plus\-policy\-geography baseline\. Raw and PCA produced very similar anomaly\-like volumes and heavily overlapping flagged rows, which suggests that PCA preserves most of the useful linear anomaly signal without needing to carry the full scaled feature space downstream\. UMAP behaved differently: it often generated a more strongly structured anomaly surface, shared much less overlap with the linear representations, and contributed a substantial set of unique anomaly\-like rows\. The coherence check showed that those UMAP\-only rows were generally organized enough to take seriously rather than dismiss as random fragmentation, especially in Bus, FHVHV, and Multimodal\. Taken together, the evidence supports a two\-representation downstream handoff: PCA as the primary linear anomaly space and UMAP as the nonlinear complement, while Raw is retired as a separate anomaly\-generation branch\.

\* Bus: PCA stayed close to Raw in both anomaly volume and overlap, while UMAP added a distinct but still coherent anomaly surface, making Bus a clear case for advancing both PCA and UMAP\.

\* Subway: PCA again tracked Raw closely, whereas UMAP produced a much more aggressively separated anomaly structure; Subway 
supports carrying UMAP forward, but with more interpretive caution because its nonlinear behavior is especially strong\.

\* Taxi: Raw and PCA were highly redundant, while UMAP contributed a different anomaly surface with mixed coherence signals; Taxi still supports the two\-representation handoff, though downstream notebooks should watch carefully how nonlinear structure behaves\.

\* FHVHV: PCA captured the linear anomaly core efficiently, and UMAP added a coherent supplementary surface that was less tied to the same dominant hotspot pattern, making the PCA\-plus\-UMAP decision look well supported\.

\* Multimodal: Multimodal showed one of the strongest arguments for keeping UMAP: Raw and PCA remained close, while UMAP surfaced a large unique anomaly layer that still looked structured enough to matter downstream\.

## 3\.3\.1\.5 Compare Taxi anomaly\-like behavior across pre/post periods under PCAsolve Taxi PCA Pre/Post\-CP Handling

Evaluate whether Taxi’s previously observed pre/post congestion\-pricing PCA drift is large enough to require special downstream treatment\. The main question is whether the shared Taxi PCA representation remains usable under the selected anomaly baseline, or whether pre/post instability materially changes anomaly\-like behavior in a way that would justify a Taxi\-specific caveat or branch\.

### Validate Taxi PCA comparison assets

Before we test whether Taxi needs special PCA handling downstream, let’s make sure the comparison assets are actually in place\. We need one shared Taxi PCA representation from 3\.2\.1, two Taxi period\-specific PCA representations from 3\.2\.2, and the Taxi calibration sample from this notebook\. This section confirms that those assets exist, that the calibration rows are covered by the appropriate PCA score tables, and that the PCA dimensions look the way we expect before we run any Taxi\-specific anomaly pilot\.

This is a compact readiness check\. The goal is not to analyze Taxi behavior yet; it is just to verify that the shared and split PCA comparison paths are complete and compatible with the Taxi calibration sample\.

In [53]:
# ---------------------------------------------------------------------
# Validate Taxi PCA comparison assets
# ---------------------------------------------------------------------

taxi_pca_comparison_asset_records = []

taxi_calibration_metadata_df = pd.read_parquet(
    ANOMALY_CALIBRATION_METADATA_PATHS_BY_SET["taxi"],
    columns=[MODEL_ID_COLUMN, PRE_POST_CP_COLUMN],
).drop_duplicates(subset=[MODEL_ID_COLUMN])

taxi_sample_rows = len(taxi_calibration_metadata_df)
taxi_pre_sample_ids = set(
    taxi_calibration_metadata_df.loc[
        taxi_calibration_metadata_df[PRE_POST_CP_COLUMN].eq("pre_cp"),
        MODEL_ID_COLUMN,
    ].tolist()
)
taxi_post_sample_ids = set(
    taxi_calibration_metadata_df.loc[
        taxi_calibration_metadata_df[PRE_POST_CP_COLUMN].eq("post_cp"),
        MODEL_ID_COLUMN,
    ].tolist()
)

asset_specs = [
    {
        "asset_name": "taxi_pca_shared",
        "policy_scope": "full_period",
        "path": TAXI_SHARED_PCA_REFERENCE_PATH,
        "expected_sample_ids": set(taxi_calibration_metadata_df[MODEL_ID_COLUMN].tolist()),
        "expected_components": PCA_80_COMPONENTS_BY_SET["taxi"],
    },
    {
        "asset_name": "taxi_pca_pre_cp",
        "policy_scope": "pre_cp_only",
        "path": TAXI_PERIOD_PCA_80_FINAL_PATHS_BY_PERIOD["pre_cp"],
        "expected_sample_ids": taxi_pre_sample_ids,
        "expected_components": 15,
    },
    {
        "asset_name": "taxi_pca_post_cp",
        "policy_scope": "post_cp_only",
        "path": TAXI_PERIOD_PCA_80_FINAL_PATHS_BY_PERIOD["post_cp"],
        "expected_sample_ids": taxi_post_sample_ids,
        "expected_components": 15,
    },
]

for asset_spec in asset_specs:
    asset_path = asset_spec["path"]
    path_exists = asset_path.exists()

    if path_exists:
        asset_df = pd.read_parquet(asset_path)
        asset_ids = set(asset_df[MODEL_ID_COLUMN].tolist())
        component_columns = [
            column_name
            for column_name in asset_df.columns
            if column_name != MODEL_ID_COLUMN
        ]
        duplicate_modeling_rows = int(asset_df[MODEL_ID_COLUMN].duplicated().sum())
        sample_rows_missing = len(asset_spec["expected_sample_ids"] - asset_ids)
        sample_rows_covered = len(asset_spec["expected_sample_ids"]) - sample_rows_missing
        source_rows = len(asset_df)
    else:
        component_columns = []
        duplicate_modeling_rows = np.nan
        sample_rows_missing = len(asset_spec["expected_sample_ids"])
        sample_rows_covered = 0
        source_rows = np.nan

    taxi_pca_comparison_asset_records.append(
        {
            "asset_name": asset_spec["asset_name"],
            "policy_scope": asset_spec["policy_scope"],
            "path_exists": path_exists,
            "source_rows": source_rows,
            "sample_rows_expected": len(asset_spec["expected_sample_ids"]),
            "sample_rows_covered": sample_rows_covered,
            "sample_rows_missing": sample_rows_missing,
            "pca_component_columns": len(component_columns),
            "expected_components": asset_spec["expected_components"],
            "duplicate_modeling_rows": duplicate_modeling_rows,
            "status": "pass"
            if (
                path_exists
                and sample_rows_missing == 0
                and len(component_columns) == asset_spec["expected_components"]
                and duplicate_modeling_rows == 0
            )
            else "review",
        }
    )

taxi_pca_comparison_asset_status_df = pd.DataFrame(
    taxi_pca_comparison_asset_records
)

display(taxi_pca_comparison_asset_status_df)

assert taxi_pca_comparison_asset_status_df["status"].eq("pass").all(), (
    "One or more Taxi PCA comparison assets is missing, incomplete, or misaligned."
)

print("Taxi PCA comparison assets validated.")

,asset_name,policy_scope,path_exists,source_rows,sample_rows_expected,sample_rows_covered,sample_rows_missing,pca_component_columns,expected_components,duplicate_modeling_rows,status
0,taxi_pca_shared,full_period,True,1541800,50000,50000,0,15,15,0,pass
1,taxi_pca_pre_cp,pre_cp_only,True,955500,30985,30985,0,15,15,0,pass
2,taxi_pca_post_cp,post_cp_only,True,586300,19015,19015,0,15,15,0,pass


Taxi PCA comparison assets validated.


Findings\. The Taxi PCA comparison assets are fully in place and aligned to the Taxi calibration sample\. The shared full\-period PCA table covers all 50,000 Taxi sample rows, while the split\-period exports cleanly cover the expected pre\_cp and post\_cp subsets with no missing sample rows, no duplicate modeling\_row\_id values, and the expected 15 PCA components in every case\.

### Materialize shared\-vs\-split Taxi PCA pilot outputs

Now let’s turn the Taxi PCA question into something measurable\. We’ll run the same lightweight anomaly pilot under two Taxi PCA handling choices: one shared full\-period PCA space, and one split period\-specific PCA setup that scores pre\-CP rows in the pre\-CP PCA space and post\-CP rows in the post\-CP PCA space\. This gives us a concrete downstream comparison instead of treating the Taxi drift question as a theoretical concern\.

This is the one expensive block in the Taxi section\. It writes a reusable row\-level artifact plus a compact summary table, so later Taxi comparison blocks can stay focused on interpretation rather than recomputation\.

In [54]:
# ---------------------------------------------------------------------
# Materialize shared-vs-split Taxi PCA pilot outputs
# ---------------------------------------------------------------------

TAXI_PCA_HANDLING_ROW_LEVEL_PATH = (
    INTERMEDIATE_331_DIR / "taxi_pca_shared_vs_split_row_level.parquet"
)
TAXI_PCA_HANDLING_SUMMARY_PATH = (
    INTERMEDIATE_331_DIR / "taxi_pca_shared_vs_split_summary.parquet"
)

should_rebuild_taxi_pca_handling = (
    REBUILD_TAXI_PCA_PERIOD_DIAGNOSTICS
    or (not TAXI_PCA_HANDLING_ROW_LEVEL_PATH.exists())
    or (not TAXI_PCA_HANDLING_SUMMARY_PATH.exists())
)

if should_rebuild_taxi_pca_handling:
    taxi_metadata_df = pd.read_parquet(
        ANOMALY_CALIBRATION_METADATA_PATHS_BY_SET["taxi"],
        columns=[
            MODEL_ID_COLUMN,
            DATE_COLUMN,
            TEMPORAL_BUCKET_COLUMN,
            PRE_POST_CP_COLUMN,
            POLICY_GEOGRAPHY_COLUMN,
            BOROUGH_COLUMN,
            ZONE_ID_COLUMN,
            ZONE_COLUMN,
        ],
    ).drop_duplicates(subset=[MODEL_ID_COLUMN])

    taxi_shared_pca_df = pd.read_parquet(TAXI_SHARED_PCA_REFERENCE_PATH)
    shared_pca_columns = [
        column_name
        for column_name in taxi_shared_pca_df.columns
        if column_name != MODEL_ID_COLUMN
    ]

    taxi_pre_pca_df = pd.read_parquet(TAXI_PERIOD_PCA_80_FINAL_PATHS_BY_PERIOD["pre_cp"])
    taxi_post_pca_df = pd.read_parquet(TAXI_PERIOD_PCA_80_FINAL_PATHS_BY_PERIOD["post_cp"])

    split_pca_columns = [
        column_name
        for column_name in taxi_pre_pca_df.columns
        if column_name != MODEL_ID_COLUMN
    ]

    taxi_split_pca_df = pd.concat(
        [
            taxi_metadata_df.loc[
                taxi_metadata_df[PRE_POST_CP_COLUMN].eq("pre_cp"),
                [MODEL_ID_COLUMN],
            ].merge(
                taxi_pre_pca_df,
                on=MODEL_ID_COLUMN,
                how="inner",
                validate="one_to_one",
            ),
            taxi_metadata_df.loc[
                taxi_metadata_df[PRE_POST_CP_COLUMN].eq("post_cp"),
                [MODEL_ID_COLUMN],
            ].merge(
                taxi_post_pca_df,
                on=MODEL_ID_COLUMN,
                how="inner",
                validate="one_to_one",
            ),
        ],
        ignore_index=True,
    )

    taxi_handling_specs = [
        {
            "pca_handling": "shared_full_period_pca",
            "representation_df": taxi_shared_pca_df,
            "feature_columns": shared_pca_columns,
            "grouping_columns": [TEMPORAL_BUCKET_COLUMN, POLICY_GEOGRAPHY_COLUMN],
        },
        {
            "pca_handling": "split_prepost_period_pca",
            "representation_df": taxi_split_pca_df,
            "feature_columns": split_pca_columns,
            "grouping_columns": [TEMPORAL_BUCKET_COLUMN, POLICY_GEOGRAPHY_COLUMN, PRE_POST_CP_COLUMN],
        },
    ]

    taxi_pca_handling_row_records = []
    taxi_pca_handling_summary_records = []

    for handling_spec in taxi_handling_specs:
        pca_handling_name = handling_spec["pca_handling"]
        representation_df = handling_spec["representation_df"]
        feature_columns = handling_spec["feature_columns"]
        grouping_columns = handling_spec["grouping_columns"]

        pilot_df = taxi_metadata_df.merge(
            representation_df[[MODEL_ID_COLUMN] + feature_columns],
            on=MODEL_ID_COLUMN,
            how="inner",
            validate="one_to_one",
        )

        pilot_df["comparison_group_key"] = (
            pilot_df[grouping_columns]
            .astype("string")
            .fillna("Missing")
            .agg(" | ".join, axis=1)
        )

        group_summary_records = []

        for comparison_group_key, group_df in pilot_df.groupby("comparison_group_key", dropna=False):
            if len(group_df) < DBSCAN_PILOT_MIN_ROWS_PER_GROUP:
                group_summary_records.append(
                    {
                        "comparison_group_key": comparison_group_key,
                        "group_rows": len(group_df),
                        "rows_evaluated": 0,
                        "eps": np.nan,
                        "noise_share": np.nan,
                        "cluster_count": np.nan,
                        "largest_cluster_share": np.nan,
                        "status": "skipped",
                    }
                )
                continue

            eval_group_df = (
                group_df.sample(
                    n=min(DBSCAN_PILOT_MAX_ROWS_PER_GROUP, len(group_df)),
                    random_state=ANOMALY_FRAMEWORK_RANDOM_STATE,
                )
                .copy()
            )

            X = eval_group_df[feature_columns].to_numpy(dtype="float32", copy=False)

            neighbor_k = min(DBSCAN_PILOT_MIN_SAMPLES, len(eval_group_df) - 1)
            if neighbor_k < 2:
                group_summary_records.append(
                    {
                        "comparison_group_key": comparison_group_key,
                        "group_rows": len(group_df),
                        "rows_evaluated": 0,
                        "eps": np.nan,
                        "noise_share": np.nan,
                        "cluster_count": np.nan,
                        "largest_cluster_share": np.nan,
                        "status": "skipped",
                    }
                )
                continue

            # Estimate eps within each Taxi comparison group so the pilot reflects
            # the local density scale implied by the shared-vs-split PCA handling choice.
            knn = NearestNeighbors(n_neighbors=neighbor_k)
            knn.fit(X)
            neighbor_distances = knn.kneighbors(X)[0][:, -1]
            eps = float(np.quantile(neighbor_distances, DBSCAN_PILOT_EPS_QUANTILE))

            dbscan_model = DBSCAN(
                eps=eps,
                min_samples=DBSCAN_PILOT_MIN_SAMPLES,
                metric="euclidean",
            )
            labels = dbscan_model.fit_predict(X)

            eval_group_df["pca_handling"] = pca_handling_name
            eval_group_df["comparison_group_key"] = comparison_group_key
            eval_group_df["pilot_eps"] = round(eps, 6)
            eval_group_df["dbscan_label"] = labels
            eval_group_df["is_noise"] = labels == -1

            cluster_labels = pd.Series(labels)
            non_noise_cluster_sizes = (
                cluster_labels.loc[cluster_labels.ne(-1)]
                .value_counts(normalize=True)
            )
            largest_cluster_share = (
                float(non_noise_cluster_sizes.iloc[0])
                if not non_noise_cluster_sizes.empty
                else np.nan
            )

            taxi_pca_handling_row_records.append(
                eval_group_df[
                    [
                        MODEL_ID_COLUMN,
                        DATE_COLUMN,
                        TEMPORAL_BUCKET_COLUMN,
                        PRE_POST_CP_COLUMN,
                        POLICY_GEOGRAPHY_COLUMN,
                        BOROUGH_COLUMN,
                        ZONE_ID_COLUMN,
                        ZONE_COLUMN,
                        "pca_handling",
                        "comparison_group_key",
                        "pilot_eps",
                        "dbscan_label",
                        "is_noise",
                    ]
                ]
            )

            group_summary_records.append(
                {
                    "comparison_group_key": comparison_group_key,
                    "group_rows": len(group_df),
                    "rows_evaluated": len(eval_group_df),
                    "eps": round(eps, 6),
                    "noise_share": float((labels == -1).mean()),
                    "cluster_count": int(len(set(labels)) - (1 if -1 in labels else 0)),
                    "largest_cluster_share": largest_cluster_share,
                    "status": "evaluated",
                }
            )

        group_summary_df = pd.DataFrame(group_summary_records)

        evaluated_group_df = group_summary_df.loc[
            group_summary_df["status"].eq("evaluated")
        ].copy()

        if len(evaluated_group_df) > 0:
            row_weights = evaluated_group_df["rows_evaluated"] / evaluated_group_df["rows_evaluated"].sum()

            weighted_noise_share = float(
                np.average(evaluated_group_df["noise_share"], weights=row_weights)
            )
            weighted_cluster_count = float(
                np.average(evaluated_group_df["cluster_count"], weights=row_weights)
            )
            weighted_largest_cluster_share = float(
                np.average(evaluated_group_df["largest_cluster_share"], weights=row_weights)
            )
        else:
            weighted_noise_share = np.nan
            weighted_cluster_count = np.nan
            weighted_largest_cluster_share = np.nan

        taxi_pca_handling_summary_records.append(
            {
                "pca_handling": pca_handling_name,
                "groups_evaluated": int(evaluated_group_df.shape[0]),
                "groups_skipped": int(group_summary_df["status"].eq("skipped").sum()),
                "rows_evaluated": int(evaluated_group_df["rows_evaluated"].sum()),
                "weighted_noise_share": weighted_noise_share,
                "weighted_cluster_count": weighted_cluster_count,
                "weighted_largest_cluster_share": weighted_largest_cluster_share,
                "status": "pass" if evaluated_group_df.shape[0] > 0 else "review",
            }
        )

    taxi_pca_handling_row_level_df = pd.concat(
        taxi_pca_handling_row_records,
        ignore_index=True,
    )
    taxi_pca_handling_summary_df = pd.DataFrame(taxi_pca_handling_summary_records)

    taxi_pca_handling_row_level_df.to_parquet(
        TAXI_PCA_HANDLING_ROW_LEVEL_PATH,
        index=False,
    )
    taxi_pca_handling_summary_df.to_parquet(
        TAXI_PCA_HANDLING_SUMMARY_PATH,
        index=False,
    )

    output_action = "written"
else:
    taxi_pca_handling_row_level_df = pd.read_parquet(TAXI_PCA_HANDLING_ROW_LEVEL_PATH)
    taxi_pca_handling_summary_df = pd.read_parquet(TAXI_PCA_HANDLING_SUMMARY_PATH)
    output_action = "loaded_existing"

taxi_pca_handling_summary_df = taxi_pca_handling_summary_df.copy()
taxi_pca_handling_summary_df["output_action"] = output_action

display(taxi_pca_handling_summary_df)

assert taxi_pca_handling_summary_df["status"].eq("pass").all(), (
    "Taxi shared-vs-split PCA pilot outputs did not materialize successfully."
)

print("Taxi shared-vs-split PCA pilot outputs are ready.")

,pca_handling,groups_evaluated,groups_skipped,rows_evaluated,weighted_noise_share,weighted_cluster_count,weighted_largest_cluster_share,status,output_action
0,shared_full_period_pca,30,0,42340,0.023453,1.705692,0.967803,pass,loaded_existing
1,split_prepost_period_pca,55,5,49590,0.025025,1.103408,0.985935,pass,loaded_existing


Taxi shared-vs-split PCA pilot outputs are ready.


Findings\. The shared and split Taxi PCA pilots are both runnable, but they do not behave the same way\. The shared full\-period PCA setup evaluates 30 temporal\-plus\-policy\-geography groups and produces a modest anomaly\-like share with about 1\.71 weighted clusters and a 0\.968 largest\-cluster share\. The split pre/post PCA setup expands to 55 evaluated groups, produces a slightly higher anomaly\-like share, but collapses much more strongly toward one dominant cluster pattern, with only about 1\.10 weighted clusters and a 0\.986 largest\-cluster share\. So at first pass, splitting Taxi PCA by period increases framework complexity without obviously improving downstream anomaly structure\.

### Compare Taxi Anomaly Behavior Under Shared vs Split PCA

We now have both Taxi PCA handling choices in hand: one shared full\-period PCA space and one split pre/post\-period PCA setup\. This section compares how those two choices behave under the same lightweight anomaly pilot so we can decide whether Taxi really needs special PCA treatment downstream\. We’ll look at three things: whether the split setup changes anomaly\-like volume and cluster structure, whether it actually flags different rows, and whether it meaningfully improves geographic concentration\. If the split\-period setup does not deliver a clear behavioral gain, then Taxi should stay on the shared PCA path with a caution note rather than a separate branch\.

Volume & Structure Check\. Let’s start with the broadest behavioral question\. Before worrying about which specific rows change, we want to know whether splitting Taxi PCA by period materially changes anomaly\-like volume or the overall DBSCAN structure of the Taxi anomaly surface\. Here, weighted\_noise\_share is the share of evaluated rows labeled as DBSCAN noise, so it works as the pilot’s anomaly\-like volume measure\. weighted\_cluster\_count summarizes how many non\-noise clusters the pilot finds across groups, so higher values indicate more internal segmentation\. weighted\_largest\_cluster\_share measures how much of the evaluated data is absorbed by the dominant cluster, so lower values mean the anomaly surface is less dominated by one broad mass\.

In [55]:
# ---------------------------------------------------------------------
# Compare Taxi anomaly volume and structure under shared vs split PCA
# ---------------------------------------------------------------------

if "taxi_pca_handling_summary_df" not in globals():
    taxi_pca_handling_summary_df = pd.read_parquet(TAXI_PCA_HANDLING_SUMMARY_PATH)

taxi_pca_behavior_summary_df = (
    taxi_pca_handling_summary_df[
        [
            "pca_handling",
            "groups_evaluated",
            "groups_skipped",
            "rows_evaluated",
            "weighted_noise_share",
            "weighted_cluster_count",
            "weighted_largest_cluster_share",
            "status",
        ]
    ]
    .copy()
)

handling_label_map = {
    "shared_full_period_pca": "Shared PCA",
    "split_prepost_period_pca": "Split pre/post PCA",
}

taxi_pca_behavior_summary_df["handling_label"] = (
    taxi_pca_behavior_summary_df["pca_handling"].map(handling_label_map)
)

display(taxi_pca_behavior_summary_df)

behavior_plot_df = taxi_pca_behavior_summary_df.melt(
    id_vars=["handling_label"],
    value_vars=[
        "weighted_noise_share",
        "weighted_cluster_count",
        "weighted_largest_cluster_share",
    ],
    var_name="metric_name",
    value_name="metric_value",
)

metric_label_map = {
    "weighted_noise_share": "Noise share",
    "weighted_cluster_count": "Weighted cluster count",
    "weighted_largest_cluster_share": "Largest-cluster share",
}
behavior_plot_df["metric_label"] = behavior_plot_df["metric_name"].map(metric_label_map)

taxi_behavior_fig = px.bar(
    behavior_plot_df,
    x="handling_label",
    y="metric_value",
    color="handling_label",
    facet_col="metric_label",
    category_orders={
        "handling_label": ["Shared PCA", "Split pre/post PCA"],
        "metric_label": [
            "Noise share",
            "Weighted cluster count",
            "Largest-cluster share",
        ],
    },
    color_discrete_map={
        "Shared PCA": BRAND_COLORS["dark_teal"],
        "Split pre/post PCA": BRAND_COLORS["terracotta"],
    },
    labels={
        "handling_label": "Taxi PCA handling",
        "metric_value": "Metric value",
        "metric_label": "",
    },
    title="Taxi Anomaly Volume and Structure Under Shared vs Split PCA",
)

apply_brand_plotly_layout(
    taxi_behavior_fig,
    title="Taxi Anomaly Volume and Structure Under Shared vs Split PCA",
    width=980,
    height=420,
)

taxi_behavior_fig.update_layout(
    showlegend=False,
    margin={"t": 70, "b": 70, "l": 60, "r": 20},
)

taxi_behavior_fig.for_each_annotation(
    lambda annotation: annotation.update(text=annotation.text.split("=")[-1])
)
taxi_behavior_fig.update_xaxes(title_text="")
taxi_behavior_fig.show()

assert taxi_pca_behavior_summary_df["status"].eq("pass").all(), (
    "Taxi PCA handling summary contains a failed status."
)

print("Taxi anomaly volume and structure compared.")

,pca_handling,groups_evaluated,groups_skipped,rows_evaluated,weighted_noise_share,weighted_cluster_count,weighted_largest_cluster_share,status,handling_label
0,shared_full_period_pca,30,0,42340,0.023453,1.705692,0.967803,pass,Shared PCA
1,split_prepost_period_pca,55,5,49590,0.025025,1.103408,0.985935,pass,Split pre/post PCA


Taxi anomaly volume and structure compared.


Findings\. The split pre/post PCA setup produces a slightly higher weighted noise share than the shared full\-period PCA setup, while also producing fewer clusters and a larger dominant cluster\. In this pilot, shared PCA has a weighted noise share of 0\.0235, weighted cluster count of 1\.7057, and weighted largest\-cluster share of 0\.9678, while split pre/post PCA has a weighted noise share of 0\.0250, weighted cluster count of 1\.1034, and weighted largest\-cluster share of 0\.9859\. So this block establishes that the two Taxi PCA handling choices do produce measurably different anomaly\-like structure under the same pilot procedure\.

Overlap Check\. Next, let’s check whether the two Taxi PCA handling choices are mostly flagging the same observations or actually producing different anomaly\-like sets\. This gives us a more direct sense of whether the split\-period setup changes Taxi anomaly behavior in substance rather than only in summary metrics\.

In [56]:
# ---------------------------------------------------------------------
# Compare Taxi anomaly overlap between shared and split PCA
# ---------------------------------------------------------------------

if "taxi_pca_handling_row_level_df" not in globals():
    taxi_pca_handling_row_level_df = pd.read_parquet(TAXI_PCA_HANDLING_ROW_LEVEL_PATH)

required_taxi_overlap_columns = [
    MODEL_ID_COLUMN,
    "pca_handling",
]

missing_taxi_overlap_columns = [
    column_name
    for column_name in required_taxi_overlap_columns
    if column_name not in taxi_pca_handling_row_level_df.columns
]

assert not missing_taxi_overlap_columns, (
    "Taxi PCA handling row-level output is missing required columns: "
    f"{missing_taxi_overlap_columns}"
)

if "is_noise" in taxi_pca_handling_row_level_df.columns:
    taxi_overlap_flag_df = taxi_pca_handling_row_level_df.copy()
    taxi_overlap_flag_df["anomaly_like_flag"] = taxi_overlap_flag_df["is_noise"].astype(bool)
elif "dbscan_label" in taxi_pca_handling_row_level_df.columns:
    taxi_overlap_flag_df = taxi_pca_handling_row_level_df.copy()
    taxi_overlap_flag_df["anomaly_like_flag"] = taxi_overlap_flag_df["dbscan_label"].eq(-1)
else:
    raise AssertionError(
        "Taxi PCA handling row-level output must contain is_noise or dbscan_label."
    )

taxi_overlap_wide_df = (
    taxi_overlap_flag_df.pivot_table(
        index=MODEL_ID_COLUMN,
        columns="pca_handling",
        values="anomaly_like_flag",
        aggfunc="max",
        fill_value=False,
    )
    .reset_index()
)

for handling_name in ["shared_full_period_pca", "split_prepost_period_pca"]:
    if handling_name not in taxi_overlap_wide_df.columns:
        taxi_overlap_wide_df[handling_name] = False

taxi_overlap_wide_df["shared_full_period_pca"] = (
    taxi_overlap_wide_df["shared_full_period_pca"].astype(bool)
)
taxi_overlap_wide_df["split_prepost_period_pca"] = (
    taxi_overlap_wide_df["split_prepost_period_pca"].astype(bool)
)

shared_flag_count = int(taxi_overlap_wide_df["shared_full_period_pca"].sum())
split_flag_count = int(taxi_overlap_wide_df["split_prepost_period_pca"].sum())
shared_split_overlap = int(
    (
        taxi_overlap_wide_df["shared_full_period_pca"]
        & taxi_overlap_wide_df["split_prepost_period_pca"]
    ).sum()
)
shared_only_rows = int(
    (
        taxi_overlap_wide_df["shared_full_period_pca"]
        & ~taxi_overlap_wide_df["split_prepost_period_pca"]
    ).sum()
)
split_only_rows = int(
    (
        taxi_overlap_wide_df["split_prepost_period_pca"]
        & ~taxi_overlap_wide_df["shared_full_period_pca"]
    ).sum()
)
union_rows = shared_flag_count + split_flag_count - shared_split_overlap
jaccard_similarity = shared_split_overlap / union_rows if union_rows > 0 else np.nan

taxi_pca_overlap_summary_df = pd.DataFrame(
    [
        {
            "rows_compared": len(taxi_overlap_wide_df),
            "shared_flag_count": shared_flag_count,
            "split_flag_count": split_flag_count,
            "shared_split_overlap": shared_split_overlap,
            "shared_only_rows": shared_only_rows,
            "split_only_rows": split_only_rows,
            "union_rows": union_rows,
            "jaccard_similarity": round(jaccard_similarity, 4),
            "status": "pass",
        }
    ]
)

display(taxi_pca_overlap_summary_df)

taxi_overlap_plot_df = pd.DataFrame(
    [
        {"segment": "Shared overlap", "rows": shared_split_overlap},
        {"segment": "Shared only", "rows": shared_only_rows},
        {"segment": "Split only", "rows": split_only_rows},
    ]
)

taxi_overlap_fig = px.bar(
    taxi_overlap_plot_df,
    x=["Taxi PCA handling comparison"] * len(taxi_overlap_plot_df),
    y="rows",
    color="segment",
    color_discrete_map={
        "Shared overlap": BRAND_COLORS["seafoam"],
        "Shared only": BRAND_COLORS["dark_teal"],
        "Split only": BRAND_COLORS["terracotta"],
    },
    labels={
        "x": "",
        "rows": "Anomaly-like rows",
        "segment": "Flag segment",
    },
    title="Taxi Shared-vs-Split PCA Anomaly Overlap",
)

apply_brand_plotly_layout(
    taxi_overlap_fig,
    title="Taxi Shared-vs-Split PCA Anomaly Overlap",
    width=700,
    height=420,
    legend_title="Flag segment",
)

taxi_overlap_fig.update_layout(
    barmode="stack",
    margin={"t": 70, "b": 70, "l": 60, "r": 20},
    legend={
        "title": {"text": "Flag segment"},
        "orientation": "h",
        "yanchor": "top",
        "y": -0.18,
        "xanchor": "center",
        "x": 0.5,
        "bgcolor": "rgba(255,255,255,0.85)",
        "bordercolor": BRAND_COLORS["seafoam"],
        "borderwidth": 1,
        "font": {"color": BRAND_COLORS["dark_teal"]},
    },
)
taxi_overlap_fig.update_xaxes(showticklabels=False, title_text="")
taxi_overlap_fig.show()

print("Taxi shared-vs-split PCA anomaly overlap compared.")

,rows_compared,shared_flag_count,split_flag_count,shared_split_overlap,shared_only_rows,split_only_rows,union_rows,jaccard_similarity,status
0,50000,993,1241,847,146,394,1387,0.6107,pass


Taxi shared-vs-split PCA anomaly overlap compared.


Findings\. The shared\-period and split\-period Taxi PCA pilots still agree on most anomaly\-like rows, but not completely\. Out of 50,000 compared rows, the two setups overlap on 847 anomaly\-like rows, with 146 flagged only by the shared PCA version and 394 flagged only by the split pre/post PCA version\. The split setup therefore adds a noticeably larger unique segment than the shared setup, and the overall Jaccard similarity of 0\.6107 indicates substantial but far\-from\-perfect agreement between the two Taxi anomaly surfaces\.

Geographic Overlap\. Finally, let’s test the part that matters most for interpretation\. Even if the two Taxi PCA handling choices produce similar anomaly volumes, the split\-period setup might still earn its keep if it meaningfully reduces Manhattan, CBD, or top\-zone concentration in the resulting anomaly\-like surface\.

In [57]:
# ---------------------------------------------------------------------
# Compare Taxi geographic concentration under shared vs split PCA
# ---------------------------------------------------------------------

required_taxi_geo_columns = [
    MODEL_ID_COLUMN,
    "pca_handling",
    PRE_POST_CP_COLUMN,
    POLICY_GEOGRAPHY_COLUMN,
    BOROUGH_COLUMN,
    ZONE_ID_COLUMN,
]

missing_taxi_geo_columns = [
    column_name
    for column_name in required_taxi_geo_columns
    if column_name not in taxi_overlap_flag_df.columns
]

assert not missing_taxi_geo_columns, (
    "Taxi PCA handling row-level output is missing geography columns: "
    f"{missing_taxi_geo_columns}"
)

taxi_geo_source_df = taxi_overlap_flag_df.copy()
taxi_geo_source_df["anomaly_like_flag"] = taxi_geo_source_df["anomaly_like_flag"].astype(bool)

taxi_geo_concentration_records = []

for handling_name, handling_df in taxi_geo_source_df.groupby("pca_handling", dropna=False):
    evaluated_df = handling_df.copy()
    anomaly_df = evaluated_df.loc[evaluated_df["anomaly_like_flag"]].copy()

    source_manhattan_share = (
        evaluated_df[BOROUGH_COLUMN].astype("string").eq("Manhattan").mean()
    )
    anomaly_manhattan_share = (
        anomaly_df[BOROUGH_COLUMN].astype("string").eq("Manhattan").mean()
        if len(anomaly_df) > 0 else np.nan
    )

    source_cbd_share = (
        evaluated_df[POLICY_GEOGRAPHY_COLUMN].astype("string").eq("cbd").mean()
    )
    anomaly_cbd_share = (
        anomaly_df[POLICY_GEOGRAPHY_COLUMN].astype("string").eq("cbd").mean()
        if len(anomaly_df) > 0 else np.nan
    )

    top10_zone_ids = (
        evaluated_df[ZONE_ID_COLUMN]
        .value_counts()
        .head(10)
        .index.tolist()
    )

    source_top10_zone_share = (
        evaluated_df[ZONE_ID_COLUMN].isin(top10_zone_ids).mean()
    )
    anomaly_top10_zone_share = (
        anomaly_df[ZONE_ID_COLUMN].isin(top10_zone_ids).mean()
        if len(anomaly_df) > 0 else np.nan
    )

    taxi_geo_concentration_records.append(
        {
            "pca_handling": handling_name,
            "evaluated_rows": len(evaluated_df),
            "anomaly_rows": len(anomaly_df),
            "source_manhattan_share": source_manhattan_share,
            "anomaly_manhattan_share": anomaly_manhattan_share,
            "manhattan_lift": (
                anomaly_manhattan_share / source_manhattan_share
                if source_manhattan_share > 0 and pd.notna(anomaly_manhattan_share)
                else np.nan
            ),
            "source_cbd_share": source_cbd_share,
            "anomaly_cbd_share": anomaly_cbd_share,
            "cbd_lift": (
                anomaly_cbd_share / source_cbd_share
                if source_cbd_share > 0 and pd.notna(anomaly_cbd_share)
                else np.nan
            ),
            "source_top10_zone_share": source_top10_zone_share,
            "anomaly_top10_zone_share": anomaly_top10_zone_share,
            "top10_zone_lift": (
                anomaly_top10_zone_share / source_top10_zone_share
                if source_top10_zone_share > 0 and pd.notna(anomaly_top10_zone_share)
                else np.nan
            ),
            "status": "pass",
        }
    )

taxi_geo_concentration_df = pd.DataFrame(taxi_geo_concentration_records)
taxi_geo_concentration_df["handling_label"] = (
    taxi_geo_concentration_df["pca_handling"].map(handling_label_map)
)

display(taxi_geo_concentration_df)

taxi_geo_plot_df = taxi_geo_concentration_df.melt(
    id_vars=["handling_label"],
    value_vars=["manhattan_lift", "cbd_lift", "top10_zone_lift"],
    var_name="metric_name",
    value_name="lift_value",
)

geo_metric_label_map = {
    "manhattan_lift": "Manhattan lift",
    "cbd_lift": "CBD lift",
    "top10_zone_lift": "Top-10 zone lift",
}
taxi_geo_plot_df["metric_label"] = taxi_geo_plot_df["metric_name"].map(geo_metric_label_map)

taxi_geo_fig = px.bar(
    taxi_geo_plot_df,
    x="handling_label",
    y="lift_value",
    color="handling_label",
    facet_col="metric_label",
    category_orders={
        "handling_label": ["Shared PCA", "Split pre/post PCA"],
        "metric_label": ["Manhattan lift", "CBD lift", "Top-10 zone lift"],
    },
    color_discrete_map={
        "Shared PCA": BRAND_COLORS["dark_teal"],
        "Split pre/post PCA": BRAND_COLORS["terracotta"],
    },
    labels={
        "handling_label": "Taxi PCA handling",
        "lift_value": "Lift relative to source prevalence",
        "metric_label": "",
    },
    title="Taxi Geographic Concentration Under Shared vs Split PCA",
)

apply_brand_plotly_layout(
    taxi_geo_fig,
    title="Taxi Geographic Concentration Under Shared vs Split PCA",
    width=980,
    height=420,
)

taxi_geo_fig.update_layout(
    showlegend=False,
    margin={"t": 70, "b": 70, "l": 60, "r": 20},
)
taxi_geo_fig.for_each_annotation(
    lambda annotation: annotation.update(text=annotation.text.split("=")[-1])
)
taxi_geo_fig.update_xaxes(title_text="")
taxi_geo_fig.show()

assert taxi_geo_concentration_df["status"].eq("pass").all(), (
    "Taxi geographic concentration comparison failed."
)

print("Taxi geographic concentration compared.")

,pca_handling,evaluated_rows,anomaly_rows,source_manhattan_share,anomaly_manhattan_share,manhattan_lift,source_cbd_share,anomaly_cbd_share,cbd_lift,source_top10_zone_share,anomaly_top10_zone_share,top10_zone_lift,status,handling_label
0,shared_full_period_pca,42340,993,0.274091,0.323263,1.179401,0.173477,0.154079,0.888181,0.056802,0.067472,1.187849,pass,Shared PCA
1,split_prepost_period_pca,49590,1241,0.248034,0.261080,1.052597,0.148115,0.111201,0.750775,0.052228,0.032232,0.617138,pass,Split pre/post PCA


Taxi geographic concentration compared.


Findings\. The split pre/post Taxi PCA setup produces less geographic concentration in its anomaly\-like rows than the shared full\-period PCA setup\. Manhattan lift declines from 1\.179 to 1\.053, CBD lift declines from 0\.888 to 0\.751, and top\-10\-zone lift declines much more sharply from 1\.188 to 0\.617\. So this block shows that splitting Taxi PCA by policy period changes not just which rows are flagged, but also where those anomaly\-like rows concentrate geographically\.

> 🎯 Decision: Use split pre/post PCA handling for Taxi only\. The split Taxi PCA branch changes a substantial share of anomaly\-like rows relative to shared PCA and reduces geographic concentration, especially top\-zone dominance\. Although the broad DBSCAN structure summary is not uniformly stronger, the policy\-period split appears meaningful enough to justify Taxi\-specific handling downstream\.

## 3\.3\.1\.6 Select Shared Anomaly Framework Defaults

At this point, the notebook has already done the actual decision work: it selected the shared anomaly representations, established the calibration sample design, chose the downstream comparison universe, and surfaced the main caution areas that later anomaly notebooks should inherit\. This section turns those choices into a clean shared handoff for DBSCAN, Isolation Forest, and Gaussian Mixture workflows\.

The goal here is not to reopen the analysis\. It is to make the downstream framework explicit so later notebooks can start from the same defaults instead of redefining representations, samples, baselines, and interpretation rules independently\.

Let’s start by writing down the representation decision in one place\. This gives the downstream anomaly notebooks a single shared representation registry: which representations advance, which role each one plays, and whether any modality\-specific handling needs to be respected\.

In [58]:
# ---------------------------------------------------------------------
# Record the shared downstream representation set
# ---------------------------------------------------------------------

# Set the Taxi PCA handling choice here once we are ready to lock it in.
# Valid options:
# - "shared_full_period"
# - "split_prepost_period"
TAXI_PCA_HANDLING_DECISION = "split_prepost_period"

shared_representation_records = []

for feature_set_name in MODEL_FEATURE_SET_NAMES:
    # Raw scaled is retained only as an upstream source/reference surface,
    # not as a shared downstream anomaly-generation branch.
    shared_representation_records.append(
        {
            "feature_set": feature_set_name,
            "representation_name": "raw_scaled",
            "advance_downstream": False,
            "representation_role": "reference_only",
            "downstream_status": "retired_as_primary_branch",
            "path": str(SCALED_MATRIX_PATHS_BY_SET[feature_set_name]),
            "notes": "Retained as an upstream scaled source, but not advanced as a shared anomaly-generation branch.",
        }
    )

    # PCA is the primary reduced linear representation.
    if feature_set_name == "taxi" and TAXI_PCA_HANDLING_DECISION == "split_prepost_period":
        pca_path_value = json.dumps(
            {
                "pre_cp": str(TAXI_PERIOD_PCA_80_FINAL_PATHS_BY_PERIOD["pre_cp"]),
                "post_cp": str(TAXI_PERIOD_PCA_80_FINAL_PATHS_BY_PERIOD["post_cp"]),
            }
        )
        pca_notes = (
            "Taxi uses period-specific PCA assets because pre/post congestion-pricing "
            "structure was judged meaningful enough to preserve downstream."
        )
    else:
        pca_path_value = str(PCA_80_FINAL_PATHS_BY_SET[feature_set_name])
        pca_notes = (
            "Uses the shared 80%-variance PCA representation exported for this modality."
        )

    shared_representation_records.append(
        {
            "feature_set": feature_set_name,
            "representation_name": "pca_80pct",
            "advance_downstream": True,
            "representation_role": "primary_linear_representation",
            "downstream_status": "shared_default",
            "path": pca_path_value,
            "notes": pca_notes,
        }
    )

    # UMAP is retained as the nonlinear complement.
    shared_representation_records.append(
        {
            "feature_set": feature_set_name,
            "representation_name": "umap_pca_to_umap_10d",
            "advance_downstream": True,
            "representation_role": "nonlinear_complement_representation",
            "downstream_status": "shared_default",
            "path": str(UMAP_PCA_TO_UMAP_10D_FINAL_PATHS_BY_SET[feature_set_name]),
            "notes": (
                "Retained as the shared nonlinear anomaly representation built from "
                "PCA inputs and exported in 10 dimensions."
            ),
        }
    )

shared_representation_defaults_df = pd.DataFrame(shared_representation_records)

display(shared_representation_defaults_df)

,feature_set,representation_name,advance_downstream,representation_role,downstream_status,path,notes
0,bus,raw_scaled,False,reference_only,retired_as_primary_branch,/datasets/_deepnote_work/pipeline_data/3.1.1.f...,"Retained as an upstream scaled source, but not..."
1,bus,pca_80pct,True,primary_linear_representation,shared_default,/datasets/_deepnote_work/pipeline_data/3.2.1.f...,Uses the shared 80%-variance PCA representatio...
2,bus,umap_pca_to_umap_10d,True,nonlinear_complement_representation,shared_default,/datasets/_deepnote_work/pipeline_data/3.2.4.f...,Retained as the shared nonlinear anomaly repre...
3,subway,raw_scaled,False,reference_only,retired_as_primary_branch,/datasets/_deepnote_work/pipeline_data/3.1.1.f...,"Retained as an upstream scaled source, but not..."
4,subway,pca_80pct,True,primary_linear_representation,shared_default,/datasets/_deepnote_work/pipeline_data/3.2.1.f...,Uses the shared 80%-variance PCA representatio...
5,subway,umap_pca_to_umap_10d,True,nonlinear_complement_representation,shared_default,/datasets/_deepnote_work/pipeline_data/3.2.4.f...,Retained as the shared nonlinear anomaly repre...
6,taxi,raw_scaled,False,reference_only,retired_as_primary_branch,/datasets/_deepnote_work/pipeline_data/3.1.1.f...,"Retained as an upstream scaled source, but not..."
7,taxi,pca_80pct,True,primary_linear_representation,shared_default,"{""pre_cp"": ""/datasets/_deepnote_work/pipeline_...",Taxi uses period-specific PCA assets because p...
8,taxi,umap_pca_to_umap_10d,True,nonlinear_complement_representation,shared_default,/datasets/_deepnote_work/pipeline_data/3.2.4.f...,Retained as the shared nonlinear anomaly repre...
9,fhvhv,raw_scaled,False,reference_only,retired_as_primary_branch,/datasets/_deepnote_work/pipeline_data/3.1.1.f...,"Retained as an upstream scaled source, but not..."


Findings\. The downstream anomaly framework will advance two shared representations for every modality: pca\_80pct as the primary linear representation and umap\_pca\_to\_umap\_10d as the nonlinear complement\. raw\_scaled is retained only as an upstream reference surface and will not continue as a separate anomaly\-generation branch\. Taxi is the one special case: its PCA handoff uses separate pre\_cp and post\_cp assets, while the other four modalities continue with their shared full\-period PCA exports\.

This block writes down the common sample and metadata assets that every downstream anomaly notebook should inherit\. It keeps the shared evaluation footing explicit without reopening any sampling decisions\.

In [59]:
# ---------------------------------------------------------------------
# Record the shared calibration sample and metadata layer
# ---------------------------------------------------------------------

shared_calibration_records = []

for feature_set_name in MODEL_FEATURE_SET_NAMES:
    shared_calibration_records.append(
        {
            "feature_set": feature_set_name,
            "sample_id_path": str(FINAL_ANOMALY_CALIBRATION_ID_PATHS_BY_SET[feature_set_name]),
            "sample_metadata_path": str(FINAL_ANOMALY_CALIBRATION_METADATA_PATHS_BY_SET[feature_set_name]),
            "target_sample_rows": ANOMALY_CALIBRATION_SAMPLE_ROWS,
            "sample_design": "temporal_bucket_plus_pre_post_stratified",
            "shared_metadata_columns": ", ".join(DEFAULT_METADATA_COLUMNS),
            "downstream_status": "shared_default",
            "notes": (
                "Use the shared calibration ids and aligned metadata as the common "
                "foundation for DBSCAN, Isolation Forest, and GMM diagnostics."
            ),
        }
    )

shared_calibration_defaults_df = pd.DataFrame(shared_calibration_records)

display(shared_calibration_defaults_df)

,feature_set,sample_id_path,sample_metadata_path,target_sample_rows,sample_design,shared_metadata_columns,downstream_status,notes
0,bus,/datasets/_deepnote_work/pipeline_data/3.3.1.f...,/datasets/_deepnote_work/pipeline_data/3.3.1.f...,50000,temporal_bucket_plus_pre_post_stratified,"modeling_row_id, taxi_zone_id, date, temporal_...",shared_default,Use the shared calibration ids and aligned met...
1,subway,/datasets/_deepnote_work/pipeline_data/3.3.1.f...,/datasets/_deepnote_work/pipeline_data/3.3.1.f...,50000,temporal_bucket_plus_pre_post_stratified,"modeling_row_id, taxi_zone_id, date, temporal_...",shared_default,Use the shared calibration ids and aligned met...
2,taxi,/datasets/_deepnote_work/pipeline_data/3.3.1.f...,/datasets/_deepnote_work/pipeline_data/3.3.1.f...,50000,temporal_bucket_plus_pre_post_stratified,"modeling_row_id, taxi_zone_id, date, temporal_...",shared_default,Use the shared calibration ids and aligned met...
3,fhvhv,/datasets/_deepnote_work/pipeline_data/3.3.1.f...,/datasets/_deepnote_work/pipeline_data/3.3.1.f...,50000,temporal_bucket_plus_pre_post_stratified,"modeling_row_id, taxi_zone_id, date, temporal_...",shared_default,Use the shared calibration ids and aligned met...
4,multimodal,/datasets/_deepnote_work/pipeline_data/3.3.1.f...,/datasets/_deepnote_work/pipeline_data/3.3.1.f...,50000,temporal_bucket_plus_pre_post_stratified,"modeling_row_id, taxi_zone_id, date, temporal_...",shared_default,Use the shared calibration ids and aligned met...


Findings\. All five modalities will inherit the same shared anomaly\-calibration design: a 50,000\-row calibration sample with aligned metadata, stratified by temporal\_bucket and pre\_post\_cp\. This gives DBSCAN, Isolation Forest, and GMM a common set of observations and the same interpretation fields, so downstream differences reflect method behavior rather than sample drift\.

Now let’s write down the comparison universe that downstream anomaly scoring should actually use\. This is where we make the shared temporal\-plus\-policy\-geography baseline explicit, while keeping Taxi’s pre/post variant as a targeted diagnostic rather than a second full default\.

In [60]:
# ---------------------------------------------------------------------
# Record the comparison universes advancing downstream
# ---------------------------------------------------------------------

shared_comparison_universe_records = []

for feature_set_name in MODEL_FEATURE_SET_NAMES:
    shared_comparison_universe_records.append(
        {
            "feature_set": feature_set_name,
            "comparison_universe": "same_temporal_bucket_and_policy_geography",
            "grouping_columns": ", ".join(
                COMPARISON_UNIVERSE_GROUPING_COLUMNS["same_temporal_bucket_and_policy_geography"]
            ),
            "downstream_status": "shared_default",
            "downstream_role": "primary_scoring_baseline",
            "notes": (
                "Primary anomaly comparison universe. Score observations against rows "
                "from the same temporal bucket and policy-geography class."
            ),
        }
    )

    if feature_set_name == "taxi":
        shared_comparison_universe_records.append(
            {
                "feature_set": feature_set_name,
                "comparison_universe": "same_temporal_bucket_and_pre_post_cp",
                "grouping_columns": ", ".join(
                    COMPARISON_UNIVERSE_GROUPING_COLUMNS["same_temporal_bucket_and_pre_post_cp"]
                ),
                "downstream_status": "targeted_diagnostic",
                "downstream_role": "taxi_policy_sanity_check",
                "notes": (
                    "Retained only for Taxi as a targeted policy-period diagnostic, "
                    "not as the shared default scoring baseline."
                ),
            }
        )

shared_comparison_universe_defaults_df = pd.DataFrame(shared_comparison_universe_records)

display(shared_comparison_universe_defaults_df)

,feature_set,comparison_universe,grouping_columns,downstream_status,downstream_role,notes
0,bus,same_temporal_bucket_and_policy_geography,"temporal_bucket, policy_geography_class",shared_default,primary_scoring_baseline,Primary anomaly comparison universe. Score obs...
1,subway,same_temporal_bucket_and_policy_geography,"temporal_bucket, policy_geography_class",shared_default,primary_scoring_baseline,Primary anomaly comparison universe. Score obs...
2,taxi,same_temporal_bucket_and_policy_geography,"temporal_bucket, policy_geography_class",shared_default,primary_scoring_baseline,Primary anomaly comparison universe. Score obs...
3,taxi,same_temporal_bucket_and_pre_post_cp,"temporal_bucket, pre_post_cp",targeted_diagnostic,taxi_policy_sanity_check,Retained only for Taxi as a targeted policy-pe...
4,fhvhv,same_temporal_bucket_and_policy_geography,"temporal_bucket, policy_geography_class",shared_default,primary_scoring_baseline,Primary anomaly comparison universe. Score obs...
5,multimodal,same_temporal_bucket_and_policy_geography,"temporal_bucket, policy_geography_class",shared_default,primary_scoring_baseline,Primary anomaly comparison universe. Score obs...


Findings\. The shared downstream scoring baseline is same\_temporal\_bucket\_and\_policy\_geography for all five modalities\. Taxi retains one extra diagnostic branch, same\_temporal\_bucket\_and\_pre\_post\_cp, as a targeted policy\-period check, but it does not replace the shared default baseline for the rest of the downstream workflow\.

This final handoff block records the interpretation defaults that later notebooks should inherit\. It keeps the framework practical: which representation roles are active, which baseline is standard, and which caution flags still matter when we start generating real anomaly outputs\.

In [61]:
# ---------------------------------------------------------------------
# Record the provisional self-normalization and geography-caution defaults
# ---------------------------------------------------------------------

shared_anomaly_framework_default_records = []

for feature_set_name in MODEL_FEATURE_SET_NAMES:
    if feature_set_name == "taxi":
        taxi_pca_handling_note = (
            "Use split pre/post PCA handling for Taxi."
            if TAXI_PCA_HANDLING_DECISION == "split_prepost_period"
            else "Use shared full-period PCA handling for Taxi."
        )
    else:
        taxi_pca_handling_note = "Not applicable."

    shared_anomaly_framework_default_records.append(
        {
            "feature_set": feature_set_name,
            "primary_linear_representation": "pca_80pct",
            "nonlinear_complement_representation": "umap_pca_to_umap_10d",
            "raw_scaled_branch_status": "retired_as_primary_branch",
            "shared_scoring_baseline": "same_temporal_bucket_and_policy_geography",
            "assume_features_are_provisionally_self_normalizing": True,
            "retain_core_geography_caution": True,
            "retain_top_zone_dominance_caution": True,
            "taxi_pca_handling": taxi_pca_handling_note,
            "downstream_note": (
                "Proceed with PCA and UMAP as the shared anomaly representations. "
                "Interpret anomaly outputs with continued attention to geographic "
                "concentration and repeated-zone dominance."
            ),
        }
    )

shared_anomaly_framework_defaults_df = pd.DataFrame(
    shared_anomaly_framework_default_records
)

display(shared_anomaly_framework_defaults_df)

,feature_set,primary_linear_representation,nonlinear_complement_representation,raw_scaled_branch_status,shared_scoring_baseline,assume_features_are_provisionally_self_normalizing,retain_core_geography_caution,retain_top_zone_dominance_caution,taxi_pca_handling,downstream_note
0,bus,pca_80pct,umap_pca_to_umap_10d,retired_as_primary_branch,same_temporal_bucket_and_policy_geography,True,True,True,Not applicable.,Proceed with PCA and UMAP as the shared anomal...
1,subway,pca_80pct,umap_pca_to_umap_10d,retired_as_primary_branch,same_temporal_bucket_and_policy_geography,True,True,True,Not applicable.,Proceed with PCA and UMAP as the shared anomal...
2,taxi,pca_80pct,umap_pca_to_umap_10d,retired_as_primary_branch,same_temporal_bucket_and_policy_geography,True,True,True,Use split pre/post PCA handling for Taxi.,Proceed with PCA and UMAP as the shared anomal...
3,fhvhv,pca_80pct,umap_pca_to_umap_10d,retired_as_primary_branch,same_temporal_bucket_and_policy_geography,True,True,True,Not applicable.,Proceed with PCA and UMAP as the shared anomal...
4,multimodal,pca_80pct,umap_pca_to_umap_10d,retired_as_primary_branch,same_temporal_bucket_and_policy_geography,True,True,True,Not applicable.,Proceed with PCA and UMAP as the shared anomal...


Findings\. The downstream notebooks will proceed with pca\_80pct as the primary linear representation, umap\_pca\_to\_umap\_10d as the nonlinear complement, and same\_temporal\_bucket\_and\_policy\_geography as the shared scoring baseline\. The framework will also carry forward two standing interpretation cautions: anomaly outputs should still be checked for core\-geography concentration and for repeated\-zone dominance, with Taxi additionally inheriting its modality\-specific PCA handling decision\.

## 3\.3\.1\.7 Export Shared Anomaly Framework Assets

By this point, the notebook has already settled the main shared framework choices: which representations advance, how the calibration sample is defined, which comparison universe should be treated as the downstream default, and which modality\-specific cautions later notebooks need to inherit\. This final section packages those decisions into a reusable handoff so 3\.3\.2, 3\.3\.3, and 3\.3\.4 can start from the same anomaly framework instead of rebuilding it independently\.

The exports here are intentionally compact\. They are meant to preserve the shared defaults, supporting asset references, and framework\-level summaries that later anomaly notebooks will need in order to stay aligned\.

This block assembles the final handoff tables and writes them into 3\.3\.1\.final\_tables\.

In [62]:
# ---------------------------------------------------------------------
# Export shared anomaly framework tables
# ---------------------------------------------------------------------

ANOMALY_FRAMEWORK_FINAL_EXPORT_PATHS = {
    "shared_representation_defaults": FINAL_331_DIR / "shared_representation_defaults.csv",
    "shared_calibration_defaults": FINAL_331_DIR / "shared_calibration_defaults.csv",
    "shared_comparison_universe_defaults": FINAL_331_DIR / "shared_comparison_universe_defaults.csv",
    "shared_framework_interpretation_defaults": FINAL_331_DIR / "shared_framework_interpretation_defaults.csv",
    "shared_framework_asset_manifest": FINAL_331_DIR / "shared_framework_asset_manifest.csv",
}

# Keep the manifest lightweight and path-focused so downstream notebooks
# can discover the important framework assets from one place.
shared_framework_asset_manifest_df = pd.DataFrame(
    [
        {
            "asset_group": "representations",
            "asset_name": "shared_representation_defaults",
            "path": str(ANOMALY_FRAMEWORK_FINAL_EXPORT_PATHS["shared_representation_defaults"]),
            "description": "Shared representation handoff registry for Raw, PCA, and UMAP branches.",
        },
        {
            "asset_group": "calibration",
            "asset_name": "shared_calibration_defaults",
            "path": str(ANOMALY_FRAMEWORK_FINAL_EXPORT_PATHS["shared_calibration_defaults"]),
            "description": "Shared calibration sample ids, metadata paths, and sample-design defaults.",
        },
        {
            "asset_group": "baselines",
            "asset_name": "shared_comparison_universe_defaults",
            "path": str(ANOMALY_FRAMEWORK_FINAL_EXPORT_PATHS["shared_comparison_universe_defaults"]),
            "description": "Downstream comparison-universe defaults, including the Taxi policy sanity check.",
        },
        {
            "asset_group": "interpretation",
            "asset_name": "shared_framework_interpretation_defaults",
            "path": str(ANOMALY_FRAMEWORK_FINAL_EXPORT_PATHS["shared_framework_interpretation_defaults"]),
            "description": "Framework-level interpretation defaults, cautions, and Taxi PCA handling choice.",
        },
        {
            "asset_group": "supporting_inputs",
            "asset_name": "comparison_universe_group_sizes",
            "path": str(DBSCAN_UNIVERSE_GROUP_SIZE_PATH),
            "description": "Feasibility summary for candidate comparison universes.",
        },
        {
            "asset_group": "supporting_inputs",
            "asset_name": "dbscan_baseline_pilot_summary",
            "path": str(DBSCAN_PILOT_SUMMARY_PATH),
            "description": "Pilot DBSCAN baseline summary used to choose the shared comparison universe.",
        },
        {
            "asset_group": "calibration",
            "asset_name": "shared_calibration_id_parquets",
            "path": str(FINAL_331_DIR),
            "description": "Final parquet exports for shared anomaly calibration sample ids by modality.",
        },
        {
            "asset_group": "calibration",
            "asset_name": "shared_calibration_metadata_parquets",
            "path": str(FINAL_331_DIR),
            "description": "Final parquet exports for shared anomaly calibration sample metadata by modality.",
        },
    ]
)

shared_framework_export_tables = {
    "shared_representation_defaults": shared_representation_defaults_df.copy(),
    "shared_calibration_defaults": shared_calibration_defaults_df.copy(),
    "shared_comparison_universe_defaults": shared_comparison_universe_defaults_df.copy(),
    "shared_framework_interpretation_defaults": shared_anomaly_framework_defaults_df.copy(),
    "shared_framework_asset_manifest": shared_framework_asset_manifest_df.copy(),
}

shared_framework_export_records = []

for export_name, export_df in shared_framework_export_tables.items():
    export_path = ANOMALY_FRAMEWORK_FINAL_EXPORT_PATHS[export_name]
    export_path.parent.mkdir(parents=True, exist_ok=True)

    if should_rebuild(WRITE_331_OUTPUTS, export_path):
        export_df.to_csv(export_path, index=False)
        output_action = "written"
    else:
        output_action = "kept_existing"

    shared_framework_export_records.append(
        {
            "export_name": export_name,
            "output_path": str(export_path),
            "rows": int(len(export_df)),
            "columns": int(len(export_df.columns)),
            "output_action": output_action,
            "status": "pass" if export_path.exists() else "missing",
        }
    )

shared_framework_export_summary_df = pd.DataFrame(shared_framework_export_records)

display(shared_framework_export_summary_df)

assert shared_framework_export_summary_df["status"].eq("pass").all(), (
    "One or more shared anomaly framework exports was not written successfully."
)

,export_name,output_path,rows,columns,output_action,status
0,shared_representation_defaults,/datasets/_deepnote_work/pipeline_data/3.3.1.f...,15,7,written,pass
1,shared_calibration_defaults,/datasets/_deepnote_work/pipeline_data/3.3.1.f...,5,8,written,pass
2,shared_comparison_universe_defaults,/datasets/_deepnote_work/pipeline_data/3.3.1.f...,6,6,written,pass
3,shared_framework_interpretation_defaults,/datasets/_deepnote_work/pipeline_data/3.3.1.f...,5,10,written,pass
4,shared_framework_asset_manifest,/datasets/_deepnote_work/pipeline_data/3.3.1.f...,8,4,written,pass


Findings\. The shared anomaly framework handoff tables were assembled and written successfully into 3\.3\.1\.final\_tables\. These exports capture the downstream representation decisions, calibration\-sample defaults, comparison\-universe defaults, interpretation defaults, and a compact manifest of the supporting framework assets\.

The shared calibration sample is now part of the formal downstream anomaly framework, so its parquet assets should live in 3\.3\.1\.final\_tables rather than remain only in the intermediate workspace\. This block promotes the shared sample ids and aligned metadata into final handoff assets\.

In [63]:
# ---------------------------------------------------------------------
# Promote shared calibration parquet assets to final tables
# ---------------------------------------------------------------------

shared_calibration_asset_export_records = []

for feature_set_name in MODEL_FEATURE_SET_NAMES:
    export_pairs = [
        (
            "sample_ids",
            ANOMALY_CALIBRATION_ID_PATHS_BY_SET[feature_set_name],
            FINAL_ANOMALY_CALIBRATION_ID_PATHS_BY_SET[feature_set_name],
        ),
        (
            "sample_metadata",
            ANOMALY_CALIBRATION_METADATA_PATHS_BY_SET[feature_set_name],
            FINAL_ANOMALY_CALIBRATION_METADATA_PATHS_BY_SET[feature_set_name],
        ),
    ]

    for asset_kind, source_path, final_path in export_pairs:
        final_path.parent.mkdir(parents=True, exist_ok=True)

        if should_rebuild(final_path, WRITE_331_OUTPUTS):
            asset_df = pd.read_parquet(source_path)
            asset_df.to_parquet(final_path, index=False)
            output_action = "written"
        else:
            output_action = "kept_existing"

        exported_rows = len(pd.read_parquet(final_path))

        shared_calibration_asset_export_records.append(
            {
                "feature_set": feature_set_name,
                "asset_kind": asset_kind,
                "source_path": str(source_path),
                "final_path": str(final_path),
                "rows": int(exported_rows),
                "output_action": output_action,
                "status": "pass" if final_path.exists() else "missing",
            }
        )

shared_calibration_asset_export_df = pd.DataFrame(
    shared_calibration_asset_export_records
)

display(shared_calibration_asset_export_df)

assert shared_calibration_asset_export_df["status"].eq("pass").all(), (
    "One or more shared calibration parquet assets was not promoted to 3.3.1.final_tables."
)

,feature_set,asset_kind,source_path,final_path,rows,output_action,status
0,bus,sample_ids,/datasets/_deepnote_work/pipeline_data/3.3.1.i...,/datasets/_deepnote_work/pipeline_data/3.3.1.f...,50000,written,pass
1,bus,sample_metadata,/datasets/_deepnote_work/pipeline_data/3.3.1.i...,/datasets/_deepnote_work/pipeline_data/3.3.1.f...,50000,written,pass
2,subway,sample_ids,/datasets/_deepnote_work/pipeline_data/3.3.1.i...,/datasets/_deepnote_work/pipeline_data/3.3.1.f...,50000,written,pass
3,subway,sample_metadata,/datasets/_deepnote_work/pipeline_data/3.3.1.i...,/datasets/_deepnote_work/pipeline_data/3.3.1.f...,50000,written,pass
4,taxi,sample_ids,/datasets/_deepnote_work/pipeline_data/3.3.1.i...,/datasets/_deepnote_work/pipeline_data/3.3.1.f...,50000,written,pass
5,taxi,sample_metadata,/datasets/_deepnote_work/pipeline_data/3.3.1.i...,/datasets/_deepnote_work/pipeline_data/3.3.1.f...,50000,written,pass
6,fhvhv,sample_ids,/datasets/_deepnote_work/pipeline_data/3.3.1.i...,/datasets/_deepnote_work/pipeline_data/3.3.1.f...,50000,written,pass
7,fhvhv,sample_metadata,/datasets/_deepnote_work/pipeline_data/3.3.1.i...,/datasets/_deepnote_work/pipeline_data/3.3.1.f...,50000,written,pass
8,multimodal,sample_ids,/datasets/_deepnote_work/pipeline_data/3.3.1.i...,/datasets/_deepnote_work/pipeline_data/3.3.1.f...,50000,written,pass
9,multimodal,sample_metadata,/datasets/_deepnote_work/pipeline_data/3.3.1.i...,/datasets/_deepnote_work/pipeline_data/3.3.1.f...,50000,written,pass


Findings\. The shared anomaly calibration parquet assets were promoted into 3\.3\.1\.final\_tables for all five modalities\. That makes the calibration sample itself part of the formal downstream framework handoff rather than leaving later notebooks dependent on 3\.3\.1\.intermediate\_tables\.

Let’s do one quick validation pass so the next notebooks can trust these files without re\-checking the whole section manually\.

In [64]:
# ---------------------------------------------------------------------
# Validate shared anomaly framework exports
# ---------------------------------------------------------------------

shared_framework_validation_records = []

for export_name, export_path in ANOMALY_FRAMEWORK_FINAL_EXPORT_PATHS.items():
    export_df = pd.read_csv(export_path)

    shared_framework_validation_records.append(
        {
            "export_name": export_name,
            "path_exists": export_path.exists(),
            "rows": int(len(export_df)),
            "columns": int(len(export_df.columns)),
            "status": "pass" if export_path.exists() and len(export_df) > 0 else "review",
        }
    )

shared_framework_validation_df = pd.DataFrame(shared_framework_validation_records)

display(shared_framework_validation_df)

assert shared_framework_validation_df["status"].eq("pass").all(), (
    "One or more shared anomaly framework exports failed validation."
)

,export_name,path_exists,rows,columns,status
0,shared_representation_defaults,True,15,7,pass
1,shared_calibration_defaults,True,5,8,pass
2,shared_comparison_universe_defaults,True,6,6,pass
3,shared_framework_interpretation_defaults,True,5,10,pass
4,shared_framework_asset_manifest,True,8,4,pass


Findings\. All exported shared\-framework tables exist and loaded back successfully, with non\-empty contents and the expected basic structure\. That gives the downstream anomaly notebooks a clean and validated handoff point\.

### 3\.3\.1 Final Tables Inventory

shared\_representation\_defaults\.csv
Shared downstream representation registry, including the Taxi PCA handling decision\.

shared\_calibration\_defaults\.csv
Shared calibration\-sample ids, metadata paths, sample size, and sampling design\.

shared\_comparison\_universe\_defaults\.csv
Shared downstream scoring baseline definitions and the Taxi\-only policy sanity check\.

shared\_framework\_interpretation\_defaults\.csv
Framework\-level interpretation defaults, including geography\-caution and repeated\-zone caution settings\.

shared\_framework\_asset\_manifest\.csv
Compact manifest pointing downstream notebooks to the key framework assets and supporting summaries\.

\{feature\_set\}\_anomaly\_calibration\_ids\.parquet
Final shared calibration sample ids for each modality\.

\{feature\_set\}\_anomaly\_calibration\_metadata\.parquet
Final shared calibration sample metadata for each modality\.

## Close

Notebook 3\.3\.1 established the shared anomaly\-detection framework that the method\-specific notebooks will inherit\. We loaded and validated the downstream candidate representations, built a common calibration sample, chose temporal\_bucket \+ policy\_geography\_class as the shared scoring baseline, advanced pca\_80pct and umap\_pca\_to\_umap\_10d as the downstream anomaly representations, and resolved the Taxi exception by carrying forward split pre/post PCA handling\. The result is a single framework handoff for 3\.3\.2, 3\.3\.3, and 3\.3\.4: shared samples, shared baseline logic, shared metadata expectations, and a clear record of the modality\-specific cautions that still matter during downstream anomaly generation\.

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=4a322346-8e1e-4650-8cef-fe9b767d96fb' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>